<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.pt/cap05/cap05_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 5 Transformadas e Compressão

🚧 **Em construção!**

Nos capítulos anteriores, todas as operações foram realizadas **no domínio espacial**, em que os algoritmos atuam diretamente sobre os valores de intensidade dos pixels.

Neste capítulo será apresentada uma abordagem complementar: o **domínio da frequência**, no qual a imagem é representada pelas variações espaciais de intensidade, e não apenas pelos valores individuais dos pixels.

O conceito de **frequência espacial** descreve a rapidez com que a intensidade varia ao longo da imagem. Variações lentas correspondem a **baixas frequências**, enquanto bordas, detalhes finos e ruídos correspondem a **altas frequências**.

Essa representação baseia-se no fato de que qualquer imagem digital discreta pode ser decomposta em uma combinação de **funções ortogonais**. A **Transformada de Fourier** utiliza uma **base de exponenciais complexas bidimensionais** (equivalentes a senoides com orientação e frequência específicas). Outras transformadas, como a **Transformada de Cossenos (DCT)** e a **Transformada *Wavelet* (DWT)**, utilizam diferentes famílias de funções de base — cossenos bidimensionais no caso da DCT, e funções com suporte compacto no caso das *wavelets*.

Entre as principais aplicações dessa representação destacam-se:

1. **Filtragem no domínio da frequência**, para atenuar ou realçar determinadas faixas de frequência;
2. **Análise multirresolução por meio de transformadas *wavelet***, que representa estruturas em diferentes escalas;
3. **Compressão de imagens**, pela redução do número de coeficientes necessários para representar a imagem.

## 5.1 Objetivos

Ao concluir este capítulo, você será capaz de:

- **Interpretar o espectro de Fourier** de uma imagem, distinguindo magnitude, fase e componentes de frequência;
- **Aplicar o Teorema da Convolução** para realizar filtragem no domínio da frequência utilizando a Transformada Rápida de Fourier (FFT);
- **Projetar e analisar filtros no domínio da frequência**, compreendendo o funcionamento de filtros passa-baixa, passa-alta e *notch*;
- **Compreender a análise multirresolução por transformadas *wavelet*** e sua aplicação na representação hierárquica de imagens;
- **Descrever o processo de compressão de imagens**, incluindo a Transformada Discreta do Cosseno (DCT) e a quantização dos coeficientes;
- **Selecionar formatos de armazenamento de imagens**, como JPEG, PNG e WebP, de acordo com os requisitos da aplicação.

## 5.2 Configuração do Ambiente

In [ ]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build da trilha C++

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# Kernel Python mesmo na trilha C++. cpp=True baixa morph.hpp + stb; as
# células %%writefile *.cpp deste capítulo compilam COM OpenCV
# (-DMM_USE_OPENCV + pkg-config opencv4).
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

## 5.3 Transformada de Fourier Discreta 2D

A análise de Fourier baseia-se no princípio de que qualquer sinal periódico pode ser representado como uma soma de funções senoidais com diferentes frequências, amplitudes e fases. Esse conceito também se aplica às imagens digitais, permitindo representá-las no **domínio da frequência** em vez do domínio espacial.

A [Figura fig-decomposicao-1d](#fig-decomposicao-1d) ilustra essa decomposição para um sinal unidimensional. No caso de uma imagem, a Transformada Discreta de Fourier (DFT) converte a matriz de intensidades $f(x,y)$ em um conjunto de coeficientes que descreve a contribuição das diferentes frequências espaciais presentes na imagem.

### 5.3.1 Simulador: Reconstruindo Sinais com Senoides

Antes de estudar imagens bidimensionais, o simulador da [Figura 5.1](#fig-05-sim-05-freq) ilustra o princípio da análise de Fourier para sinais unidimensionais: **uma forma de onda pode ser aproximada pela soma de senoides com diferentes frequências e amplitudes**.

À medida que novos termos são adicionados, a soma das senoides (curva preta) aproxima-se da forma de onda de referência (tracejada). O gráfico inferior apresenta o espectro de amplitudes, indicando a contribuição de cada frequência para a reconstrução do sinal.

> ### 💡 Atividade
>
> Explore o simulador e responda:
>
> 1. Quantos termos são necessários para obter uma boa aproximação da onda quadrada?
> 2. Qual das três formas de onda converge mais rapidamente? Justifique sua resposta.
> 3. Como o espectro de amplitudes se altera ao trocar a onda quadrada pela triangular?

> ### 📝 Respostas
>
> **1. Quantos termos são necessários para uma boa aproximação da onda quadrada?**
>
> Com aproximadamente 15 a 20 termos, a forma da onda já se aproxima bem da referência. Entretanto, próximo às descontinuidades permanece uma pequena oscilação, conhecida como **fenômeno de Gibbs**, que não desaparece mesmo com a adição de mais termos.
>
> **2. Qual forma converge mais rapidamente? Por quê?**
>
> A **onda triangular** converge mais rapidamente, pois as amplitudes de seus harmônicos decaem mais rápido que as da onda quadrada e da onda dente-de-serra. Como consequência, poucos termos já produzem uma boa aproximação.
>
> **3. Como o espectro muda entre a onda quadrada e a triangular?**
>
> Ambas possuem apenas **harmônicos ímpares**, mas, na onda triangular, as amplitudes diminuem muito mais rapidamente. Assim, poucos harmônicos são suficientes para reconstruir o sinal com boa precisão.

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-05-freq" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-freq * { box-sizing: border-box; }
  #sim-05-freq canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-freq button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; }
  #sim-05-freq button:hover { background: #e8dfcf; }
  #sim-05-freq button.sim05fft_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim05fft_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05fft_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim05fft_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 90px; }
  .sim05fft_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim05fft_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">∿ Simulador: Decomposição de Fourier 1D</span>
  <span class="sim05fft_pill">soma de senoides</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Termos</div><div id="sim05fft_nTerms" class="sim05fft_stat_value" style="color:#2980b9;">1</div></div>
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Erro RMS</div><div id="sim05fft_rms" class="sim05fft_stat_value" style="color:#c0392b;">–</div></div>
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Forma Alvo</div><div id="sim05fft_target" class="sim05fft_stat_value" style="color:#27ae60; font-size:13px;">quadrada</div></div>
  </div>

  <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:12px;text-align:center;">
    <canvas id="sim05fft_Canvas" width="660" height="200" style="margin:0 auto;"></canvas>
    <canvas id="sim05fft_SpecCanvas" width="660" height="80" style="margin:8px auto 0 auto;"></canvas>
  </div>

  <div style="display:flex; gap:12px; margin-top:12px; flex-wrap:wrap;">
    <div class="sim05fft_panel" style="flex:1; min-width:200px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Forma Alvo</div>
      <div style="display:flex; gap:6px; flex-wrap:wrap;">
        <button id="sim05fft_sq" class="sim05fft_active" onclick="sim05fft_setTarget('square')" style="flex:1; justify-content:center;">Quadrada</button>
        <button id="sim05fft_tr" onclick="sim05fft_setTarget('triangle')" style="flex:1; justify-content:center;">Triangular</button>
        <button id="sim05fft_sw" onclick="sim05fft_setTarget('sawtooth')" style="flex:1; justify-content:center;">Dente-Serra</button>
      </div>
    </div>
    
    <div class="sim05fft_panel" style="flex:1; min-width:180px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Nº de Termos</div>
      <div style="display:flex; align-items:center; gap:8px;">
        <input type="range" id="sim05fft_slider" min="1" max="25" value="1" style="flex:1; cursor:pointer; height:4px;">
        <span id="sim05fft_slVal" style="font-size:12px; font-family:monospace; font-weight:700; min-width:22px; color:#26241d;">1</span>
      </div>
    </div>

    <div class="sim05fft_panel" style="flex:1; min-width:140px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Exibição</div>
      <div style="display:flex; gap:6px;">
        <button id="sim05fft_chk_comp" class="sim05fft_active" onclick="sim05fft_toggleComp()" style="flex:1; justify-content:center;">Componentes</button>
        <button id="sim05fft_chk_sum" class="sim05fft_active" onclick="sim05fft_toggleSum()" style="flex:1; justify-content:center;">Soma</button>
      </div>
    </div>
  </div>

  <div id="sim05fft_termList" style="margin-top:12px; display:flex; gap:6px; flex-wrap:wrap; justify-content:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim05FFT(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const C = root.querySelector('#sim05fft_Canvas');
    const S = root.querySelector('#sim05fft_SpecCanvas');
    const ctx = C.getContext('2d');
    const sctx = S.getContext('2d');
    const sim05fft_N = 512;
    let sim05fft_nTerms = 1, sim05fft_showComp = true, sim05fft_showSum = true, sim05fft_targetType = 'square';

    const sim05fft_PALETTE = ['#2980b9','#27ae60','#b9770e','#c0392b','#8e44ad','#16a085','#d35400','#2c3e50'];

    function sim05fft_getTerms(type, n) {
      const terms = [];
      for (let k = 1; k <= n; k++) {
        let freq, amp, phase = 0;
        if (type === 'square') {
          const m = 2*k - 1;
          freq = m; amp = (4/Math.PI) * (1/m);
        } else if (type === 'triangle') {
          const m = 2*k - 1;
          freq = m; amp = (8/Math.PI**2) * (1/m**2);
          phase = -Math.PI/2;
        } else {
          freq = k; amp = (2/Math.PI) * (1/k);
          phase = Math.PI;
        }
        terms.push({freq, amp, phase});
      }
      return terms;
    }

    function sim05fft_getTarget(type) {
      const t = new Float32Array(sim05fft_N);
      for (let i = 0; i < sim05fft_N; i++) {
        const x = i / sim05fft_N;
        if (type === 'square') t[i] = x < 0.5 ? 1 : -1;
        else if (type === 'triangle') t[i] = x < 0.5 ? (4*x - 1) : (3 - 4*x);
        else t[i] = 2*x - 1;
      }
      return t;
    }

    function sim05fft_evalTerms(terms) {
      const sig = new Float32Array(sim05fft_N);
      for (const {freq, amp, phase} of terms) {
        for (let i = 0; i < sim05fft_N; i++) {
          sig[i] += amp * Math.sin(2*Math.PI*freq*i/sim05fft_N + phase);
        }
      }
      return sig;
    }

    function sim05fft_draw() {
      const W = C.width, H = C.height;
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#ffffff'; ctx.fillRect(0, 0, W, H);

      const terms = sim05fft_getTerms(sim05fft_targetType, sim05fft_nTerms);
      const sum = sim05fft_evalTerms(terms);
      const target = sim05fft_getTarget(sim05fft_targetType);
      let rms = 0;
      for (let i = 0; i < sim05fft_N; i++) rms += (sum[i]-target[i])**2;
      rms = Math.sqrt(rms/sim05fft_N);

      // Grid
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 0.5; ctx.setLineDash([3,3]);
      ctx.beginPath(); ctx.moveTo(0,H/2); ctx.lineTo(W,H/2); ctx.stroke();
      ctx.setLineDash([]);

      // Componentes individuais
      if (sim05fft_showComp) {
        for (let k = 0; k < terms.length; k++) {
          const {freq, amp, phase} = terms[k];
          ctx.strokeStyle = sim05fft_PALETTE[k % sim05fft_PALETTE.length] + '55';
          ctx.lineWidth = 1;
          ctx.beginPath();
          for (let i = 0; i < sim05fft_N; i++) {
            const x = i * W / sim05fft_N;
            const y = H/2 - amp * Math.sin(2*Math.PI*freq*i/sim05fft_N + phase) * H/3;
            i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
          }
          ctx.stroke();
        }
      }

      // Sinal alvo
      ctx.strokeStyle = '#8a8371'; ctx.lineWidth = 1.5; ctx.setLineDash([4,4]);
      ctx.beginPath();
      for (let i = 0; i < sim05fft_N; i++) {
        const x = i * W / sim05fft_N;
        const y = H/2 - target[i] * H/3;
        i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
      }
      ctx.stroke(); ctx.setLineDash([]);

      // Soma
      if (sim05fft_showSum) {
        ctx.strokeStyle = '#26241d'; ctx.lineWidth = 2.5;
        ctx.beginPath();
        for (let i = 0; i < sim05fft_N; i++) {
          const x = i * W / sim05fft_N;
          const y = H/2 - sum[i] * H/3;
          i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
        }
        ctx.stroke();
      }

      // Espectro
      const SW = S.width, SH = S.height;
      sctx.clearRect(0, 0, SW, SH);
      sctx.fillStyle = '#fafaf7'; sctx.fillRect(0,0,SW,SH);
      const maxFreq = sim05fft_getTerms(sim05fft_targetType, 25)[24].freq;
      for (let k = 0; k < terms.length; k++) {
        const {freq, amp} = terms[k];
        const x = freq/maxFreq * SW;
        const h2 = amp / 2 * SH * 0.8;
        sctx.fillStyle = sim05fft_PALETTE[k % sim05fft_PALETTE.length];
        sctx.fillRect(x-2, SH-h2, 4, h2);
      }
      sctx.strokeStyle = '#e4dcc8'; sctx.lineWidth = 0.5;
      sctx.strokeRect(0,0,SW,SH);
      sctx.fillStyle = '#8a8371'; sctx.font = '9.5px monospace'; sctx.textAlign='left';
      sctx.fillText('Espectro de Amplitude (frequência →)', 6, 14);

      // Stats
      root.querySelector('#sim05fft_nTerms').textContent = sim05fft_nTerms;
      root.querySelector('#sim05fft_rms').textContent = rms.toFixed(3);

      // Term list
      const tl = root.querySelector('#sim05fft_termList');
      tl.innerHTML = '';
      for (let k = 0; k < Math.min(terms.length, 8); k++) {
        const {freq, amp} = terms[k];
        const d = document.createElement('span');
        d.style.cssText = 'font-size:10px; display:flex; align-items:center; gap:4px; background:#fafaf7; border:1px solid #e9e3d3; padding:4px 8px; border-radius:6px;';
        d.innerHTML = '<span style="width:10px;height:10px;border-radius:3px;background:' + sim05fft_PALETTE[k%sim05fft_PALETTE.length] + ';display:inline-block;"></span><span style="font-weight:700; color:#5e5a4a;">k=' + freq + ' A=' + amp.toFixed(2) + '</span>';
        tl.appendChild(d);
      }
    }

    window.sim05fft_setTarget = function(t) {
      sim05fft_targetType = t;
      ['sq','tr','sw'].forEach(id => {
        const el = root.querySelector('#sim05fft_' + id);
        if (el) el.classList.remove('sim05fft_active');
      });
      const map = {square:'sq', triangle:'tr', sawtooth:'sw'};
      root.querySelector('#sim05fft_' + map[t]).classList.add('sim05fft_active');
      root.querySelector('#sim05fft_target').textContent = {square:'quadrada',triangle:'triangular',sawtooth:'dente-serra'}[t];
      sim05fft_draw();
    };

    window.sim05fft_toggleComp = function() {
      sim05fft_showComp = !sim05fft_showComp;
      root.querySelector('#sim05fft_chk_comp').classList.toggle('sim05fft_active', sim05fft_showComp);
      sim05fft_draw();
    };
    
    window.sim05fft_toggleSum = function() {
      sim05fft_showSum = !sim05fft_showSum;
      root.querySelector('#sim05fft_chk_sum').classList.toggle('sim05fft_active', sim05fft_showSum);
      sim05fft_draw();
    };

    root.querySelector('#sim05fft_slider').addEventListener('input', function(){
      sim05fft_nTerms = +this.value;
      root.querySelector('#sim05fft_slVal').textContent = sim05fft_nTerms;
      sim05fft_draw();
    });

    sim05fft_draw();
  }

  function tryInitSim05FFT(){
    var root = document.getElementById('sim-05-freq');
    if (root) initSim05FFT(root); else setTimeout(tryInitSim05FFT, 200);
  }
  tryInitSim05FFT();
})();
</script>
""")

**Figura 5.1:** Simulador interativo da decomposição de Fourier 1D: visualização da soma de senoides com diferentes frequências, amplitudes e fases. Adicione termos e observe a convergência para formas de onda arbitrárias.


### 5.3.2 Interpretação do espectro de frequência

Ao aplicar a Transformada Discreta de Fourier (DFT) a uma imagem e visualizar o módulo de seus coeficientes (ver [Figura 5.4](#fig-05-espectro-conceitual)), obtém-se o **espectro de magnitude**, que mostra a distribuição das frequências espaciais presentes na imagem.

O coeficiente localizado na origem da DFT, denominado **componente DC** (*Direct Current*), corresponde à frequência nula e representa a intensidade média da imagem. Por convenção, esse coeficiente é armazenado no canto superior esquerdo do espectro. Para facilitar sua interpretação, aplica-se a operação **FFT Shift**, que desloca a componente DC para o centro da imagem. Após esse deslocamento, as baixas frequências concentram-se na região central, enquanto as altas frequências ficam próximas às bordas, como resume a [Tabela 5.1](#tbl-05-espectro-regioes).

<a id="tbl-05-espectro-regioes"></a>

**Tabela 5.1:** Correspondência entre as regiões do espectro de magnitude após a aplicação do FFT Shift.

| Região do espectro | Componentes predominantes | Exemplos na imagem |
|:---|:---|:---|
| **Centro** (baixas frequências) | Variações espaciais lentas | Iluminação, regiões homogêneas e formas globais |
| **Região intermediária** (médias frequências) | Variações de escala intermediária | Texturas e padrões repetitivos |
| **Bordas** (altas frequências) | Variações espaciais rápidas | Contornos, detalhes finos e ruído |


Essa organização facilita a interpretação do espectro e o projeto de filtros. A atenuação das baixas frequências reduz as variações globais de intensidade, enquanto a atenuação das altas frequências suaviza a imagem ao reduzir detalhes finos e parte do ruído.

### 5.3.3 O Experimento da Grade: Construindo uma Imagem a partir de um Único Coeficiente

Antes de apresentar a formulação matemática da Transformada Discreta de Fourier (DFT), é útil analisar sua inversa, denominada Transformada Discreta Inversa de Fourier (IDFT). Considere um espectro em que todos os coeficientes sejam nulos, exceto um. Um exemplo dessa construção é apresentado no código da [Figura 5.2](#fig-05-grade-2d) e pode ser explorado interativamente no simulador da [Figura 5.3](#fig-05-sim-05-grade-2d).

A imagem reconstruída é uma senoide bidimensional. A posição do coeficiente no espectro determina sua **orientação** e sua **frequência espacial**, enquanto sua magnitude e sua fase definem, respectivamente, sua amplitude e seu deslocamento espacial. Assim, cada coeficiente da DFT representa uma componente senoidal, e a imagem original pode ser reconstruída pela soma de todas essas componentes.

In [ ]:
%%writefile tmp/fig_05_grade_2d.cpp
#define MM_OUT "tmp/fig_05_grade_2d.png"
//| label: fig-05-grade-2d
//| fig-cap: "Toda frequência no espectro (ponto isolado) corresponde a uma onda senoidal 2D rotacionada no domínio espacial."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <complex>
#include <vector>
#include <cmath>
#include "morph.hpp"

// Função auxiliar para fftshift manual (troca quadrantes)
void fftShift(cv::Mat& mat) {
    int cx = mat.cols / 2;
    int cy = mat.rows / 2;

    cv::Mat q0(mat, cv::Rect(0, 0, cx, cy));   // top-left
    cv::Mat q1(mat, cv::Rect(cx, 0, cx, cy));  // top-right
    cv::Mat q2(mat, cv::Rect(0, cy, cx, cy));  // bottom-left
    cv::Mat q3(mat, cv::Rect(cx, cy, cx, cy)); // bottom-right

    cv::Mat tmp;
    q0.copyTo(tmp);
    q3.copyTo(q0);
    tmp.copyTo(q3);

    q1.copyTo(tmp);
    q2.copyTo(q1);
    tmp.copyTo(q2);
}

// Função auxiliar para ifftshift manual
void ifftShift(cv::Mat& mat) {
    fftShift(mat); // ifftshift é igual a fftshift para tamanhos pares
}

int main() {
    int N_grid = 100;
    cv::Mat espectro_vazio = cv::Mat::zeros(N_grid, N_grid, CV_32FC2); // 2 canais: real e imaginário

    // Acendendo um único ponto (frequência) fora do centro
    int u0 = 10, v0 = 5;
    int idx_y = N_grid/2 - v0;
    int idx_x = N_grid/2 - u0;
    espectro_vazio.at<cv::Vec2f>(idx_y, idx_x) = cv::Vec2f(1000.0f, 0.0f);

    // Retornando para o domínio espacial (IDFT)
    cv::Mat espectro_shifted;
    espectro_vazio.copyTo(espectro_shifted);
    ifftShift(espectro_shifted); // ifftshift antes da IDFT

    cv::Mat espectro_float[2];
    cv::split(espectro_shifted, espectro_float);

    cv::Mat espectro_complex;
    cv::merge(espectro_float, 2, espectro_complex);

    cv::Mat onda_complex;
    cv::idft(espectro_complex, onda_complex, cv::DFT_SCALE);

    cv::Mat onda_reais[2];
    cv::split(onda_complex, onda_reais);
    cv::Mat onda_2d = onda_reais[0]; // parte real

    // Normalização para visualização
    cv::Mat onda_vis;
    cv::normalize(onda_2d, onda_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    cv::Mat espectro_mag;
    cv::Mat espectro_canais[2];
    cv::split(espectro_vazio, espectro_canais);
    cv::magnitude(espectro_canais[0], espectro_canais[1], espectro_mag);

    cv::Mat espectro_vis;
    cv::normalize(espectro_mag, espectro_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Destaque visual do ponto
    cv::Mat espectro_color;
    cv::cvtColor(espectro_vis, espectro_color, cv::COLOR_GRAY2BGR);
    cv::circle(espectro_color, cv::Point(N_grid/2 - u0, N_grid/2 - v0), 2, cv::Scalar(0, 0, 255), -1);

    // Converter para mm::Image e exibir
    mm::Image img_espectro(espectro_color.rows, espectro_color.cols, espectro_color.channels());
    std::memcpy(img_espectro.data.data(), espectro_color.data, img_espectro.data.size());

    mm::Image img_onda(onda_vis.rows, onda_vis.cols, onda_vis.channels());
    std::memcpy(img_onda.data.data(), onda_vis.data, img_onda.data.size());

    mm::show({img_espectro, img_onda},
             MM_OUT,
             {"Espectro (1 ponto ativo)", "Onda 2D Resultante (IDFT)"},
             2);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_grade_2d.cpp -o tmp/fig_05_grade_2d \
  && ./tmp/fig_05_grade_2d \
  && test -f "tmp/fig_05_grade_2d.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_grade_2d.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_grade_2d.png"), figsize=(10, 4))

**Figura 5.2:** Toda frequência no espectro (ponto isolado) corresponde a uma onda senoidal 2D rotacionada no domínio espacial.


In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-05-grade-2d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-grade-2d * { box-sizing: border-box; }
  #sim-05-grade-2d canvas { display: block; background: #ffffff; border: 1px solid #e4dcc8; border-radius: 8px; }
  #sim-05-grade-2d button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-grade-2d button:hover { background: #e8dfcf; }
  .sim-05-grade-2d_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-05-grade-2d_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-05-grade-2d_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 80px; }
  .sim-05-grade-2d_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-05-grade-2d_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-05-grade-2d_slider_container { display: flex; align-items: center; gap: 8px; margin-bottom: 8px; }
  .sim-05-grade-2d_slider_container label { font-size: 11px; font-weight: 700; min-width: 120px; display: inline-block; color: #5e5a4a; }
  .sim-05-grade-2d_slider_container input[type=range] { flex: 1; cursor: pointer; height: 4px; }
  .sim-05-grade-2d_slider_val { font-size: 12px; font-family: monospace; font-weight: 700; min-width: 25px; text-align: right; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">∿ Simulador: Síntese de Frequência 2D (IDFT)</span>
  <span class="sim-05-grade-2d_pill">Espaço de Fourier</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Frequência u</div><div id="sim-05-grade-2d_valU" class="sim-05-grade-2d_stat_value" style="color:#2980b9;">10</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Frequência v</div><div id="sim-05-grade-2d_valV" class="sim-05-grade-2d_stat_value" style="color:#27ae60;">5</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Distância R</div><div id="sim-05-grade-2d_valR" class="sim-05-grade-2d_stat_value" style="color:#b9770e;">11.18</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Ângulo θ</div><div id="sim-05-grade-2d_valAng" class="sim-05-grade-2d_stat_value" style="color:#c0392b;">26.6°</div></div>
  </div>

  <!-- Exibição Central (Espectro e Espaço) -->
  <div style="display: flex; gap: 16px; justify-content: center; align-items: center; margin-bottom: 14px; flex-wrap: wrap;">
    <div style="text-align: center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
      <div style="font-size: 10.5px; font-weight: 700; color: #5e5a4a; margin-bottom: 8px;">Espectro (Clique para mover o ponto)</div>
      <canvas id="sim-05-grade-2d_CanvasSpec" width="220" height="220" style="margin:0 auto;"></canvas>
    </div>
    <div style="font-size: 20px; color: #8a8371; font-weight: bold;">➔</div>
    <div style="text-align: center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
      <div style="font-size: 10.5px; font-weight: 700; color: #5e5a4a; margin-bottom: 8px;">Onda 2D Resultante (Domínio Espacial)</div>
      <canvas id="sim-05-grade-2d_CanvasSpace" width="220" height="220" style="margin:0 auto;"></canvas>
    </div>
  </div>

  <!-- Controles Deslizantes -->
  <div class="sim-05-grade-2d_panel">
    <div class="sim-05-grade-2d_slider_container">
      <label style="color:#2980b9;">Deslocamento u (X):</label>
      <input type="range" id="sim-05-grade-2d_sliderU" min="-30" max="30" value="10">
      <span id="sim-05-grade-2d_slValU" class="sim-05-grade-2d_slider_val">10</span>
    </div>
    <div class="sim-05-grade-2d_slider_container" style="margin-bottom:0;">
      <label style="color:#27ae60;">Deslocamento v (Y):</label>
      <input type="range" id="sim-05-grade-2d_sliderV" min="-30" max="30" value="5">
      <span id="sim-05-grade-2d_slValV" class="sim-05-grade-2d_slider_val">5</span>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Exp2D(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const specC = root.querySelector('#sim-05-grade-2d_CanvasSpec');
    const spaceC = root.querySelector('#sim-05-grade-2d_CanvasSpace');
    const sctx = specC.getContext('2d');
    const spctx = spaceC.getContext('2d');

    let u = 10;
    let v = 5;
    const N = 220;

    function updateMetrics() {
      const r = Math.sqrt(u*u + v*v);
      let angle = Math.atan2(v, u) * (180 / Math.PI);
      if (angle < 0) angle += 360;

      root.querySelector('#sim-05-grade-2d_valU').textContent = u;
      root.querySelector('#sim-05-grade-2d_valV').textContent = v;
      root.querySelector('#sim-05-grade-2d_valR').textContent = r.toFixed(2);
      root.querySelector('#sim-05-grade-2d_valAng').textContent = angle.toFixed(1) + '°';

      root.querySelector('#sim-05-grade-2d_sliderU').value = u;
      root.querySelector('#sim-05-grade-2d_sliderV').value = v;
      root.querySelector('#sim-05-grade-2d_slValU').textContent = u;
      root.querySelector('#sim-05-grade-2d_slValV').textContent = v;
    }

    function render() {
      updateMetrics();

      // 1. Desenhar Espectro
      sctx.fillStyle = '#fafaf7';
      sctx.fillRect(0, 0, N, N);

      sctx.strokeStyle = '#e4dcc8';
      sctx.lineWidth = 1;
      sctx.beginPath();
      sctx.moveTo(N/2, 0); sctx.lineTo(N/2, N);
      sctx.moveTo(0, N/2); sctx.lineTo(N, N/2);
      sctx.stroke();

      sctx.fillStyle = '#8a8371';
      sctx.beginPath();
      sctx.arc(N/2, N/2, 2.5, 0, 2*Math.PI);
      sctx.fill();

      let ptX = N/2 + u;
      let ptY = N/2 - v;

      sctx.strokeStyle = '#b9770e88';
      sctx.lineWidth = 1.5;
      sctx.beginPath();
      sctx.moveTo(N/2, N/2);
      sctx.lineTo(ptX, ptY);
      sctx.stroke();

      let symX = N/2 - u;
      let symY = N/2 + v;
      sctx.fillStyle = '#c0392b88';
      sctx.beginPath();
      sctx.arc(symX, symY, 4, 0, 2*Math.PI);
      sctx.fill();

      sctx.fillStyle = '#2980b9';
      sctx.strokeStyle = '#ffffff';
      sctx.lineWidth = 1.5;
      sctx.beginPath();
      sctx.arc(ptX, ptY, 5.5, 0, 2*Math.PI);
      sctx.fill();
      sctx.stroke();

      // 2. Desenhar Onda Espacial 2D Resultante
      const imgData = spctx.createImageData(N, N);
      const data = imgData.data;
      const freqScale = 2 * Math.PI / N;

      for (let y = 0; y < N; y++) {
        const ny = y - N/2;
        for (let x = 0; x < N; x++) {
          const nx = x - N/2;
          const val = Math.cos(freqScale * (u * nx + v * (-ny)));
          const intensity = Math.floor((val + 1) * 127.5);

          const idx = (y * N + x) * 4;
          data[idx]     = intensity;
          data[idx + 1] = intensity;
          data[idx + 2] = intensity;
          data[idx + 3] = 255;
        }
      }
      spctx.putImageData(imgData, 0, 0);
    }

    root.querySelector('#sim-05-grade-2d_sliderU').addEventListener('input', function() {
      u = parseInt(this.value, 10);
      render();
    });

    root.querySelector('#sim-05-grade-2d_sliderV').addEventListener('input', function() {
      v = parseInt(this.value, 10);
      render();
    });

    specC.addEventListener('mousedown', function(e) {
      const rect = specC.getBoundingClientRect();
      const clickX = e.clientX - rect.left;
      const clickY = e.clientY - rect.top;

      let newU = Math.round(clickX - N/2);
      let newV = Math.round(N/2 - clickY);

      u = Math.max(-30, Math.min(30, newU));
      v = Math.max(-30, Math.min(30, newV));

      render();
    });

    render();
  }

  function tryInitSim05Exp2D(){
    var root = document.getElementById('sim-05-grade-2d');
    if (root) initSim05Exp2D(root); else setTimeout(tryInitSim05Exp2D, 200);
  }
  tryInitSim05Exp2D();
})();
</script>
""")

**Figura 5.3:** Simulador interativo da síntese de Fourier 2D. Altere a posição horizontal ($u$) e vertical ($v$) do coeficiente no espectro de frequências centrado e observe como a distância em relação ao centro dita a frequência espacial (espessura) e o ângulo dita a orientação da onda senoidal gerada.


### 5.3.4 Definição Matemática

Considere uma imagem $f(x,y)$ com dimensões $M \times N$. Sua **Transformada Discreta de Fourier 2D** (DFT) é definida por:

<a id="eq-05-dft"></a>
$$
F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)} \tag{5.1}
$$


em que $u = 0, 1, \ldots, M-1$ e $v = 0, 1, \ldots, N-1$ representam as frequências discretas nas direções horizontal e vertical, respectivamente. O termo exponencial corresponde a uma senoide bidimensional, cuja frequência e orientação são determinadas pelos índices $(u,v)$.

A **Transformada Discreta Inversa de Fourier 2D** (IDFT) reconstrói a imagem original a partir de seus coeficientes:

<a id="eq-05-idft"></a>
$$
f(x,y) = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F(u,v)\, e^{j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)} \tag{5.2}
$$


As Equações [Equação 5.1](#eq-05-dft) e [Equação 5.2](#eq-05-idft) mostram que a DFT e a IDFT formam um par de transformações: a primeira converte a imagem para o domínio da frequência, enquanto a segunda reconstrói exatamente a imagem original a partir de seus coeficientes.

> ### 📝 5.3.4.1 Sobre o símbolo $j$
>
> O termo $j$ denota a **unidade imaginária**, definida por $j^2 = -1$. Em engenharia e processamento de sinais, adota-se $j$ em vez de $i$ para evitar conflito com a notação de corrente elétrica. Sua utilização na exponencial complexa, regida pela fórmula de Euler ($e^{j\theta} = \cos\theta + j\sin\theta$), permite representar de forma compacta a amplitude e a fase de cada frequência espacial presente na imagem.

> ### 📝 O que é o componente DC?
>
> O coeficiente $F(0,0)$, denominado **componente DC** (*Direct Current*), é igual à soma das intensidades de todos os pixels da imagem (ver [Figura 5.4](#fig-05-espectro-conceitual)):
>
> $$
> F(0,0)=MN\,\bar{f},
> $$
>
> em que $\bar{f}$ é a intensidade média da imagem. Por isso, o componente DC representa o nível médio de intensidade e, na maioria das imagens naturais, possui a maior magnitude do espectro.
>
> Os demais coeficientes representam variações em torno dessa média. Após a aplicação do **FFT Shift**, o componente DC é deslocado para o centro do espectro, concentrando as baixas frequências na região central e as altas frequências nas bordas.

In [ ]:
from IPython.display import HTML

HTML("""
<div style="font-family:sans-serif; max-width:880px; margin:0 auto; padding:10px;">
<div style="text-align:center; font-size:12px; font-weight:bold; color:#374151; margin-bottom:8px;">
  Anatomia do Espectro de Fourier 2D (após fftshift)
</div>
<svg viewBox="0 0 640 320" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#f9fafb;">
  <!-- Fundo gradiente radial simulado -->
  <defs>
    <radialGradient id="specGrad" cx="50%" cy="50%" r="50%">
      <stop offset="0%" style="stop-color:#1e3a5f;stop-opacity:1"/>
      <stop offset="20%" style="stop-color:#1a5276;stop-opacity:1"/>
      <stop offset="50%" style="stop-color:#0d2137;stop-opacity:1"/>
      <stop offset="100%" style="stop-color:#050e1a;stop-opacity:1"/>
    </radialGradient>
    <radialGradient id="brightCenter" cx="50%" cy="50%" r="15%">
      <stop offset="0%" style="stop-color:#ffffff;stop-opacity:1"/>
      <stop offset="60%" style="stop-color:#f0c040;stop-opacity:0.9"/>
      <stop offset="100%" style="stop-color:#1a5276;stop-opacity:0"/>
    </radialGradient>
  </defs>
  <rect x="20" y="10" width="380" height="300" fill="url(#specGrad)" rx="6"/>
  <rect x="20" y="10" width="380" height="300" fill="url(#brightCenter)" rx="6"/>
  <!-- Cruzes de alta energia (bordas horizontais/verticais) -->
  <line x1="210" y1="10" x2="210" y2="310" stroke="#4a9eda" stroke-width="1.5" opacity="0.4"/>
  <line x1="20" y1="160" x2="400" y2="160" stroke="#4a9eda" stroke-width="1.5" opacity="0.4"/>
  <!-- Círculos de frequência -->
  <circle cx="210" cy="160" r="30" fill="none" stroke="#f0c040" stroke-width="1" stroke-dasharray="4,3" opacity="0.7"/>
  <circle cx="210" cy="160" r="70" fill="none" stroke="#7dd3fc" stroke-width="1" stroke-dasharray="4,3" opacity="0.5"/>
  <circle cx="210" cy="160" r="120" fill="none" stroke="#93c5fd" stroke-width="0.8" stroke-dasharray="4,3" opacity="0.3"/>
  <!-- Ponto DC -->
  <circle cx="210" cy="160" r="6" fill="#ffffff"/>
  <!-- Rótulos no espectro -->
  <text x="210" y="148" font-size="9" fill="#fff" text-anchor="middle" font-weight="bold">DC</text>
  <text x="210" y="205" font-size="8" fill="#f0c040" text-anchor="middle">baixas freq.</text>
  <text x="210" y="245" font-size="8" fill="#7dd3fc" text-anchor="middle">médias freq.</text>
  <text x="330" y="110" font-size="8" fill="#93c5fd" text-anchor="middle">altas freq.</text>
  <text x="210" y="295" font-size="9" fill="#cbd5e1" text-anchor="middle" font-style="italic">Espectro de Magnitude |F(u,v)| — escala log</text>
  <!-- Painel direito: explicações -->
  <rect x="420" y="10" width="200" height="300" fill="#ffffff" rx="6" stroke="#e5e7eb"/>
  <text x="520" y="35" font-size="10" fill="#1e293b" text-anchor="middle" font-weight="bold">Regiões do Espectro</text>
  <!-- DC -->
  <circle cx="440" cy="65" r="7" fill="#ffffff" stroke="#f0c040" stroke-width="2"/>
  <text x="455" y="61" font-size="9" fill="#374151" font-weight="bold">DC (0,0)</text>
  <text x="455" y="73" font-size="8" fill="#6b7280">Média global dos pixels</text>
  <!-- Baixas -->
  <rect x="433" y="95" width="14" height="14" rx="2" fill="#f0c040" opacity="0.7"/>
  <text x="455" y="105" font-size="9" fill="#374151" font-weight="bold">Baixas frequências</text>
  <text x="455" y="116" font-size="8" fill="#6b7280">Forma, fundo, iluminação</text>
  <!-- Médias -->
  <rect x="433" y="135" width="14" height="14" rx="2" fill="#7dd3fc" opacity="0.7"/>
  <text x="455" y="145" font-size="9" fill="#374151" font-weight="bold">Médias frequências</text>
  <text x="455" y="156" font-size="8" fill="#6b7280">Texturas, padrões</text>
  <!-- Altas -->
  <rect x="433" y="175" width="14" height="14" rx="2" fill="#1e3a5f" stroke="#93c5fd" stroke-width="1"/>
  <text x="455" y="185" font-size="9" fill="#374151" font-weight="bold">Altas frequências</text>
  <text x="455" y="196" font-size="8" fill="#6b7280">Bordas, ruído, detalhes</text>
  <!-- Seta de eixos -->
  <text x="440" y="235" font-size="8" fill="#6b7280">u → freq. horizontal</text>
  <text x="440" y="248" font-size="8" fill="#6b7280">v → freq. vertical</text>
  <line x1="440" y1="265" x2="600" y2="265" stroke="#d1d5db" stroke-width="0.8"/>
  <text x="520" y="280" font-size="8" fill="#9ca3af" text-anchor="middle">Visualização em escala log</text>
  <text x="520" y="292" font-size="8" fill="#9ca3af" text-anchor="middle">log(1 + |F|) comprime o intervalo</text>
</svg>
</div>
""")

**Figura 5.4:** Diagrama conceitual do espectro de Fourier 2D centrado.


### 5.3.5 Magnitude e Fase

Cada coeficiente da Transformada Discreta de Fourier (DFT) é um número complexo e pode ser escrito como

$$
F(u,v)=R(u,v)+j\,I(u,v),
$$

em que $R(u,v)$ e $I(u,v)$ correspondem, respectivamente, às partes **real** e **imaginária** do coeficiente. Da Equação [Equação 5.1](#eq-05-dft), obtêm-se

$$
R(u,v)=
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\cos\!\left(
2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)
\right),
$$

e

$$
I(u,v)=
-
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\sin\!\left(
2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)
\right).
$$

A partir dessa representação, definem-se duas grandezas fundamentais:

- **Magnitude**, que indica a intensidade da componente de frequência,

$$
|F(u,v)|=\sqrt{R(u,v)^2+I(u,v)^2};
$$

- **Fase**, que determina o alinhamento (ou deslocamento) espacial da componente,

$$
\phi(u,v)=\operatorname{atan2}\!\left(I(u,v),\,R(u,v)\right).
$$

Assim, cada coeficiente também pode ser escrito em sua forma polar,

$$
F(u,v)=|F(u,v)|\,e^{j\phi(u,v)}.
$$

O espectro de Fourier pode, portanto, ser visualizado por meio de duas imagens distintas: o **espectro de magnitude**, normalmente utilizado para analisar a distribuição das frequências, e o **espectro de fase**, que descreve a organização espacial das componentes senoidais.

Embora o espectro de magnitude seja o mais utilizado para inspeção visual, a fase contém grande parte das informações estruturais da imagem. A combinação de magnitude e fase permite reconstruir exatamente a imagem original por meio da IDFT.

### 5.3.6 O que a Magnitude e a Fase carregam?

Uma demonstração clássica consiste em combinar a magnitude de uma imagem com a fase de outra e reconstruir o resultado. Esse experimento evidencia que:

- **A fase** preserva a estrutura espacial da imagem, incluindo a posição dos objetos, seus contornos e sua geometria. Pequenas alterações na fase podem provocar grandes mudanças visuais.
- **A magnitude** controla como a energia é distribuída entre as frequências espaciais, influenciando principalmente o contraste e a textura.

Quando uma imagem é reconstruída com a magnitude de A e a fase de B, o resultado tende a **assemelhar-se mais a B do que a A**, evidenciando que a fase é o principal componente responsável pela organização espacial da cena. Entretanto, a magnitude continua sendo importante, pois modula o contraste das estruturas reconstruídas. Assim, uma reconstrução fiel depende da combinação consistente entre magnitude e fase.

Um exemplo desse comportamento é apresentado na [Figura 5.5](#fig-05-dft-intro).

> ### 📝 Analogia com Áudio: Limitações e Cuidados
>
> A fase de um sinal desempenha papéis distintos em áudio e imagens:
>
> - **Áudio estéreo ou multicanal:** a fase relativa entre os canais é fundamental para a percepção da posição das fontes sonoras, por meio das diferenças interaurais de tempo (ITD, *Interaural Time Differences*).
> - **Áudio monaural:** a fase absoluta exerce pouca influência perceptual direta.
> - **Imagens (DFT):** a fase é o principal fator responsável pela organização espacial da cena, enquanto a magnitude modula o contraste e a distribuição da energia entre as frequências.
>
> Em ambos os domínios, a **magnitude** está relacionada à intensidade das componentes de frequência: em áudio, influencia o timbre e a intensidade percebida; em imagens, influencia o contraste e a textura.

In [ ]:
# Ainda não portado para esta linguagem nesta versão — referência conceitual em Python.
 
# ── Experimento: A Importância da Fase ───────────────────────────────────────
# ── Carregamento da imagem ────────────────────────────────────────────────────
url     = "https://upload.wikimedia.org/wikipedia/commons/2/25/GAZI.MD.AHAD_11.jpg"
caminho = "imagens/coins.jpg"

if not os.path.exists(caminho):
    os.makedirs("imagens", exist_ok=True)
    img_obj = mm.read(url, pil=True)
    mm.write(img_obj, caminho)
else:
    img_obj = mm.read(caminho, pil=True)

img_color = np.array(img_obj)
img_gray  = mm.gray(img_color)

img_a = cv2.resize(img_gray, (400, 400))

# Criar uma imagem B sintética (padrão geométrico)
img_b = np.zeros((400, 400), dtype=np.uint8)
cv2.rectangle(img_b, (100, 100), (300, 300), 255, -1)
cv2.circle(img_b, (200, 200), 150, 128, 10)

FA = np.fft.fft2(img_a)
FB = np.fft.fft2(img_b)

# Troca de Fase
rec_A_mag_B_fase = np.real(np.fft.ifft2(np.abs(FA) * np.exp(1j * np.angle(FB))))
rec_B_mag_A_fase = np.real(np.fft.ifft2(np.abs(FB) * np.exp(1j * np.angle(FA))))

mm.show(
    [img_a, img_b, rec_A_mag_B_fase, rec_B_mag_A_fase],
    titles=["Imagem A", "Imagem B", "Mag(A) + Fase(B)", "Mag(B) + Fase(A)"],
    cols=4, figsize=(16, 4)
)

print("💡 A fase preserva bordas e contornos; a magnitude controla contraste e")
print("textura. Em áudio estéreo, a fase afeta a localização espacial; em")
print("imagens, determina a organização da cena.")

**Figura 5.5:** Experimento de troca de fase: Imagem A (moedas) e Imagem B (padrão geométrico) reconstruídas com magnitudes e fases trocadas. O resultado mostra que a estrutura visual é **muito mais sensível à fase** do que à magnitude: quando a fase de B é mantida, a imagem resultante preserva a organização espacial de B, mesmo com a magnitude de A. A magnitude, por sua vez, influencia principalmente o contraste e a textura. Observe que a qualidade da reconstrução não é perfeita — há artefatos visíveis —, evidenciando a interdependência entre fase e magnitude para uma representação fiel da imagem.


## 5.4 Teorema da Convolução e Estratégias de Filtragem

O **Teorema da Convolução** estabelece uma relação fundamental entre os domínios espacial e da frequência:

<a id="eq-05-conv-teorema"></a>
$$
f(x,y) \circledast h(x,y) \;\overset{\mathcal{F}}{\longleftrightarrow}\; F(u,v)\,H(u,v) \tag{5.3}
$$


em que $\circledast$ representa a **convolução circular discreta**. Assim, a convolução entre uma imagem $f(x,y)$ e um filtro $h(x,y)$ pode ser substituída pela multiplicação de seus espectros.

Na prática, para obter o mesmo resultado da convolução linear realizada no domínio espacial, aplica-se **zero-padding** antes da Transformada Rápida de Fourier (FFT), evitando artefatos nas bordas da imagem.

Entretanto, nem sempre a filtragem no domínio da frequência é a alternativa mais eficiente. Para filtros como o Gaussiano e o filtro da média (*Box Filter*), a propriedade de **separabilidade** permite reduzir significativamente o custo computacional da convolução no domínio espacial.

### 5.4.1 *Kernel* Separável vs. Não Separável

Um ***kernel* separável** pode ser escrito como o produto externo de dois vetores unidimensionais,

$$
H = v\,h^T,
$$

permitindo que a convolução bidimensional seja substituída por duas convoluções unidimensionais consecutivas: uma na direção horizontal e outra na vertical.

Já um ***kernel* não separável** não admite essa decomposição e, portanto, sua convolução deve ser realizada diretamente sobre a vizinhança bidimensional.

Na prática, para um *kernel* de dimensão $K \times K$, a convolução direta exige $K^2$ multiplicações por pixel, enquanto um *kernel* separável requer apenas $2K$ multiplicações, reduzindo significativamente o custo computacional.

### 5.4.2 Análise de Eficiência Computacional

Considere uma imagem de dimensões $M \times N$ e um filtro quadrado de tamanho $K \times K$. A [Tabela 5.2](#tbl-05-fft-complexity-expanded) compara a complexidade das principais estratégias de filtragem.

<a id="tbl-05-fft-complexity-expanded"></a>

**Tabela 5.2:** Comparação da complexidade da convolução direta, separável e via Transformada Rápida de Fourier (FFT).

| Método de Filtragem | Complexidade Assintótica | Dependência de $K$ | Aplicação típica |
| --- | --- | --- | --- |
| **Espacial não separável** | $\mathcal{O}(MNK^2)$ | Quadrática | *Kernels* pequenos e não separáveis |
| **Espacial separável** | $\mathcal{O}(MNK)$ | Linear | Filtros Gaussiano e da média |
| **Via FFT** | $\mathcal{O}(MN\log(MN))$ | Independente de $K$ | *Kernels* grandes |


Para *kernels* pequenos, a convolução espacial, especialmente quando o filtro é separável, costuma ser mais eficiente devido ao baixo custo das operações. À medida que o tamanho do *kernel* aumenta, a filtragem via FFT torna-se mais vantajosa, pois seu custo praticamente independe da dimensão do filtro.

### 5.4.3 Discussão dos resultados experimentais

O gráfico obtido no ensaio com a imagem das moedas ($2560 \times 1920$), apresentado na [Figura 5](#fig-05-conv-eficiencia), confirma o comportamento previsto pela análise de complexidade computacional.

1. **Convolução não separável ($\mathcal{O}(MNK^2)$)**  
A convolução direta apresenta crescimento quadrático com o tamanho do *kernel*. Para valores pequenos de $K$, o custo é baixo, mas aumenta rapidamente à medida que o *kernel* cresce, tornando-se inviável para aplicações em tempo real.

2. **Filtragem via FFT ($\mathcal{O}(MN \log(MN))$)**  
O custo da FFT depende apenas do tamanho da imagem, sendo independente de $K$. Por isso, seu desempenho permanece aproximadamente constante ao variar o *kernel*, tornando-a vantajosa para filtros grandes ou não separáveis.

3. **Convolução separável ($\mathcal{O}(MNK)$)**  
A decomposição do *kernel* em dois filtros unidimensionais reduz significativamente o custo computacional. Na prática, essa abordagem tende a ser a mais eficiente para filtros separáveis, especialmente em implementações otimizadas.

Em geral, a escolha do método depende do tamanho e da estrutura do *kernel*. Filtros separáveis são mais eficientes no domínio espacial, enquanto a FFT se torna mais vantajosa para *kernels* grandes ou múltiplas convoluções no domínio da frequência.

<a id="eq-05-filter-comparison"></a>
$$
g = \mathcal{F}^{-1}\bigl[\mathcal{F}(f)\cdot \mathcal{F}(h)\bigr]
\quad \text{(FFT)}
\qquad
g = f \circledast h
\quad \text{(convolução direta)}
\qquad
g = (f \circledast v) \circledast h^T
\quad \text{(separável)} \tag{5.4}
$$


onde:

* $f(x,y)$ representa a imagem de entrada;
* $h(x,y)$ é o *kernel* bidimensional do filtro;
* $v$ e $h^T$ são, respectivamente, os vetores vertical e horizontal que compõem o *kernel* separável.

> ### ❗ 5.4.4 O problema da convolução circular (*wrap-around*)
>
> A Transformada Discreta de Fourier (DFT) assume que a imagem é **periodicamente estendida no espaço**, isto é, que suas bordas se repetem indefinidamente.
>
> Nessa condição, a multiplicação no domínio da frequência corresponde a uma **convolução circular** no domínio espacial. Como consequência, regiões opostas da imagem (topo e base, esquerda e direita) passam a interagir artificialmente, conforme ilustrado na [Figura 5.6](#fig-05-padding-error).
>
> A aplicação de *zero-padding* antes da FFT reduz esse efeito ao estender a imagem com valores nulos nas bordas, aproximando o resultado da convolução linear. Esse comportamento pode ser interpretado à luz do Teorema da Convolução, apresentado na [Figura 5.7](#fig-05-conv-teorema).

In [ ]:
%%writefile tmp/fig_05_padding_error.cpp
#define MM_OUT "tmp/fig_05_padding_error.png"
#include <opencv2/opencv.hpp>
#include <complex>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"

//| label: fig-05-padding-error
//| fig-cap: "Sem *padding*, um deslocamento severo faz a imagem vazar para o lado oposto (convolução circular)."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_19.png");
// [pdi:state-io:end]

    // img_gray is provided as mm::Image

    // ── Definir M e N ─────────────────────────────────────────────────────────────
    int M = img_gray.h;
    int N = img_gray.w;

    // Converter para cv::Mat para processamento
    cv::Mat img_mat(M, N, CV_8UC1, img_gray.data.data());

    // Converter para float e depois para complex (2 canais)
    cv::Mat img_float;
    img_mat.convertTo(img_float, CV_32F);

    // Simulação de um filtro de deslocamento brutal
    // Criar o filtro H_shift como matriz complexa (2 canais: real e imaginário)
    cv::Mat H_shift_real(M, N, CV_32F);
    cv::Mat H_shift_imag(M, N, CV_32F);

    for (int u = 0; u < M; u++) {
        for (int v = 0; v < N; v++) {
            float phase = -2.0f * M_PI * (u * 120.0f / M + v * 120.0f / N);
            H_shift_real.at<float>(u, v) = std::cos(phase);
            H_shift_imag.at<float>(u, v) = std::sin(phase);
        }
    }

    // Combinar em uma matriz de 2 canais (complexa)
    std::vector<cv::Mat> channels = {H_shift_real, H_shift_imag};
    cv::Mat H_shift;
    cv::merge(channels, H_shift);

    // Filtragem SEM padding (causa o wrap-around)
    // FFT da imagem
    cv::Mat F_img;
    cv::dft(img_float, F_img, cv::DFT_COMPLEX_OUTPUT);

    // Multiplicação no domínio da frequência
    // Precisamos separar os canais para multiplicar manualmente
    std::vector<cv::Mat> F_channels;
    cv::split(F_img, F_channels);

    std::vector<cv::Mat> H_channels;
    cv::split(H_shift, H_channels);

    // Multiplicação complexa: (a+bi)(c+di) = (ac-bd) + (ad+bc)i
    cv::Mat prod_real, prod_imag;
    cv::multiply(F_channels[0], H_channels[0], prod_real);
    cv::Mat temp;
    cv::multiply(F_channels[1], H_channels[1], temp);
    prod_real -= temp;

    cv::multiply(F_channels[0], H_channels[1], prod_imag);
    cv::multiply(F_channels[1], H_channels[0], temp);
    prod_imag += temp;

    // Combinar produto complexo
    std::vector<cv::Mat> prod_channels = {prod_real, prod_imag};
    cv::Mat prod;
    cv::merge(prod_channels, prod);

    // IFFT
    cv::Mat img_vazada;
    cv::idft(prod, img_vazada, cv::DFT_SCALE | cv::DFT_REAL_OUTPUT);

    // Normalizar para visualização
    cv::Mat img_vazada_vis;
    cv::normalize(img_vazada, img_vazada_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Converter de volta para mm::Image
    mm::Image img_vazada_mm(img_vazada_vis.rows, img_vazada_vis.cols, 1);
    std::memcpy(img_vazada_mm.data.data(), img_vazada_vis.data, img_vazada_mm.data.size());

    // Mostrar resultado
    mm::show(std::vector<mm::Image>{img_gray, img_vazada_mm},
             MM_OUT,
             std::vector<std::string>{"Original", "Filtragem s/ Padding (Vazamento)"}, 2);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_padding_error.cpp -o tmp/fig_05_padding_error \
  && ./tmp/fig_05_padding_error \
  && test -f "tmp/fig_05_padding_error.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_padding_error.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_padding_error.png"), figsize=(10, 4))

**Figura 5.6:** Sem *padding*, um deslocamento severo faz a imagem vazar para o lado oposto (convolução circular).


In [ ]:
%%writefile tmp/fig_05_conv_teorema.cpp
#define MM_OUT "tmp/fig_05_conv_teorema.png"
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include <iostream>
#include <algorithm>
#include <numeric>
#include <cstdint>
#include "morph.hpp"

//| label: fig-05-conv-teorema
//| fig-cap: "Verificação do Teorema da Convolução: a diferença pixel a pixel entre a convolução espacial (cv2.filter2D) e a multiplicação em frequência (FFT) é numericamente nula — confirmando a equivalência teórica."
//| echo: true
//| output: true

// Helper para trocar quadrantes (equivalente ao fftshift)
void fftShift(cv::Mat& mat) {
    int cx = mat.cols / 2;
    int cy = mat.rows / 2;
    cv::Mat q0(mat, cv::Rect(0, 0, cx, cy));
    cv::Mat q1(mat, cv::Rect(cx, 0, cx, cy));
    cv::Mat q2(mat, cv::Rect(0, cy, cx, cy));
    cv::Mat q3(mat, cv::Rect(cx, cy, cx, cy));
    cv::Mat tmp;
    q0.copyTo(tmp);
    q3.copyTo(q0);
    tmp.copyTo(q3);
    q1.copyTo(tmp);
    q2.copyTo(q1);
    tmp.copyTo(q2);
}

// Função para encontrar a próxima potência de 2 maior ou igual a n
int nextPow2(int n) {
    int p = 1;
    while (p < n) p <<= 1;
    return p;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_19.png");
// [pdi:state-io:end]

    // A variável img_gray (mm::Image) será fornecida automaticamente

    // ── Kernel Gaussiano 11×11 
    double sigma = 3.0;
    int K = 11;
    std::vector<double> ks(K);
    std::vector<double> gauss1d(K);
    for (int i = 0; i < K; i++) {
        ks[i] = i - K / 2;
        gauss1d[i] = std::exp(-ks[i] * ks[i] / (2.0 * sigma * sigma));
    }
    double sum_gauss = 0.0;
    for (int i = 0; i < K; i++) sum_gauss += gauss1d[i];
    for (int i = 0; i < K; i++) gauss1d[i] /= sum_gauss;

    // kernel 2D separável (produto externo)
    cv::Mat kernel(K, K, CV_64F);
    for (int i = 0; i < K; i++) {
        for (int j = 0; j < K; j++) {
            kernel.at<double>(i, j) = gauss1d[i] * gauss1d[j];
        }
    }

    // Converte img_gray para cv::Mat com float
    cv::Mat imgGrayMat(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    cv::Mat f_float;
    imgGrayMat.convertTo(f_float, CV_64F);

    // ── Método 1: Convolução espacial direta ─────────────────────────────────────
    cv::Mat conv_esp;
    cv::filter2D(f_float, conv_esp, -1, kernel, cv::Point(-1, -1), 0.0, cv::BORDER_CONSTANT);

    // ── Método 2: Multiplicação em frequência (via FFT) ──────────────────────────
    int M = f_float.rows;
    int N = f_float.cols;

    // Padding para convolução linear (evita aliasing circular)
    int Mpad = nextPow2(M + K - 1);
    int Npad = nextPow2(N + K - 1);

    // Posiciona o kernel com a origem no (0,0) e padding com zeros
    cv::Mat kernel_pad = cv::Mat::zeros(Mpad, Npad, CV_64F);
    for (int i = 0; i < K; i++) {
        for (int j = 0; j < K; j++) {
            kernel_pad.at<double>(i, j) = kernel.at<double>(i, j);
        }
    }

    // FFT da imagem (com padding)
    cv::Mat f_float_pad;
    cv::copyMakeBorder(f_float, f_float_pad, 0, Mpad - M, 0, Npad - N, 
                       cv::BORDER_CONSTANT, cv::Scalar(0));

    cv::Mat F_img, F_kern;
    cv::Mat f_float_pad_32f, kernel_pad_32f;
    f_float_pad.convertTo(f_float_pad_32f, CV_32F);
    kernel_pad.convertTo(kernel_pad_32f, CV_32F);

    cv::dft(f_float_pad_32f, F_img, cv::DFT_COMPLEX_OUTPUT);
    cv::dft(kernel_pad_32f, F_kern, cv::DFT_COMPLEX_OUTPUT);

    // Multiplicação em frequência
    std::vector<cv::Mat> plane_img, plane_kern;
    cv::split(F_img, plane_img);
    cv::split(F_kern, plane_kern);

    cv::Mat prod_real = plane_img[0].mul(plane_kern[0]) - plane_img[1].mul(plane_kern[1]);
    cv::Mat prod_imag = plane_img[0].mul(plane_kern[1]) + plane_img[1].mul(plane_kern[0]);

    std::vector<cv::Mat> prod_planes = {prod_real, prod_imag};
    cv::Mat F_prod;
    cv::merge(prod_planes, F_prod);

    // FFT inversa
    cv::Mat conv_freq_full;
    cv::idft(F_prod, conv_freq_full, cv::DFT_REAL_OUTPUT | cv::DFT_SCALE);

    // Recorte para compensar o deslocamento introduído pelo posicionamento do kernel
    int offset = K / 2;
    conv_freq_full.convertTo(conv_freq_full, CV_64F);

    cv::Mat conv_freq_crop;
    cv::Rect roi(offset, offset, N, M);
    conv_freq_full(roi).copyTo(conv_freq_crop);

    // ── Verificação numérica ──────────────────────────────────────────────────────
    cv::Mat diff;
    cv::absdiff(conv_esp, conv_freq_crop, diff);

    double min_diff, max_diff;
    cv::minMaxLoc(diff, &min_diff, &max_diff);

    cv::Scalar mean_diff = cv::mean(diff);

    std::cout << "Diferença máxima  (|conv_esp - conv_freq|): " 
              << std::scientific << max_diff << std::endl;
    std::cout << "Diferença média   (|conv_esp - conv_freq|): " 
              << std::scientific << mean_diff[0] << std::endl;
    std::cout << "→ Teorema da Convolução verificado numericamente." << std::endl;

    // Normalização para visualização
    cv::Mat conv_esp_vis, conv_freq_vis, diff_vis;
    cv::normalize(conv_esp, conv_esp_vis, 0, 255, cv::NORM_MINMAX, CV_8U);
    cv::normalize(conv_freq_crop, conv_freq_vis, 0, 255, cv::NORM_MINMAX, CV_8U);
    cv::normalize(diff, diff_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Preparando imagens para exibição
    std::vector<mm::Image> images;

    // Imagem original
    mm::Image imgOriginal(img_gray.h, img_gray.w, 1);
    std::memcpy(imgOriginal.data.data(), imgGrayMat.data, imgGrayMat.total() * imgGrayMat.elemSize());
    images.push_back(imgOriginal);

    // Convolução espacial
    mm::Image imgConvEsp(conv_esp_vis.rows, conv_esp_vis.cols, 1);
    std::memcpy(imgConvEsp.data.data(), conv_esp_vis.data, 
                conv_esp_vis.total() * conv_esp_vis.elemSize());
    images.push_back(imgConvEsp);

    // Multiplicação em frequência
    mm::Image imgConvFreq(conv_freq_vis.rows, conv_freq_vis.cols, 1);
    std::memcpy(imgConvFreq.data.data(), conv_freq_vis.data, 
                conv_freq_vis.total() * conv_freq_vis.elemSize());
    images.push_back(imgConvFreq);

    // Diferença
    mm::Image imgDiff(diff_vis.rows, diff_vis.cols, 1);
    std::memcpy(imgDiff.data.data(), diff_vis.data, 
                diff_vis.total() * diff_vis.elemSize());
    images.push_back(imgDiff);

    // Títulos
    char title_diff[256];
    sprintf(title_diff, "Diferença (máx=%.1e)", max_diff);

    std::vector<std::string> titles = {
        "Original",
        "Convolução espacial",
        "Multiplicação em frequência",
        title_diff
    };

    // Exibição
    mm::show(images, MM_OUT, titles, 4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_conv_teorema.cpp -o tmp/fig_05_conv_teorema \
  && ./tmp/fig_05_conv_teorema \
  && test -f "tmp/fig_05_conv_teorema.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_conv_teorema.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_conv_teorema.png"), figsize=(16, 5))

**Figura 5.7:** Verificação do Teorema da Convolução: a diferença pixel a pixel entre a convolução espacial (cv2.filter2D) e a multiplicação em frequência (FFT) é numericamente nula — confirmando a equivalência teórica.


> ### 📝 5.5 Sobre a diferença numérica
>
> A diferença residual da ordem de $10^{-13}$ não viola o Teorema da Convolução, mas reflete **limitações computacionais** inerentes à aritmética de ponto flutuante (precisão dupla, ~$10^{-16}$) e à **ordem de operações** entre os dois métodos:
>
> - **Convolução espacial:** soma ponderada de vizinhos com arredondamentos sucessivos.
> - **Convolução em frequência:** envolve três transformadas FFT e uma multiplicação complexa, sujeita a erros de truncamento e quantização.
>
> Portanto, a igualdade teórica é exata, mas a implementação numérica produz uma diferença praticamente nula (erro relativo < $10^{-12}$), confirmando o teorema dentro da precisão da máquina.

## 5.6 Filtros no Domínio da Frequência

Um filtro no domínio da frequência pode ser interpretado como uma **função de transferência aplicada ao espectro da imagem**. Nessa representação, cada coeficiente de frequência é multiplicado por um valor entre 0 e 1, que determina sua atenuação ou preservação. A forma dessa função define o efeito visual do filtro.

**Corte abrupto e *ringing*.** Filtros ideais com transição instantânea em uma frequência de corte $D_0$ produzem descontinuidades no domínio da frequência. Essa descontinuidade se reflete no domínio espacial como oscilações próximas a bordas, conhecidas como *ringing*. Esse efeito está associado à convolução com funções de suporte infinito no espaço, como a função *sinc*, conforme ilustrado na [Figura 5](#fig-05-conv-teorema-zoom).

**Filtros com transição suave.** Alternativas como os filtros Gaussiano e Butterworth suavizam a transição entre regiões de passagem e rejeição, reduzindo o *ringing*. Em contrapartida, essa suavização implica uma fronteira de separação menos definida entre frequências preservadas e atenuadas.

### 5.6.1 Filtros Passa-Baixa

Filtros passa-baixa atenuam componentes de alta frequência, resultando em suavização da imagem e redução de ruído. Após a centralização do espectro (FFT Shift), a distância de cada ponto ao centro é dada por:

<a id="eq-05-dist-centro"></a>
$$
D(u,v) = \sqrt{\left(u - \tfrac{M}{2}\right)^2 + \left(v - \tfrac{N}{2}\right)^2} \tag{5.5}
$$


**Filtro Ideal (LPFI):**
<a id="eq-05-lpf-ideal"></a>
$$
H_{\text{ideal}}(u,v) =
\begin{cases}
1, & D(u,v) \leq D_0 \\
0, & D(u,v) > D_0
\end{cases} \tag{5.6}
$$


O corte abrupto em $D_0$ introduz descontinuidades no domínio da frequência, resultando em oscilações no domínio espacial conhecidas como *ringing*. Esse efeito está associado à convolução com funções de suporte infinito.

**Filtro Gaussiano (LPFG):**
<a id="eq-05-lpf-gauss"></a>
$$
H_{\text{gauss}}(u,v) = e^{-D^2(u,v)/(2\sigma^2)} \tag{5.7}
$$


A suavidade da função Gaussiana no domínio da frequência evita descontinuidades, o que elimina o *ringing* e produz uma transição gradual entre frequências preservadas e atenuadas.

**Filtro Butterworth (LPFB) de ordem $n$:**
<a id="eq-05-lpf-butterworth"></a>
$$
H_{\text{BW}}(u,v) = \frac{1}{1 + \left[D(u,v)/D_0\right]^{2n}} \tag{5.8}
$$


O parâmetro $n$ controla a suavidade da transição entre passagem e rejeição de frequências. Valores pequenos produzem transições suaves, enquanto valores grandes aproximam o comportamento do filtro ideal, com maior risco de *ringing*. Um exemplo comparativo é apresentado na [Figura 5.8](#fig-05-filtros-passa-baixa).

In [ ]:
from IPython.display import HTML
HTML("""
<div style="font-family:sans-serif;max-width:680px;margin:0 auto;padding:10px;">
<div style="text-align:center;font-size:12px;font-weight:bold;color:#374151;margin-bottom:8px;">
  Perfis dos Filtros Passa-Baixa — comparação visual (D₀ = 30)
</div>
<svg viewBox="0 0 640 200" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#fff;">
  <defs>
    <marker id="ah" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L6,3 L0,6 Z" fill="#9ca3af"/>
    </marker>
  </defs>
  <!-- Grid -->
  <line x1="60" y1="20" x2="60" y2="170" stroke="#e5e7eb" stroke-width="0.8"/>
  <line x1="60" y1="170" x2="610" y2="170" stroke="#e5e7eb" stroke-width="0.8"/>
  <!-- Eixos -->
  <line x1="60" y1="170" x2="605" y2="170" stroke="#9ca3af" stroke-width="1" marker-end="url(#ah)"/>
  <line x1="60" y1="175" x2="60" y2="15" stroke="#9ca3af" stroke-width="1" marker-end="url(#ah)"/>
  <text x="612" y="174" font-size="9" fill="#6b7280">D(u,v)</text>
  <text x="63" y="14" font-size="9" fill="#6b7280">H</text>
  <!-- Rótulos eixo Y -->
  <text x="52" y="35" font-size="8" fill="#6b7280" text-anchor="end">1.0</text>
  <text x="52" y="102" font-size="8" fill="#6b7280" text-anchor="end">0.5</text>
  <text x="52" y="173" font-size="8" fill="#6b7280" text-anchor="end">0.0</text>
  <line x1="57" y1="33" x2="63" y2="33" stroke="#9ca3af" stroke-width="0.8"/>
  <line x1="57" y1="100" x2="63" y2="100" stroke="#9ca3af" stroke-width="0.8"/>
  <!-- D0 marker -->
  <line x1="210" y1="30" x2="210" y2="175" stroke="#d1d5db" stroke-width="0.8" stroke-dasharray="3,3"/>
  <text x="210" y="184" font-size="8" fill="#9ca3af" text-anchor="middle">D₀</text>
  <!-- Filtro Ideal (vermelho) -->
  <polyline points="60,33 210,33 210,170 610,170" fill="none" stroke="#D85A30" stroke-width="2"/>
  <!-- Filtro Gaussiano (verde) -->
  <path d="M60,33 C100,33 130,40 160,60 S210,110 250,140 S320,168 610,170" fill="none" stroke="#1D9E75" stroke-width="2"/>
  <!-- Filtro Butterworth n=2 (azul) -->
  <path d="M60,33 C130,33 165,45 195,75 S225,130 250,148 S310,168 610,170" fill="none" stroke="#534AB7" stroke-width="2"/>
  <!-- Butterworth n=5 (roxo claro) -->
  <path d="M60,33 C170,33 195,40 208,70 S215,140 225,158 S260,170 610,170" fill="none" stroke="#9333ea" stroke-width="1.5" stroke-dasharray="5,3"/>
  <!-- Legenda -->
  <rect x="430" y="25" width="170" height="100" fill="#f9fafb" stroke="#e5e7eb" rx="4"/>
  <line x1="440" y1="45" x2="465" y2="45" stroke="#D85A30" stroke-width="2"/>
  <text x="470" y="49" font-size="9" fill="#374151">Ideal (corte perfeito)</text>
  <text x="470" y="60" font-size="8" fill="#9ca3af">→ ringing nas bordas</text>
  <line x1="440" y1="78" x2="465" y2="78" stroke="#1D9E75" stroke-width="2"/>
  <text x="470" y="82" font-size="9" fill="#374151">Gaussiano</text>
  <text x="470" y="93" font-size="8" fill="#9ca3af">→ sem ringing</text>
  <line x1="440" y1="106" x2="465" y2="106" stroke="#534AB7" stroke-width="2"/>
  <text x="470" y="110" font-size="9" fill="#374151">Butterworth n=2</text>
  <line x1="440" y1="118" x2="453" y2="118" stroke="#9333ea" stroke-width="1.5" stroke-dasharray="4,2"/>
  <text x="470" y="122" font-size="9" fill="#374151">Butterworth n=5</text>
  <!-- Zona de transição -->
  <text x="240" y="85" font-size="8" fill="#6b7280" font-style="italic">zona de</text>
  <text x="240" y="96" font-size="8" fill="#6b7280" font-style="italic">transição</text>
</svg>
<div style="font-size:10px;color:#6b7280;margin-top:6px;text-align:center;">
  À medida que a ordem do Butterworth aumenta, o perfil se aproxima do filtro Ideal — e o ringing aumenta.
</div>
</div>
""")

**Figura 5.8:** Filtros passa-baixa.


### 5.6.2 Filtros Passa-Alta e Passa-Banda

**Filtros passa-alta** podem ser obtidos a partir de um filtro passa-baixa complementar, definido como:

$$
H_{\text{HP}}(u,v) = 1 - H_{\text{LP}}(u,v)
$$

Esse tipo de filtro preserva componentes de alta frequência, realçando bordas e detalhes, enquanto atenua regiões de variação suave.

**Filtros passa-banda** preservam apenas uma faixa intermediária de frequências, limitada por dois raios $D_L$ e $D_H$:

$$
H_{\text{BP}}(u,v) =
H_{\text{LP}}^{(D_H)}(u,v)\cdot
\left[1 - H_{\text{LP}}^{(D_L)}(u,v)\right]
$$

Esse tipo de filtragem é útil quando se deseja remover simultaneamente componentes de baixa e alta frequência, preservando apenas estruturas de escala intermediária.

Uma aplicação importante é a remoção de **ruído periódico**, no qual padrões regulares aparecem como picos localizados no espectro de magnitude. Esses picos podem ser atenuados por meio de filtros *notch* (rejeita-banda), posicionados especificamente nas frequências indesejadas.

Exemplos de filtros no domínio da frequência são apresentados no simulador da [Figura 5.9](#fig-05-sim-05-filtros), [Figura 5](#fig-05-filtros-freq) e [Figura 5.10](#fig-05-filtros-passa-alta).

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-05-filtros" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-05-filtros * { box-sizing: border-box; }
  #sim-05-filtros canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-filtros button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-filtros button:hover { background: #e8dfcf; }
  #sim-05-filtros .sim05_sf_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  
  .sf_legend { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 8px; margin-bottom: 14px; }
  .sf_leg_item { display: flex; align-items: flex-start; gap: 10px; padding: 10px 12px; border-radius: 10px; border: 1px solid #e9e3d3; cursor: pointer; background: #fafaf7; transition: opacity .15s; }
  .sf_leg_item.sf_off { opacity: .35; }
  .sf_leg_swatch { width: 32px; min-width: 32px; height: 3px; margin-top: 8px; border-radius: 2px; }
  .sf_leg_name { font-size: 12.5px; font-weight: 700; }
  .sf_leg_desc { font-size: 10.5px; color: #8a8371; line-height: 1.4; margin-top: 2px; }
  
  .sf_controls { display: flex; align-items: center; gap: 12px; flex-wrap: wrap; margin-bottom: 14px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sf_ctrl_lbl { font-size: 11.5px; color: #5e5a4a; white-space: nowrap; font-weight: 600; }
  .sf_ctrl_val { font-size: 12px; font-weight: 700; min-width: 25px; color: #26241d; font-family: monospace; }
  .sf_radio_grp { display: flex; gap: 12px; }
  .sf_radio_grp label { display: flex; align-items: center; gap: 6px; font-size: 11.5px; color: #5e5a4a; cursor: pointer; font-weight: 600; }
  
  .sf_charts { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 14px; }
  .sf_card { background: #fafaf7; border-radius: 12px; padding: 12px; border: 1px solid #e9e3d3; }
  .sf_card_lbl { font-size: 10.5px; color: #5e5a4a; margin-bottom: 8px; font-weight: 700; text-transform: uppercase; letter-spacing: .04em; }
  .sf_stats { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 14px; }
  .sf_pill { font-size: 10.5px; padding: 4px 10px; border-radius: 8px; background: #fafaf7; color: #5e5a4a; border: 1px solid #e9e3d3; font-weight: 600; }
  .sf_pill b { color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎛️ Simulador: Filtros no Domínio da Frequência</span>
  <span class="sim05_sf_pill">Passa-Baixa / Passa-Alta</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <div class="sf_legend" id="sf_legend"></div>

  <div class="sf_controls">
    <span class="sf_ctrl_lbl">Frequência de corte D₀</span>
    <input type="range" id="sf_d0" min="5" max="100" value="30" step="1" style="flex:1;min-width:120px;max-width:240px;accent-color:#2980b9;cursor:pointer;">
    <span class="sf_ctrl_val" id="sf_d0v">30</span>
    <span class="sf_ctrl_lbl" style="margin-left:8px">Tipo de filtro</span>
    <div class="sf_radio_grp">
      <label><input type="radio" name="sf_ft" value="lp" checked style="cursor:pointer;"> passa-baixa</label>
      <label><input type="radio" name="sf_ft" value="hp" style="cursor:pointer;"> passa-alta</label>
    </div>
  </div>

  <div class="sf_charts">
    <div class="sf_card">
      <div class="sf_card_lbl">Resposta H(D)</div>
      <canvas id="sf_c1" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Espectro filtrado |F · H|</div>
      <canvas id="sf_c2" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Sinal 1D — original vs filtrado</div>
      <canvas id="sf_c3" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Energia retida por banda (%)</div>
      <canvas id="sf_c4" style="width:100%;display:block"></canvas>
    </div>
  </div>

  <div class="sf_stats" id="sf_stats"></div>

</div>
</div>

<script>
(function(){
  function initSim05Filtros(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    var SF = [
      {key:'ideal', name:'Ideal',          desc:'Corte perfeito em D₀ — causa ringing',       color:'#c0392b', dash:null },
      {key:'gauss', name:'Gaussiano',      desc:'Transição suave — sem ringing',                 color:'#27ae60', dash:[6,3]},
      {key:'bw2',   name:'Butterworth n=2', desc:'Compromisso: suave com banda controlável',       color:'#2980b9', dash:[4,2]},
      {key:'bw5',   name:'Butterworth n=5', desc:'Aproxima o ideal mantendo transição suave',    color:'#b9770e', dash:[2,2]},
    ];

    var sf_D0 = 30, sf_hp = false;
    var sf_on = {ideal:true, gauss:true, bw2:true, bw5:true};

    function sf_H(D, key, d0, hp){
      var h;
      if(key === 'ideal')      h = D <= d0 ? 1 : 0;
      else if(key === 'gauss') h = Math.exp(-D*D/(2*d0*d0));
      else if(key === 'bw2')   h = 1/(1+Math.pow(D/d0,4));
      else                     h = 1/(1+Math.pow(D/d0,10));
      return hp ? 1-h : h;
    }

    function sf_setup(id, h){
      var c = root.querySelector('#' + id);
      var w = c.parentElement.clientWidth - 24;
      if(w < 100) w = 280;
      c.width  = w;
      c.height = h || 180;
      return {c:c, ctx:c.getContext('2d'), w:c.width, h:c.height};
    }

    function sf_axes(ctx, w, h, pad, xmax, ymin, ymax, xlabel, ylabel){
      var l=pad.l, r=pad.r, t=pad.t, b=pad.b;
      ctx.clearRect(0,0,w,h);

      ctx.strokeStyle='rgba(0,0,0,.06)'; ctx.lineWidth=0.5;
      var nx=4, ny=4;
      for(var i=0; i<=nx; i++){
        var x = l + (w-l-r) * i / nx;
        ctx.beginPath(); ctx.moveTo(x, t); ctx.lineTo(x, h-b); ctx.stroke();
      }
      for(var j=0; j<=ny; j++){
        var y = t + (h-t-b) * j / ny;
        ctx.beginPath(); ctx.moveTo(l, y); ctx.lineTo(w-r, y); ctx.stroke();
      }

      ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=1;
      ctx.beginPath(); ctx.moveTo(l, t); ctx.lineTo(l, h-b); ctx.lineTo(w-r, h-b); ctx.stroke();

      ctx.fillStyle='#8a8371'; ctx.font='10px monospace'; ctx.textAlign='center';
      for(var i=0; i<=nx; i++){
        var x = l + (w-l-r) * i / nx;
        var val = Math.round(xmax * i / nx);
        ctx.fillText(val, x, h-b+12);
      }
      ctx.textAlign='right';
      for(var j=0; j<=ny; j++){
        var y = t + (h-t-b) * j / ny;
        var val = ymax - (ymax-ymin) * j / ny;
        ctx.fillText(val.toFixed(2), l-4, y+3);
      }

      ctx.fillStyle='#5e5a4a'; ctx.font='10.5px Inter,sans-serif'; ctx.textAlign='center';
      ctx.fillText(xlabel, l+(w-l-r)/2, h-2);
      ctx.save(); ctx.translate(11, t+(h-t-b)/2); ctx.rotate(-Math.PI/2);
      ctx.fillText(ylabel, 0, 0); ctx.restore();

      var d0x = l + (sf_D0/xmax)*(w-l-r);
      if(d0x > l && d0x < w-r){
        ctx.strokeStyle='#b9770e'; ctx.lineWidth=1; ctx.setLineDash([4,3]);
        ctx.beginPath(); ctx.moveTo(d0x, t); ctx.lineTo(d0x, h-b); ctx.stroke();
        ctx.setLineDash([]);
        ctx.fillStyle='#b9770e'; ctx.font='10px monospace'; ctx.textAlign='center';
        ctx.fillText('D₀', d0x, t-2);
      }

      return {
        toX: function(v){ return l + (v/xmax)*(w-l-r); },
        toY: function(v){ return (h-b) - (v-ymin)/(ymax-ymin)*(h-t-b); }
      };
    }

    function sf_line(ctx, pts, color, dash, fill){
      if(!pts.length) return;
      ctx.strokeStyle = color; ctx.lineWidth = 2;
      ctx.setLineDash(dash || []);
      if(fill){
        ctx.fillStyle = color.replace(')', ', 0.12)').replace('rgb', 'rgba');
        ctx.beginPath();
        ctx.moveTo(pts[0].x, pts[0].baseY);
        pts.forEach(function(p){ ctx.lineTo(p.x, p.y); });
        ctx.lineTo(pts[pts.length-1].x, pts[pts.length-1].baseY);
        ctx.closePath(); ctx.fill();
      }
      ctx.beginPath();
      pts.forEach(function(p, i){ i===0 ? ctx.moveTo(p.x, p.y) : ctx.lineTo(p.x, p.y); });
      ctx.stroke();
      ctx.setLineDash([]);
    }

    function sf_drawProfile(){
      var s = sf_setup('sf_c1'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var ax = sf_axes(ctx, s.w, s.h, pad, 300, 0, 1, 'D(u,v)', 'H(D)');
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var d=0; d<=300; d+=2){
          pts.push({x: ax.toX(d), y: ax.toY(sf_H(d, f.key, sf_D0, sf_hp)), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, false);
      });
    }

    function sf_drawSpectrum(){
      var s = sf_setup('sf_c2'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var ax = sf_axes(ctx, s.w, s.h, pad, 300, 0, 1, 'D(u,v)', '|F·H|');
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var d=0; d<=300; d+=2){
          var y = Math.max(0, Math.exp(-d*d/(2*80*80)) * sf_H(d, f.key, sf_D0, sf_hp));
          pts.push({x: ax.toX(d), y: ax.toY(y), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, true);
      });
    }

    function sf_drawSignal(){
      var s = sf_setup('sf_c3'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var N = 128;
      var all = [];
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        for(var t=0; t<N; t++){
          var v = 0;
          for(var fr=1; fr<80; fr++) v += sf_H(fr, f.key, sf_D0, sf_hp) * Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
          all.push(v/7);
        }
      });
      for(var t=0; t<N; t++){
        var v = 0;
        for(var fr=1; fr<60; fr++) v += Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
        all.push(v/7);
      }
      var ymin = Math.min.apply(null, all)*1.15, ymax = Math.max.apply(null, all)*1.15;
      if(ymax - ymin < 0.1){ymin = -0.5; ymax = 0.5;}
      var ax = sf_axes(ctx, s.w, s.h, pad, N, ymin, ymax, 'amostras', 'amp');

      var orig = [];
      for(var t=0; t<N; t++){
        var v = 0;
        for(var fr=1; fr<60; fr++) v += Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
        orig.push({x: ax.toX(t), y: ax.toY(v/7), baseY: ax.toY(0)});
      }
      ctx.globalAlpha = 0.4;
      sf_line(ctx, orig, '#8a8371', [4,3], false);
      ctx.globalAlpha = 1;

      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var t=0; t<N; t++){
          var v = 0;
          for(var fr=1; fr<80; fr++) v += sf_H(fr, f.key, sf_D0, sf_hp) * Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
          pts.push({x: ax.toX(t), y: ax.toY(v/7), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, false);
      });
    }

    function sf_drawEnergy(){
      var s = sf_setup('sf_c4'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:32};
      var bands = [[0,20],[20,40],[40,60],[60,80],[80,100]];
      var labels = ['0–20','20–40','40–60','60–80','80–100'];
      var active = SF.filter(function(f){ return sf_on[f.key]; });
      if(active.length === 0) return;

      ctx.clearRect(0, 0, s.w, s.h);
      ctx.strokeStyle = 'rgba(0,0,0,.06)'; ctx.lineWidth = 0.5;
      for(var j=0; j<=4; j++){
        var y = pad.t + (s.h-pad.t-pad.b) * j / 4;
        ctx.beginPath(); ctx.moveTo(pad.l, y); ctx.lineTo(s.w-pad.r, y); ctx.stroke();
      }
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 1;
      ctx.beginPath(); ctx.moveTo(pad.l, pad.t); ctx.lineTo(pad.l, s.h-pad.b);
      ctx.lineTo(s.w-pad.r, s.h-pad.b); ctx.stroke();

      ctx.fillStyle = '#8a8371'; ctx.font = '10px monospace'; ctx.textAlign = 'right';
      for(var j=0; j<=4; j++){
        var y = pad.t + (s.h-pad.t-pad.b) * j / 4;
        ctx.fillText((100 - j*25) + '%', pad.l-4, y+3);
      }

      var bw = (s.w - pad.l - pad.r) / bands.length;
      var gw = bw * 0.12, fw = (bw - gw*(active.length+1)) / active.length;
      if(fw < 2) fw = 2;

      bands.forEach(function(band, bi){
        var energy = function(key){
          var e = 0, n = 0;
          for(var d = band[0]; d < band[1]; d++){ e += Math.pow(sf_H(d, key, sf_D0, sf_hp), 2); n++; }
          return n > 0 ? Math.round(e/n * 100) : 0;
        };
        var bx = pad.l + bi * bw;
        active.forEach(function(f, fi){
          var val = energy(f.key);
          var x = bx + gw*(fi+1) + fw*fi;
          var barH = (val/100) * (s.h - pad.t - pad.b);
          var y = (s.h - pad.b) - barH;
          ctx.fillStyle = f.color + 'bb';
          ctx.fillRect(x, y, fw, barH);
          ctx.strokeStyle = f.color; ctx.lineWidth = 0.5;
          ctx.strokeRect(x, y, fw, barH);
        });
        ctx.fillStyle = '#8a8371'; ctx.font = '10px monospace'; ctx.textAlign = 'center';
        ctx.fillText(labels[bi], bx + bw/2, s.h - pad.b + 14);
      });

      ctx.fillStyle = '#5e5a4a'; ctx.font = '10.5px Inter,sans-serif'; ctx.textAlign = 'center';
      ctx.save(); ctx.translate(11, pad.t + (s.h-pad.t-pad.b)/2); ctx.rotate(-Math.PI/2);
      ctx.fillText('energia (%)', 0, 0); ctx.restore();
      ctx.fillText('banda de frequência', pad.l + (s.w-pad.l-pad.r)/2, s.h - 1);
    }

    function sf_buildLegend(){
      var el = root.querySelector('#sf_legend');
      el.innerHTML = '';
      SF.forEach(function(f){
        var d = document.createElement('div');
        d.className = 'sf_leg_item' + (sf_on[f.key] ? '' : ' sf_off');
        d.style.borderColor = sf_on[f.key] ? f.color : '#e9e3d3';
        var swatchStyle = 'background:' + f.color;
        if(f.dash){
          var seg = f.dash[0], gap = f.dash[1];
          swatchStyle = 'background:repeating-linear-gradient(90deg,' + f.color + ' 0 ' + seg + 'px,transparent ' + seg + 'px ' + (seg+gap) + 'px)';
        }
        d.innerHTML =
          '<div class="sf_leg_swatch" style="' + swatchStyle + '"></div>' +
          '<div><div class="sf_leg_name" style="color:' + f.color + '">' + f.name + '</div>' +
          '<div class="sf_leg_desc">' + f.desc + '</div></div>';
        d.addEventListener('click', function(){
          sf_on[f.key] = !sf_on[f.key]; sf_buildLegend(); sf_draw();
        });
        el.appendChild(d);
      });
    }

    function sf_stats(){
      var el = root.querySelector('#sf_stats');
      el.innerHTML = SF.filter(function(f){ return sf_on[f.key]; }).map(function(f){
        var h50 = sf_H(sf_D0, f.key, sf_D0, sf_hp).toFixed(2);
        var en = Math.round(function(){
          var s = 0;
          for(var i=0; i<200; i++) s += Math.pow(sf_H(i*0.5, f.key, sf_D0, sf_hp), 2);
          return s/200;
        }() * 100);
        return '<div class="sf_pill" style="border-color:' + f.color + '55">' +
          '<b style="color:' + f.color + '">' + f.name + '</b>' +
          ' H(D₀)=<b>' + h50 + '</b> &middot; energia=<b>' + en + '%</b></div>';
      }).join('');
    }

    function sf_draw(){
      sf_drawProfile();
      sf_drawSpectrum();
      sf_drawSignal();
      sf_drawEnergy();
      sf_stats();
    }

    root.querySelector('#sf_d0').addEventListener('input', function(){
      sf_D0 = +this.value;
      root.querySelector('#sf_d0v').textContent = sf_D0;
      sf_draw();
    });

    root.querySelectorAll('input[name="sf_ft"]').forEach(function(r){
      r.addEventListener('change', function(e){
        sf_hp = e.target.value === 'hp';
        sf_draw();
      });
    });

    sf_buildLegend();
    sf_draw();
    window.addEventListener('resize', sf_draw);
  }

  function tryInitSim05Filtros(){
    var root = document.getElementById('sim-05-filtros');
    if (root) initSim05Filtros(root); else setTimeout(tryInitSim05Filtros, 200);
  }
  tryInitSim05Filtros();
})();
</script>
""")

**Figura 5.9:** Simulador interativo de filtros no domínio da frequência.


In [ ]:
# Ainda não portado para esta linguagem nesta versão — referência conceitual em Python.

# Filtro passa-alta: complemento do passa-baixa Gaussiano
# Reutiliza aplicar_filtro_freq() definida na célula anterior
H_alta   = 1 - H_gauss
img_alta = aplicar_filtro_freq(img_gray, H_alta)

mm.show(
    [img_gray, img_alta],
    titles=["Original", "Passa-alta Gaussiano ($D_0=30$)"],
    cols=2
)

**Figura 5.10:** Filtro passa-alta Gaussiano. (a) Original; (b) Filtro passa-alta (D₀=30) - as bordas das moedas e fundo texturizado são realçados.


### 5.6.3 Remoção de Ruído Periódico

Ruído periódico — associado a interferências elétricas, padrões regulares de sensores ou artefatos de digitalização — aparece no espectro de Fourier como **picos pontuais simétricos em torno do centro**.

O filtro **rejeita-banda (*notch*)** atenua seletivamente essas frequências, preservando as demais componentes da imagem. Um exemplo de aplicação é apresentado na [Figura 5.11](#fig-05-ruido-periodico).

In [ ]:
%%writefile tmp/fig_05_ruido_periodico.cpp
#define MM_OUT "tmp/fig_05_ruido_periodico.png"
//| label: fig-05-ruido-periodico
//| fig-cap: "Remoção de ruído periódico via filtro *notch* no domínio da frequência: (a) imagem com ruído senoidal, (b) espectro mostrando os picos do ruído, (c) máscara *notch* centrada nos picos, (d) imagem restaurada."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <iostream>
#include <vector>
#include <string>
#include <cmath>
#include <algorithm>
#include "morph.hpp"

// Helper para fftshift (trocar quadrantes)
void fftShift(cv::Mat& mat) {
    int cx = mat.cols / 2;
    int cy = mat.rows / 2;
    cv::Mat q0(mat, cv::Rect(0, 0, cx, cy));   // Top-Left
    cv::Mat q1(mat, cv::Rect(cx, 0, cx, cy));  // Top-Right
    cv::Mat q2(mat, cv::Rect(0, cy, cx, cy));  // Bottom-Left
    cv::Mat q3(mat, cv::Rect(cx, cy, cx, cy)); // Bottom-Right

    cv::Mat tmp;
    q0.copyTo(tmp);
    q3.copyTo(q0);
    tmp.copyTo(q3);

    q1.copyTo(tmp);
    q2.copyTo(q1);
    tmp.copyTo(q2);
}

// Helper para ifftshift
void ifftShift(cv::Mat& mat) {
    fftShift(mat); // Mesma operação para tamanhos pares
}

// Função para suprimir pico (zerar disco)
cv::Mat suprimir_pico(const cv::Mat& mask, int cy, int cx, int r) {
    cv::Mat result = mask.clone();
    for (int y = 0; y < mask.rows; ++y) {
        for (int x = 0; x < mask.cols; ++x) {
            double dist = std::sqrt(std::pow(double(y - cy), 2) + std::pow(double(x - cx), 2));
            if (dist <= r) {
                result.at<double>(y, x) = 0.0;
            }
        }
    }
    return result;
}

// SSIM implementação com GaussianBlur (Wang et al.)
cv::Scalar computeSSIM(const cv::Mat& img1, const cv::Mat& img2) {
    const double C1 = 6.5025, C2 = 58.5225;
    cv::Mat I1, I2;
    img1.convertTo(I1, CV_32F);
    img2.convertTo(I2, CV_32F);

    cv::Mat I1_2 = I1.mul(I1);
    cv::Mat I2_2 = I2.mul(I2);
    cv::Mat I1_I2 = I1.mul(I2);

    cv::Mat mu1, mu2;
    cv::GaussianBlur(I1, mu1, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(I2, mu2, cv::Size(11, 11), 1.5);

    cv::Mat mu1_2 = mu1.mul(mu1);
    cv::Mat mu2_2 = mu2.mul(mu2);
    cv::Mat mu1_mu2 = mu1.mul(mu2);

    cv::Mat sigma1_2, sigma2_2, sigma12;
    cv::GaussianBlur(I1_2, sigma1_2, cv::Size(11, 11), 1.5);
    sigma1_2 -= mu1_2;
    cv::GaussianBlur(I2_2, sigma2_2, cv::Size(11, 11), 1.5);
    sigma2_2 -= mu2_2;
    cv::GaussianBlur(I1_I2, sigma12, cv::Size(11, 11), 1.5);
    sigma12 -= mu1_mu2;

    cv::Mat t1, t2, t3;
    t1 = 2 * mu1_mu2 + C1;
    t2 = 2 * sigma12 + C2;
    t3 = t1.mul(t2);

    cv::Mat t4, t5, t6;
    t4 = mu1_2 + mu2_2 + C1;
    t5 = sigma1_2 + sigma2_2 + C2;
    t6 = t4.mul(t5);

    cv::Mat ssim_map;
    cv::divide(t3, t6, ssim_map);

    return cv::mean(ssim_map);
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_19.png");
// [pdi:state-io:end]

    // Converter mm::Image para cv::Mat
    int h_img = img_gray.h;
    int w_img = img_gray.w;
    cv::Mat img_gray_mat(h_img, w_img, CV_8UC1, img_gray.data.data());

    // ── Imagem com ruído periódico sintético ──────────────────────────────────────
    std::vector<double> x(w_img), y(h_img);
    for (int i = 0; i < w_img; ++i) x[i] = i;
    for (int i = 0; i < h_img; ++i) y[i] = i;

    // Usar frequências inteiras e consistentes (importante para restauração perfeita)
    int u0 = 20, v0 = 20;  // frequências exatas do ruído

    cv::Mat img_ruidosa_float(h_img, w_img, CV_64F);
    for (int j = 0; j < h_img; ++j) {
        for (int i = 0; i < w_img; ++i) {
            double ruido = 40 * std::sin(2 * M_PI * (u0 * x[i] / w_img + v0 * y[j] / h_img));
            double val = double(img_gray_mat.at<uchar>(j, i)) + ruido;
            img_ruidosa_float.at<double>(j, i) = std::max(0.0, std::min(255.0, val));
        }
    }

    cv::Mat img_ruidosa;
    img_ruidosa_float.convertTo(img_ruidosa, CV_8UC1);

    // ── Espectro da imagem ruidosa ───────────────────────────────────────────────
    cv::Mat img_ruidosa_64F;
    img_ruidosa.convertTo(img_ruidosa_64F, CV_64F);

    cv::Mat F_r;
    cv::dft(img_ruidosa_64F, F_r, cv::DFT_COMPLEX_OUTPUT);

    cv::Mat F_r_shifted[2];
    cv::Mat planes[2];
    cv::split(F_r, planes);
    fftShift(planes[0]);
    fftShift(planes[1]);
    planes[0].copyTo(F_r_shifted[0]);
    planes[1].copyTo(F_r_shifted[1]);

    cv::Mat mag_r;
    cv::magnitude(F_r_shifted[0], F_r_shifted[1], mag_r);
    cv::Mat mag_r_log;
    cv::log(1.0 + mag_r, mag_r_log);

    cv::Mat mag_vis;
    cv::normalize(mag_r_log, mag_vis, 0, 255, cv::NORM_MINMAX, CV_8UC1);

    // ── Máscara notch ────────────────────────────────────────────────────────────
    cv::Mat mascara = cv::Mat::ones(h_img, w_img, CV_64F);
    int r_notch = 8;  // raio do notch (ajuste fino se necessário)

    // Coordenadas centrais
    int cy = h_img / 2, cx = w_img / 2;

    // Suprimir os 4 picos simétricos (importante!)
    int deltas[4][2] = {{v0, u0}, {-v0, -u0}, {v0, -u0}, {-v0, u0}};
    for (auto& d : deltas) {
        mascara = suprimir_pico(mascara, cy + d[0], cx + d[1], r_notch);
    }

    cv::Mat mascara_vis;
    mascara.convertTo(mascara_vis, CV_8UC1, 255.0);

    // ── Filtragem e reconstrução ─────────────────────────────────────────────────
    // Multiplicar espectro pela máscara (partes real e imaginária)
    cv::Mat F_filtrada_real = F_r_shifted[0].mul(mascara);
    cv::Mat F_filtrada_imag = F_r_shifted[1].mul(mascara);

    // ifftshift
    fftShift(F_filtrada_real);
    fftShift(F_filtrada_imag);

    // Reconstruir espectro complexo
    cv::Mat F_filtrada[] = {F_filtrada_real, F_filtrada_imag};
    cv::Mat F_complex;
    cv::merge(F_filtrada, 2, F_complex);

    cv::Mat img_rest;
    cv::idft(F_complex, img_rest, cv::DFT_SCALE | cv::DFT_REAL_OUTPUT);

    cv::Mat img_rest_vis;
    cv::normalize(img_rest, img_rest_vis, 0, 255, cv::NORM_MINMAX, CV_8UC1);

    // Avaliação
    double psnr = cv::PSNR(img_gray_mat, img_rest_vis);

    // SSIM (implementação manual)
    cv::Scalar ssim_val = computeSSIM(img_gray_mat, img_rest_vis);

    std::cout << "PSNR (original vs restaurada): " << psnr << " dB" << std::endl;
    std::cout << "SSIM: " << ssim_val[0] << std::endl;

    // ── Visualização ─────────────────────────────────────────────────────────────
    std::vector<mm::Image> results;

    // Converter cv::Mat para mm::Image
    mm::Image img_ruidosa_mm(img_ruidosa.rows, img_ruidosa.cols, 1);
    std::memcpy(img_ruidosa_mm.data.data(), img_ruidosa.data, img_ruidosa.total() * img_ruidosa.elemSize());

    mm::Image mag_vis_mm(mag_vis.rows, mag_vis.cols, 1);
    std::memcpy(mag_vis_mm.data.data(), mag_vis.data, mag_vis.total() * mag_vis.elemSize());

    mm::Image mascara_vis_mm(mascara_vis.rows, mascara_vis.cols, 1);
    std::memcpy(mascara_vis_mm.data.data(), mascara_vis.data, mascara_vis.total() * mascara_vis.elemSize());

    mm::Image img_rest_vis_mm(img_rest_vis.rows, img_rest_vis.cols, 1);
    std::memcpy(img_rest_vis_mm.data.data(), img_rest_vis.data, img_rest_vis.total() * img_rest_vis.elemSize());

    results.push_back(img_ruidosa_mm);
    results.push_back(mag_vis_mm);
    results.push_back(mascara_vis_mm);
    results.push_back(img_rest_vis_mm);

    std::vector<std::string> titles = {
        "Com ruído periódico",
        "Espectro (log)",
        "Máscara notch",
        "Restaurada (PSNR=" + std::to_string(psnr).substr(0, 4) + " dB)"
    };

    mm::show(results, MM_OUT, titles, 4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_ruido_periodico.cpp -o tmp/fig_05_ruido_periodico \
  && ./tmp/fig_05_ruido_periodico \
  && test -f "tmp/fig_05_ruido_periodico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_ruido_periodico.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_ruido_periodico.png"), figsize=(16, 4))

**Figura 5.11:** Remoção de ruído periódico via filtro *notch* no domínio da frequência: (a) imagem com ruído senoidal, (b) espectro mostrando os picos do ruído, (c) máscara *notch* centrada nos picos, (d) imagem restaurada.


### 📌 Síntese — Filtros Espectrais

| Filtro | Efeito visual | Artefato | Uso |
|:---|:---|:---|:---|
| Passa-baixa ideal | Suavização intensa | *Ringing* | Ilustrativo |
| Passa-baixa Gaussiano | Suavização suave | Não apresenta *ringing* | Suavização geral |
| Passa-baixa Butterworth | Suavização controlada | *Ringing* (ordens altas) | Compromisso entre suavização e seletividade |
| Passa-alta | Realce de bordas | Amplificação de ruído | Detecção de contornos |
| *Notch* | Remoção seletiva de frequências | Possíveis distorções locais | Remoção de ruído periódico |

O projeto de filtros no domínio da frequência consiste na definição de máscaras espectrais. Entretanto, efeitos no domínio espacial, como *ringing* e borramento, emergem diretamente dessas escolhas no espectro.

## 5.7 *Wavelets* e Multirresolução

A Transformada de Fourier decompõe o sinal em frequências **globais**: cada coeficiente $F(u,v)$ recebe contribuições de toda a imagem, sem informação explícita sobre a localização espacial dessas frequências. Assim, estruturas localizadas, como bordas, são representadas de forma distribuída no espectro.

As ***wavelets* (ondaletas)** superam essa limitação ao utilizar funções base **localizadas no espaço**, que podem ser deslocadas e escaladas. Essas funções possuem **suporte compacto**, isto é, são diferentes de zero apenas em uma região finita do domínio, permitindo uma representação simultânea em termos de **frequência e localização espacial**.

### 5.7.1 O Limite da Transformada de Fourier: localização espacial

A Transformada de Fourier descreve com precisão **quais frequências estão presentes** em um sinal, mas não representa explicitamente **onde essas frequências ocorrem no espaço**.

No experimento apresentado na [Figura 5.12](#fig-05-fracasso-fourier), duas imagens com estruturas localizadas em posições diferentes produzem espectros de magnitude praticamente idênticos. Isso ocorre porque a representação de Fourier é global: cada coeficiente recebe contribuição de toda a imagem.

Como consequência, o espectro de magnitude não representa explicitamente a localização de bordas ou outras estruturas, apenas a distribuição das frequências presentes. Essa limitação motivou o desenvolvimento de representações multirresolução, como a Transformada *Wavelet* Discreta (DWT), capazes de descrever simultaneamente a frequência e a localização espacial das estruturas da imagem.

In [ ]:
%%writefile tmp/fig_05_fracasso_fourier.cpp
#define MM_OUT "tmp/fig_05_fracasso_fourier.png"
//| label: fig-05-fracasso-fourier
//| fig-cap: "Fourier global é cego para a posição. Os espectros não dizem onde as bordas estão."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include <algorithm>
#include "morph.hpp"

// Helper: shift FFT quadrants (fftshift equivalent)
void fftShift(cv::Mat& mat) {
    int cx = mat.cols / 2;
    int cy = mat.rows / 2;
    cv::Mat q0(mat, cv::Rect(0, 0, cx, cy));   // Top-Left
    cv::Mat q1(mat, cv::Rect(cx, 0, cx, cy));  // Top-Right
    cv::Mat q2(mat, cv::Rect(0, cy, cx, cy));  // Bottom-Left
    cv::Mat q3(mat, cv::Rect(cx, cy, cx, cy)); // Bottom-Right

    cv::Mat tmp;
    q0.copyTo(tmp);
    q3.copyTo(q0);
    tmp.copyTo(q3);

    q1.copyTo(tmp);
    q2.copyTo(q1);
    tmp.copyTo(q2);
}

// Helper to compute log magnitude spectrum from a 2D signal
cv::Mat computeMagSpectrum(const cv::Mat& signal) {
    // Convert to float for DFT
    cv::Mat floatImg;
    signal.convertTo(floatImg, CV_32F);

    // Compute 2D DFT
    cv::Mat dftImg;
    cv::dft(floatImg, dftImg, cv::DFT_COMPLEX_OUTPUT);

    // Split into real and imaginary parts
    std::vector<cv::Mat> planes;
    cv::split(dftImg, planes);

    // Compute magnitude
    cv::Mat mag;
    cv::magnitude(planes[0], planes[1], mag);

    // Shift quadrants
    fftShift(mag);

    // Apply log1p
    cv::Mat logMag;
    cv::log(mag + 1.0, logMag);

    // Normalize to 0-255 for visualization
    cv::Mat normalized;
    cv::normalize(logMag, normalized, 0, 255, cv::NORM_MINMAX, CV_8U);

    return normalized;
}

int main() {
    // Create the test signals
    cv::Mat img_sinal1 = cv::Mat::zeros(128, 128, CV_8UC1);
    img_sinal1.colRange(20, 25).setTo(255);
    img_sinal1.rowRange(100, 105).setTo(255);

    cv::Mat img_sinal2 = cv::Mat::zeros(128, 128, CV_8UC1);
    img_sinal2.colRange(90, 95).setTo(255);
    img_sinal2.rowRange(30, 35).setTo(255);

    // Compute magnitude spectra
    cv::Mat mag1 = computeMagSpectrum(img_sinal1);
    cv::Mat mag2 = computeMagSpectrum(img_sinal2);

    // Convert cv::Mat to mm::Image for display
    mm::Image out1(128, 128, 1);
    std::memcpy(out1.data.data(), img_sinal1.data, out1.data.size());

    mm::Image out2(128, 128, 1);
    std::memcpy(out2.data.data(), mag1.data, out2.data.size());

    mm::Image out3(128, 128, 1);
    std::memcpy(out3.data.data(), img_sinal2.data, out3.data.size());

    mm::Image out4(128, 128, 1);
    std::memcpy(out4.data.data(), mag2.data, out4.data.size());

    // Display results
    mm::show({out1, out2, out3, out4},
             MM_OUT,
             {"Sinal A", "Espectro A", "Sinal B (Deslocado)", "Espectro B"},
             4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_fracasso_fourier.cpp -o tmp/fig_05_fracasso_fourier \
  && ./tmp/fig_05_fracasso_fourier \
  && test -f "tmp/fig_05_fracasso_fourier.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_fracasso_fourier.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_fracasso_fourier.png"), figsize=(14, 4))

**Figura 5.12:** Fourier global é cego para a posição. Os espectros não dizem onde as bordas estão.


### 5.7.2 Transformada *Wavelet* Discreta 2D

A Transformada *Wavelet* Discreta (DWT) aplica, separadamente nas direções horizontal e vertical, dois filtros complementares: um **passa-baixa** $h$ (aproximação) e um **passa-alta** $g$ (detalhes), seguidos de subamostragem por um fator de 2 em cada dimensão. Esse processo produz quatro subbandas, cujos nomes indicam a combinação dos filtros aplicados em cada direção (L = *Low-pass*, passa-baixa; H = *High-pass*, passa-alta). As características de cada subbanda são resumidas na [Tabela 5.3](#tbl-05-dwt-subbandas).

$$
\text{DWT}(f)=\{\underbrace{\text{LL}}_{\text{aprox.}},\;
\underbrace{\text{LH}}_{\text{detalhes horizontais}},\;
\underbrace{\text{HL}}_{\text{detalhes verticais}},\;
\underbrace{\text{HH}}_{\text{detalhes diagonais}}\}.
$$

<a id="tbl-05-dwt-subbandas"></a>

**Tabela 5.3:** Subbandas produzidas pela Transformada *Wavelet* Discreta 2D (DWT), indicando os filtros aplicados em cada direção e o conteúdo predominante de cada componente.

| Subbanda | Filtros aplicados | Conteúdo visual |
|:---|:---:|:---|
| **LL** | baixa × baixa | Aproximação da imagem (versão suavizada e reduzida) |
| **LH** | baixa × alta | Bordas horizontais e variações verticais |
| **HL** | alta × baixa | Bordas verticais e variações horizontais |
| **HH** | alta × alta | Detalhes diagonais e texturas |


A decomposição pode ser aplicada recursivamente sobre a subbanda LL, gerando uma representação multirresolução. Após $J$ níveis, obtém-se uma estrutura com $3J+1$ subbandas, em que cada novo nível reduz a resolução da componente de aproximação.

> ### 📝 Conexão com CNNs
>
> A decomposição multirresolução das *wavelets* possui uma relação conceitual com as representações hierárquicas utilizadas em redes neurais convolucionais (CNNs). Em ambos os casos, sucessivas etapas de filtragem e redução de resolução produzem descrições cada vez mais abstratas da imagem. Entretanto, as ***wavelets* utilizam filtros matematicamente definidos e reconstruíveis**, enquanto as **CNNs aprendem seus filtros durante o treinamento**.

### 5.7.3 Famílias de *Wavelets*

Diferentes famílias de *wavelets* apresentam compromissos distintos entre **suporte espacial**, suavidade e capacidade de compressão. O **suporte** corresponde à extensão da função *wavelet* no domínio espacial: quanto menor o suporte, mais localizada é a função; quanto maior, mais suave tende a ser sua representação, porém com maior custo computacional. A [Tabela 5.4](#tbl-05-wavelet-familias) compara algumas das famílias mais utilizadas.

<a id="tbl-05-wavelet-familias"></a>

**Tabela 5.4:** Comparação entre famílias de *wavelets*, destacando o comprimento do suporte, o número de momentos nulos, a simetria e aplicações típicas.

| *Wavelet* | Comprimento do suporte | Momentos nulos | Simetria | Uso típico |
|:---|:---:|:---:|:---:|:---|
| Haar | 2 | 1 | Assimétrica | Introdução e análise básica |
| Daubechies db4 | 8 | 4 | Assimétrica | Compressão e análise geral |
| Symlet sym4 | 8 | 4 | Quase simétrica | Reconstrução de sinais |
| Biortogonal 5/3 | 5/3 | 2/2 | Simétrica | JPEG 2000 sem perda |
| Biortogonal 9/7 | 9/7 | 4/4 | Simétrica | JPEG 2000 com perda |


Os **momentos nulos** medem a capacidade da *wavelet* de representar regiões suaves da imagem com poucos coeficientes diferentes de zero. Uma *wavelet* com $p$ momentos nulos anula exatamente polinômios de grau até $p-1$. Em consequência, quanto maior o número de momentos nulos, maior tende a ser a eficiência de compressão em regiões homogêneas, embora isso geralmente implique funções com suporte mais longo.

A [Figura 5](#fig-05-wavelet-functions) apresenta as funções de base (*wavelets*) $\psi(t)$ no domínio espacial. Essas funções possuem **suporte compacto**, isto é, são diferentes de zero apenas em uma região finita do domínio, ao contrário das senoides da Transformada de Fourier, que se estendem por todo o domínio.

O diagrama da [Figura 5.13](#fig-05-wavelet-diagrama) ilustra a análise multirresolução realizada pela DWT, na qual a subbanda de aproximação (LL) é sucessivamente decomposta, formando uma representação hierárquica com dois níveis.

In [ ]:
from IPython.display import HTML
HTML("""
<div style="font-family:sans-serif;max-width:680px;margin:0 auto;padding:10px;">
<div style="text-align:center;font-size:12px;font-weight:bold;color:#374151;margin-bottom:8px;">
  Decomposição Wavelet 2D — Estrutura Multirresolução (2 níveis)
</div>
<svg viewBox="0 0 640 260" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#f9fafb;">
  <!-- Imagem original -->
  <rect x="20" y="80" width="100" height="100" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5" rx="3"/>
  <text x="70" y="126" font-size="10" fill="#1e40af" text-anchor="middle" font-weight="bold">f(x,y)</text>
  <text x="70" y="140" font-size="9" fill="#1e40af" text-anchor="middle">M × N</text>
  <!-- Seta 1 -->
  <line x1="120" y1="130" x2="165" y2="130" stroke="#6b7280" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="142" y="124" font-size="8" fill="#6b7280" text-anchor="middle">DWT</text>
  <!-- Bloco Nível 1: 4 subbandas -->
  <rect x="165" y="55" width="80" height="75" fill="#fef3c7" stroke="#f59e0b" stroke-width="1.2" rx="2"/>
  <text x="205" y="87" font-size="8" fill="#92400e" text-anchor="middle" font-weight="bold">LL₁</text>
  <text x="205" y="98" font-size="7" fill="#92400e" text-anchor="middle">aprox.</text>
  <text x="205" y="109" font-size="7" fill="#92400e" text-anchor="middle">M/2 × N/2</text>
  <rect x="245" y="55" width="80" height="75" fill="#dcfce7" stroke="#22c55e" stroke-width="1.2" rx="2"/>
  <text x="285" y="87" font-size="8" fill="#166534" text-anchor="middle" font-weight="bold">LH₁</text>
  <text x="285" y="98" font-size="7" fill="#166534" text-anchor="middle">horiz.</text>
  <rect x="165" y="130" width="80" height="75" fill="#fce7f3" stroke="#ec4899" stroke-width="1.2" rx="2"/>
  <text x="205" y="162" font-size="8" fill="#9d174d" text-anchor="middle" font-weight="bold">HL₁</text>
  <text x="205" y="173" font-size="7" fill="#9d174d" text-anchor="middle">vert.</text>
  <rect x="245" y="130" width="80" height="75" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.2" rx="2"/>
  <text x="285" y="162" font-size="8" fill="#5b21b6" text-anchor="middle" font-weight="bold">HH₁</text>
  <text x="285" y="173" font-size="7" fill="#5b21b6" text-anchor="middle">diag.</text>
  <!-- Rótulo nível 1 -->
  <text x="245" y="248" font-size="9" fill="#6b7280" text-anchor="middle">Nível 1 — M/2 × N/2 cada</text>
  <!-- Seta LL₁ → Nível 2 -->
  <line x1="205" y1="55" x2="205" y2="45" stroke="#6b7280" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="205" y1="45" x2="400" y2="45" stroke="#6b7280" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="400" y1="45" x2="400" y2="55" stroke="#6b7280" stroke-width="1" marker-end="url(#arr)" stroke-dasharray="3,2"/>
  <text x="302" y="40" font-size="8" fill="#6b7280" text-anchor="middle">DWT sobre LL₁</text>
  <!-- Bloco Nível 2: subbandas de LL₁ -->
  <rect x="360" y="55" width="50" height="45" fill="#fef3c7" stroke="#f59e0b" stroke-width="1.2" rx="2"/>
  <text x="385" y="75" font-size="8" fill="#92400e" text-anchor="middle" font-weight="bold">LL₂</text>
  <text x="385" y="88" font-size="7" fill="#92400e" text-anchor="middle">M/4×N/4</text>
  <rect x="410" y="55" width="50" height="45" fill="#dcfce7" stroke="#22c55e" stroke-width="1.2" rx="2"/>
  <text x="435" y="80" font-size="8" fill="#166534" text-anchor="middle" font-weight="bold">LH₂</text>
  <rect x="360" y="100" width="50" height="45" fill="#fce7f3" stroke="#ec4899" stroke-width="1.2" rx="2"/>
  <text x="385" y="125" font-size="8" fill="#9d174d" text-anchor="middle" font-weight="bold">HL₂</text>
  <rect x="410" y="100" width="50" height="45" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.2" rx="2"/>
  <text x="435" y="125" font-size="8" fill="#5b21b6" text-anchor="middle" font-weight="bold">HH₂</text>
  <text x="435" y="168" font-size="8" fill="#6b7280" text-anchor="middle">Nível 2</text>
  <!-- Legenda direita -->
  <rect x="490" y="55" width="140" height="130" fill="#fff" stroke="#e5e7eb" rx="4"/>
  <text x="560" y="73" font-size="9" fill="#374151" text-anchor="middle" font-weight="bold">Legenda</text>
  <rect x="500" y="82" width="12" height="12" fill="#fef3c7" stroke="#f59e0b"/>
  <text x="518" y="92" font-size="8" fill="#374151">LL — Aproximação</text>
  <rect x="500" y="100" width="12" height="12" fill="#dcfce7" stroke="#22c55e"/>
  <text x="518" y="110" font-size="8" fill="#374151">LH — Bordas horiz.</text>
  <rect x="500" y="118" width="12" height="12" fill="#fce7f3" stroke="#ec4899"/>
  <text x="518" y="128" font-size="8" fill="#374151">HL — Bordas vert.</text>
  <rect x="500" y="136" width="12" height="12" fill="#ede9fe" stroke="#8b5cf6"/>
  <text x="518" y="146" font-size="8" fill="#374151">HH — Detalhes diag.</text>
  <text x="560" y="168" font-size="8" fill="#6b7280" text-anchor="middle">Cada nível: ½ da</text>
  <text x="560" y="178" font-size="8" fill="#6b7280" text-anchor="middle">resolução anterior</text>
  <defs>
    <marker id="arr" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L6,3 L0,6 Z" fill="#6b7280"/>
    </marker>
  </defs>
</svg>
</div>
""")

**Figura 5.13:** Diagrama da decomposição *wavelet* 2D em dois níveis.


O simulador da [Figura 5.14](#fig-05-sim-05-wavelet) permite explorar interativamente a Transformada *Wavelet* Discreta 2D (DWT) utilizando a *wavelet* de Haar. A decomposição em subbandas evidencia a separação entre a componente de aproximação e as componentes de detalhe da imagem.

Os diferentes padrões de entrada permitem observar o comportamento direcional dos filtros. Em imagens com bordas horizontais e verticais, as subbandas LH e HL destacam, respectivamente, as variações verticais e horizontais da intensidade. Em regiões de variação suave, a maior parte da energia concentra-se na subbanda de aproximação LL, enquanto as subbandas de detalhe apresentam coeficientes próximos de zero.

A análise multirresolução também pode ser observada ao aumentar o número de níveis de decomposição. Nesse caso, apenas a subbanda $\text{LL}_1$ é novamente decomposta, originando as subbandas $\text{LL}_2$, $\text{LH}_2$, $\text{HL}_2$ e $\text{HH}_2$, que formam o segundo nível da representação hierárquica.

Em padrões formados por regiões homogêneas de grande extensão, como um degradê suave ou um tabuleiro composto por blocos grandes, a energia permanece predominantemente concentrada na subbanda LL. No degradê, isso ocorre porque as diferenças entre pixels vizinhos são pequenas. No tabuleiro, por sua vez, os pixels possuem praticamente a mesma intensidade no interior de cada bloco, de modo que apenas as fronteiras entre blocos produzem coeficientes não nulos nas subbandas de detalhe. Como essas fronteiras ocupam apenas uma pequena fração da imagem, sua contribuição para a energia total permanece reduzida.

Para viabilizar a análise visual dessas variações sutis, o simulador incorpora um controle de ganho de contraste dos detalhes (variando de 1 a 8). Esse parâmetro funciona como um fator de amplificação linear aplicado exclusivamente aos coeficientes das subbandas de detalhe (LH, HL e HH) antes de sua renderização em tela. Em cenários de transição suave (como o gradiente) ou de uniformidade local (como o interior dos blocos do tabuleiro), as diferenças numéricas calculadas pelo filtro passa-altas de Haar resultam em coeficientes muito próximos de zero, o que tornaria os quadrantes correspondentes escuros e imperceptíveis a olho nu. Ao multiplicar esses valores pelo ganho, o simulador resgata visualmente as estruturas de alta frequência ocultas e realça a orientação das bordas remanescentes.

O gráfico de energia por subbanda quantifica essa distribuição entre a componente de aproximação e as componentes de detalhe, demonstrando que o ganho visual não altera a métrica original da energia. Em imagens naturais, a maior parte da energia concentra-se na subbanda LL, enquanto as subbandas LH, HL e HH representam principalmente bordas, texturas e outras variações locais da intensidade.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-05-wavelet" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-05-wavelet * { box-sizing: border-box; }
  #sim-05-wavelet canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; image-rendering: pixelated; }
  #sim-05-wavelet select { font-size: 11px; padding: 6px 10px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; font-weight: 600; cursor: pointer; outline: none; }
  #sim-05-wavelet input[type=range] { cursor: pointer; accent-color: #2980b9; }
  #sim-05-wavelet .sim04_w_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_w_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🌊 Simulador: Decomposição Wavelet 2D</span>
  <span class="sim04_w_pill">Transformada Haar</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles Superiores -->
  <div style="display:flex; flex-wrap:wrap; gap:14px; align-items:center; margin-bottom:14px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    <div style="display:flex; flex-direction:column; gap:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Padrão Sintético</label>
      <select id="wsim_pattern">
        <option value="combined" selected>Combinado (formas + textura)</option>
        <option value="shapes">Formas (bordas h/v)</option>
        <option value="texture">Textura (alta frequência)</option>
        <option value="gradient">Gradiente suave</option>
      </select>
    </div>
    
    <div style="display:flex; flex-direction:column; gap:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Níveis de Decomposição</label>
      <div style="display:flex; gap:12px; height:32px; align-items:center;">
        <label style="display:flex; align-items:center; gap:5px; font-size:11.5px; color:#5e5a4a; font-weight:600; cursor:pointer;"><input type="radio" name="wsim_lv" value="1" style="cursor:pointer;"> 1 nível</label>
        <label style="display:flex; align-items:center; gap:5px; font-size:11.5px; color:#5e5a4a; font-weight:600; cursor:pointer;"><input type="radio" name="wsim_lv" value="2" checked style="cursor:pointer;"> 2 níveis</label>
      </div>
    </div>
    
    <div style="display:flex; flex-direction:column; gap:4px; flex:1; min-width:160px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Ganho de Contraste: <span id="wsim_gainv" style="font-weight:700; color:#2980b9;">3.0</span></label>
      <input type="range" id="wsim_gain" min="1" max="8" step="0.5" value="3" style="width:100%; height:4px;">
    </div>
  </div>

  <!-- Imagem Original vs Mosaico Wavelet -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(240px, 1fr)); gap:14px; margin-bottom:14px;">
    <div class="sim04_w_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700;">Imagem Original</div>
      <canvas id="wsim_orig" style="width:100%; display:block; margin:0 auto;"></canvas>
    </div>
    <div class="sim04_w_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700;">Decomposição Wavelet (Mosaico)</div>
      <canvas id="wsim_mosaic" style="width:100%; display:block; margin:0 auto; background:#ffffff;"></canvas>
    </div>
  </div>

  <!-- Energia por Subbanda -->
  <div class="sim04_w_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700; text-align:left;">Energia por Subbanda (%) — Soma Preservada (Parseval)</div>
    <canvas id="wsim_energy" style="width:100%; display:block; margin:0 auto;"></canvas>
  </div>

  <!-- Legendas Explicativas -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(220px, 1fr)); gap:8px;">
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#26241d,#fafaf7);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">LL — Aproximação</div><div style="font-size:10.5px; color:#8a8371;">Versão suavizada e reduzida da imagem</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">LH — Detalhe Horizontal</div><div style="font-size:10.5px; color:#8a8371;">Realça bordas horizontais (variação vertical)</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">HL — Detalhe Vertical</div><div style="font-size:10.5px; color:#8a8371;">Realça bordas verticais (variação horizontal)</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">HH — Detalhe Diagonal</div><div style="font-size:10.5px; color:#8a8371;">Texturas e cantos (variação em ambas direções)</div></div>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04Wavelet(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    var wsim_N = 128;

    function wsim_genImage(type){
      var img = [];
      for(var y=0; y<wsim_N; y++){
        var row = [];
        for(var x=0; x<wsim_N; x++){
          var v = 0;
          if(type==='gradient'){
            v = 255*(0.5*x/wsim_N + 0.5*y/wsim_N);
          } else if(type==='shapes'){
            v = 40 + 30*Math.sin(x/30);
            if(x>18&&x<58&&y>18&&y<58) v = 220;
            var cx=95, cy=95, r=24;
            if((x-cx)*(x-cx)+(y-cy)*(y-cy) < r*r) v = 195;
            if(x>70&&x<74) v = 235;
          } else if(type==='texture'){
            var period=8;
            v = ((Math.floor(x/period)+Math.floor(y/period))%2===0) ? 200 : 55;
          } else {
            v = 55 + 35*(x/wsim_N) + 15*Math.sin(y/12);
            if(x>12&&x<50&&y>12&&y<50) v = 225;
            var cx2=95, cy2=38, r2=17;
            if((x-cx2)*(x-cx2)+(y-cy2)*(y-cy2) < r2*r2) v = 205;
            if(y>82 && y<122){
              var p=6;
              v = ((Math.floor(x/p)+Math.floor(y/p))%2===0) ? 185 : 65;
            }
            if(Math.abs(x-y) < 2) v = 240;
          }
          row.push(Math.max(0,Math.min(255,v)));
        }
        img.push(row);
      }
      return img;
    }

    function wsim_dwt2(m){
      var h = m.length, w = m[0].length;
      var halfH = h / 2, halfW = w / 2;
      
      var LL = [], LH = [], HL = [], HH = [];
      for (var r = 0; r < halfH; r++) {
        LL.push(new Array(halfW));
        LH.push(new Array(halfW));
        HL.push(new Array(halfW));
        HH.push(new Array(halfW));
      }

      for(var r=0; r<halfH; r++){
        for(var c=0; c<halfW; c++){
          var a = m[2*r][2*c];
          var b = m[2*r][2*c+1];
          var g = m[2*r+1][2*c];
          var d = m[2*r+1][2*c+1];
          
          LL[r][c] = (a + b + g + d) / 2.0;
          LH[r][c] = (a - b + g - d) / 2.0;
          HL[r][c] = (a + b - g - d) / 2.0;
          HH[r][c] = (a - b - g + d) / 2.0;
        }
      }
      return {LL:LL, LH:LH, HL:HL, HH:HH};
    }

    function wsim_divCol(t){
      t = Math.max(-1, Math.min(1, t));
      if(t>=0) {
        // Interpola de branco (255,255,255) até azul forte (41,128,185)
        return [
          Math.round(255 + t*(41 - 255)),
          Math.round(255 + t*(128 - 255)),
          Math.round(255 + t*(185 - 255))
        ];
      }
      var s = -t;
      // Interpola de branco (255,255,255) até vermelho forte (192,57,43)
      return [
        Math.round(255 + s*(192 - 255)),
        Math.round(255 + s*(57 - 255)),
        Math.round(255 + s*(43 - 255))
      ];
    }

    function wsim_tileCanvas(mat, mode, gain){
      var d = mat.length;
      var cnv = document.createElement('canvas');
      cnv.width = d; cnv.height = d;
      var cctx = cnv.getContext('2d');
      var idata = cctx.createImageData(d,d);
      if(mode==='gray'){
        var mn=Infinity, mx=-Infinity;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){ var v=mat[r][c]; if(v<mn)mn=v; if(v>mx)mx=v; }
        var range=(mx-mn)||1;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){
          var v=(mat[r][c]-mn)/range*255;
          var idx=(r*d+c)*4;
          idata.data[idx]=v; idata.data[idx+1]=v; idata.data[idx+2]=v; idata.data[idx+3]=255;
        }
      } else {
        var maxAbs=0;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){ var av=Math.abs(mat[r][c]); if(av>maxAbs) maxAbs=av; }
        maxAbs = (maxAbs/gain) || 1;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){
          var t = mat[r][c]/maxAbs;
          var rgb = wsim_divCol(t);
          var idx=(r*d+c)*4;
          idata.data[idx]=rgb[0]; idata.data[idx+1]=rgb[1]; idata.data[idx+2]=rgb[2]; idata.data[idx+3]=255;
        }
      }
      cctx.putImageData(idata,0,0);
      return cnv;
    }

    function wsim_energySum(mat){
      var s=0;
      for(var r=0; r<mat.length; r++) for(var c=0; c<mat[0].length; c++) s += mat[r][c]*mat[r][c];
      return s;
    }

    var wsim_pattern='combined', wsim_level=2, wsim_gain=3;
    var wsim_currentImg = wsim_genImage(wsim_pattern);

    function wsim_setupSquare(id, cap){
      var c = root.querySelector('#' + id);
      var parentW = c.parentElement.clientWidth - 24;
      var w = Math.min(parentW, cap || 360);
      if(w<80) w = 240;
      c.width = w; c.height = w;
      return {c:c, ctx:c.getContext('2d'), size:w};
    }

    function wsim_drawOriginal(){
      var s = wsim_setupSquare('wsim_orig', 360);
      s.ctx.imageSmoothingEnabled = false;
      var tile = wsim_tileCanvas(wsim_currentImg, 'gray', wsim_gain);
      s.ctx.drawImage(tile, 0, 0, s.size, s.size);
    }

    function wsim_drawMosaic(){
      var s = wsim_setupSquare('wsim_mosaic', 360);
      var ctx = s.ctx, full = s.size, half = full/2;
      ctx.imageSmoothingEnabled = false;
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, full, full);

      var c1 = wsim_dwt2(wsim_currentImg);

      function place(mat, mode, x, y, w, h){
        var tile = wsim_tileCanvas(mat, mode, wsim_gain);
        ctx.drawImage(tile, x, y, w, h);
      }

      if(wsim_level===1){
        place(c1.LL, 'gray', 0, 0, half, half);
        place(c1.LH, 'div', half, 0, half, half);
        place(c1.HL, 'div', 0, half, half, half);
        place(c1.HH, 'div', half, half, half, half);
      } else {
        var c2 = wsim_dwt2(c1.LL);
        var q = half/2;
        place(c2.LL, 'gray', 0, 0, q, q);
        place(c2.LH, 'div', q, 0, q, q);
        place(c2.HL, 'div', 0, q, q, q);
        place(c2.HH, 'div', q, q, q, q);
        
        place(c1.LH, 'div', half, 0, half, half);
        place(c1.HL, 'div', 0, half, half, half);
        place(c1.HH, 'div', half, half, half, half);
      }

      ctx.strokeStyle = '#e4dcc8';
      ctx.lineWidth = 1.5;
      ctx.beginPath();
      ctx.moveTo(half,0); ctx.lineTo(half,full);
      ctx.moveTo(0,half); ctx.lineTo(full,half);
      ctx.stroke();
      if(wsim_level===2){
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.moveTo(half/2,0); ctx.lineTo(half/2,half);
        ctx.moveTo(0,half/2); ctx.lineTo(half,half/2);
        ctx.stroke();
      }

      ctx.font = '700 10.5px monospace';
      function lbl(t,x,y){
        ctx.fillStyle = '#26241d';
        ctx.fillText(t, x+5, y+14);
      }
      if(wsim_level===1){
        lbl('LL', 0,0); lbl('LH', half,0); lbl('HL',0,half); lbl('HH', half,half);
      } else {
        lbl('LL₂', 0,0); lbl('LH₂', half/2,0); lbl('HL₂',0,half/2); lbl('HH₂', half/2, half/2);
        lbl('LH₁', half,0); lbl('HL₁',0,half); lbl('HH₁', half,half);
      }
    }

    function wsim_drawEnergy(){
      var s = root.querySelector('#wsim_energy');
      var w = s.parentElement.clientWidth - 24;
      if(w<100) w = 280;
      s.width = w; s.height = 160;
      var ctx = s.getContext('2d');
      ctx.clearRect(0,0,w,s.height);

      var c1 = wsim_dwt2(wsim_currentImg);
      var total = wsim_energySum(wsim_currentImg);
      var bars, labels;
      if(wsim_level===1){
        bars = [wsim_energySum(c1.LL), wsim_energySum(c1.LH), wsim_energySum(c1.HL), wsim_energySum(c1.HH)];
        labels = ['LL','LH','HL','HH'];
      } else {
        var c2 = wsim_dwt2(c1.LL);
        bars = [wsim_energySum(c2.LL), wsim_energySum(c2.LH), wsim_energySum(c2.HL), wsim_energySum(c2.HH),
                wsim_energySum(c1.LH), wsim_energySum(c1.HL), wsim_energySum(c1.HH)];
        labels = ['LL₂','LH₂','HL₂','HH₂','LH₁','HL₁','HH₁'];
      }
      var pcts = bars.map(function(b){ return b/total*100; });

      var pad = {l:34, r:10, t:12, b:24};
      var plotH = s.height - pad.t - pad.b;
      var plotW = w - pad.l - pad.r;

      ctx.strokeStyle = 'rgba(0,0,0,.06)'; ctx.lineWidth=0.5;
      for(var j=0;j<=4;j++){
        var y = pad.t + plotH*j/4;
        ctx.beginPath(); ctx.moveTo(pad.l,y); ctx.lineTo(w-pad.r,y); ctx.stroke();
      }
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth=1;
      ctx.beginPath(); ctx.moveTo(pad.l,pad.t); ctx.lineTo(pad.l,s.height-pad.b); ctx.lineTo(w-pad.r,s.height-pad.b); ctx.stroke();

      ctx.fillStyle = '#8a8371'; ctx.font='9.5px monospace'; ctx.textAlign='right';
      for(var j=0;j<=4;j++){
        var y = pad.t + plotH*j/4;
        ctx.fillText(Math.round(100 - j*25) + '%', pad.l-4, y+3);
      }

      var bw = plotW/pcts.length;
      var barW = bw*0.6;
      ctx.textAlign='center';
      pcts.forEach(function(p, i){
        var x = pad.l + i*bw + (bw-barW)/2;
        var barH = (p/100)*plotH;
        var y = (s.height-pad.b) - barH;
        var isLL = labels[i].indexOf('LL') === 0;
        ctx.fillStyle = isLL ? '#2980b9' : '#c0392b';
        ctx.fillRect(x,y,barW,barH);
        ctx.strokeStyle = isLL ? '#1b4f72' : '#78281f';
        ctx.lineWidth=0.5;
        ctx.strokeRect(x,y,barW,barH);
        if(barH>16){
          ctx.fillStyle='#ffffff'; ctx.font='bold 9.5px monospace';
          ctx.fillText(Math.round(p) + '%', x+barW/2, y+13);
        }
        ctx.fillStyle='#8a8371'; ctx.font='10px monospace';
        ctx.fillText(labels[i], x+barW/2, s.height-pad.b+14);
      });
    }

    function wsim_redraw(){
      wsim_currentImg = wsim_genImage(wsim_pattern);
      wsim_drawOriginal();
      wsim_drawMosaic();
      wsim_drawEnergy();
    }

    root.querySelector('#wsim_pattern').addEventListener('change', function(e){
      wsim_pattern = e.target.value; wsim_redraw();
    });
    root.querySelectorAll('input[name="wsim_lv"]').forEach(function(r){
      r.addEventListener('change', function(e){ wsim_level = +e.target.value; wsim_drawMosaic(); wsim_drawEnergy(); });
    });
    root.querySelector('#wsim_gain').addEventListener('input', function(e){
      wsim_gain = +e.target.value;
      root.querySelector('#wsim_gainv').textContent = wsim_gain.toFixed(1);
      wsim_drawMosaic();
    });

    wsim_redraw();
    window.addEventListener('resize', wsim_redraw);
  }

  function tryInitSim04Wavelet(){
    var root = document.getElementById('sim-05-wavelet');
    if (root) initSim04Wavelet(root); else setTimeout(tryInitSim04Wavelet, 200);
  }
  tryInitSim04Wavelet();
})();
</script>
""")

**Figura 5.14:** Simulação da decomposição *wavelet* 2D.


### 5.7.4 Análise Multirresolução com a DWT 2D

A Transformada *Wavelet* Discreta 2D (DWT) decompõe uma imagem em componentes de aproximação e detalhe, organizadas de forma hierárquica em diferentes escalas e orientações. Como as subbandas de detalhe em imagens naturais frequentemente apresentam coeficientes de baixo contraste, os exemplos práticos a seguir utilizam um padrão geométrico sintético gerado em Python. Essa abordagem replica o comportamento do simulador da [Figura 5.14](#fig-05-sim-05-wavelet), tornando visualmente explícitos os efeitos da filtragem espacial e da decomposição multirresolução.

#### 5.7.4.1 Decomposição em Mosaico de Múltiplos Níveis

A [Figura 5](#fig-05-dwt-subbandas) ilustra a estrutura hierárquica da DWT em dois níveis utilizando a *wavelet* de Haar. O processo baseia-se na aplicação combinada de filtros passa-baixa e passa-alta nas direções horizontal e vertical, seguidos por subamostragem por um fator de 2.

No primeiro nível, a imagem original origina a subbanda de aproximação ($LL_1$) e as componentes de detalhe horizontal ($LH_1$), vertical ($HL_1$) e diagonal ($HH_1$). Na análise multirresolução, a subbanda $LL_1$ é novamente filtrada e subamostrada, gerando o segundo nível de decomposição ($LL_2$, $LH_2$, $HL_2$ e $HH_2$). 

Para viabilizar a interpretação visual das componentes de detalhe, o código extrai o valor absoluto de seus coeficientes e aplica uma normalização linear (*min-max*) para ocupar toda a faixa dinâmica de tons de cinza [0, 255]. Essa operação transforma regiões homogêneas (coeficientes nulos) em preto e destaca em branco as bordas e texturas extraídas em cada escala e orientação.

#### 5.7.4.2 O Compromisso entre Localização e Suavidade

A escolha da função de base (*wavelet*) influencia diretamente a forma como as feições da imagem são distribuídas e codificadas pelos coeficientes da DWT. A [Figura 5](#fig-05-dwt-wavelets) compara os resultados práticos obtidos ao aplicar quatro famílias distintas sobre o padrão geométrico sintético: `haar`, `db4`, `sym4` e `bior2.2`.

Por possuir suporte curto e formato de função degrau, a *wavelet* de Haar produz coeficientes altamente localizados nas descontinuidades espaciais, gerando bordas finas e nítidas nas subbandas de detalhe. Em contrapartida, famílias como Daubechies (`db4`) e Symlets (`sym4`), que apresentam maior suporte (filtros mais longos) e maior número de momentos nulos, geram respostas mais suaves e distribuídas ao redor das transições, o que pode introduzir leves oscilações ou borramentos nas fronteiras abruptas.

Esse comportamento evidencia o clássico compromisso (*trade-off*) da análise de multirresolução: suportes menores favorecem a localização espacial exata das bordas, enquanto suportes maiores e maior número de momentos nulos tendem a produzir representações mais esparsas e suaves. Essa suavidade e capacidade de atenuação de altas frequências garantem maior eficiência na compactação da energia, características fundamentais para aplicações de compressão de dados e remoção de ruído (*denoising*).

#### 5.7.4.3 Limiarização de Coeficientes e Compressão

Uma das principais aplicações da Transformada *Wavelet* Discreta (DWT) é a compressão de dados, impulsionada pela capacidade de representação **esparsas** dos coeficientes. A [Figura 5](#fig-05-dwt-reconstrucao) ilustra o efeito da limiarização abrupta (*hard thresholding*), técnica na qual coeficientes de detalhe com magnitude inferior a um limiar $T$ são integralmente anulados antes do processo de síntese realizado pela Transformada *Wavelet* Discreta Inversa (IDWT).

À medida que o limiar $T$ é elevado, um volume crescente de coeficientes de alta frequência é zerado. Por concentrarem menor energia, a remoção dessas componentes reduz consideravelmente a quantidade de informação necessária para representar a imagem, mantendo a componente de aproximação global (a subbanda $LL$ mais profunda) intacta para preservar a estrutura macro. Visualmente, esse descarte de coeficientes manifesta-se através do desaparecimento progressivo de texturas finas e da suavização de transições abruptas de intensidade.

A fidelidade da imagem reconstruída frente à original é quantificada pela métrica de **Pico da Relação Sinal-Ruído (PSNR, *Peak Signal-to-Noise Ratio*)**, expressa em decibéis (dB). Valores mais altos de PSNR indicam menor distorção e maior proximidade matemática com o sinal original. O experimento prático evidencia o decaimento gradual do PSNR conforme a agressividade da limiarização aumenta, permitindo avaliar numericamente o limiar ótimo para o balanço entre compressão e degradação visual.

### Síntese — Fourier vs. *Wavelets*: quando utilizar cada abordagem?

A [Tabela 5.5](#tbl-05-fourier-wavelet) sintetiza as principais diferenças estruturais e operacionais entre a Transformada Discreta de Fourier (DFT) e a Transformada *Wavelet* Discreta (DWT).

<a id="tbl-05-fourier-wavelet"></a>

**Tabela 5.5:** Comparação entre a Transformada Discreta de Fourier (DFT) e a Transformada *Wavelet* Discreta (DWT), destacando suas principais características e aplicações.

| Critério | Fourier (DFT) | *Wavelet* (DWT) |
|:---|:---|:---|
| **Funções de base** | Senoides de suporte infinito | Funções de suporte compacto |
| **Localização espacial** | Não explícita (global) | Explícita (local) |
| **Filtragem espectral** | Excelente para controle fino de frequências | Baseada em subbandas (escalas) |
| **Compressão de imagens** | Base da DCT (JPEG tradicional) | Base da DWT (JPEG 2000) |
| **Análise multiescala** | Não | Sim |
| **Remoção de ruído periódico** | Altamente eficiente | Pouco indicada |
| **Sinais não estacionários** | Limitada | Altamente eficiente |


Em termos práticos, a DFT consolida-se como a ferramenta ideal para análise espectral pura, projeto de filtros seletivos no domínio da frequência e atenuação de ruídos periódicos e harmônicos. Por outro lado, a DWT sobressai-se em cenários que exigem a preservação rigorosa da localização espacial das feições associada ao seu conteúdo frequencial, destacando-se em compressão de dados, análise multirresolução e processamento de transições abruptas. Desse modo, ambas as transformadas devem ser compreendidas como técnicas perfeitamente complementares, mapeando caminhos distintos e específicos para a resolução de problemas em PDI-VC.

> ### 📝 Analogias com Áudio: Limitações e Cuidados
>
> Ao fazer analogias entre processamento de imagens e áudio, é importante considerar as diferenças fundamentais:
>
> * Em sistemas de áudio estéreo/multicanais, a fase entre canais é crucial para a percepção de localização espacial (diferenças interaurais de fase e tempo).
>
> * Em sistemas monaurais, a fase tem influência perceptual limitada — o ouvido humano é relativamente insensível à fase absoluta de componentes senoidais isolados.
>
> * Em imagens, a fase da DFT é sempre fundamental para a localização espacial de estruturas, independentemente de ser uma imagem monocromática ou colorida.
>
> A analogia entre fase em áudio e fase em imagens deve ser usada com cautela, destacando que, embora ambas carreguem informações sobre a organização espacial/temporal do sinal, os mecanismos perceptuais são fundamentalmente diferentes.

## 5.8 Compressão de Imagens

Enquanto as *wavelets* estabelecem a fundação teórica do padrão JPEG 2000, o padrão JPEG tradicional baseia-se na **Transformada Discreta de Cossenos (DCT, *Discrete Cosine Transform*)**. Apesar das diferenças estruturais, ambas as abordagens compartilham o mesmo princípio fundamental: compactar a energia da imagem em um número reduzido de coeficientes e descartar as componentes de menor relevância com impacto visual mínimo.

O objetivo central da compressão é reduzir o volume de dados necessário para o armazenamento ou transmissão de uma imagem. Esse processo é viabilizado pela identificação e eliminação de **redundâncias** estruturais e perceptuais.

### 5.8.1 Taxonomia das Redundâncias

O desenvolvimento de algoritmos de compressão fundamenta-se na identificação e eliminação de três categorias principais de redundância, sintetizadas na [Tabela 5.6](#tbl-05-redundancias).

<a id="tbl-05-redundancias"></a>

**Tabela 5.6:** Categorias de redundância em imagens digitais e seus respectivos mecanismos de exploração.

| Tipo | Definição | Abordagem de Exploração |
|:---|:---|:---|
| **Espacial (interpixel)** | Alta correlação e dependência estatística entre pixels vizinhos. | DCT, DWT e codificação preditiva. |
| **Espectral (intercanal)** | Correlação estatística entre os canais de cor de uma mesma imagem. | Transformações de espaço de cor (ex: RGB para $YC_bC_r$). |
| **Psicovisual** | Insensibilidade do sistema visual humano (SVH) a variações de alta frequência e baixo contraste. | Processos de quantização seletiva de coeficientes. |


A depender da preservação da informação original após o processo de decodificação, os métodos de compressão dividem-se em duas classes fundamentais:

* **Sem perda (*lossless*):** Garante uma reconstrução bit a bit idêntica à imagem original. É empregada em cenários onde a integridade dos dados é estritamente crítica, como em imagens médicas, diagnósticos por imagem e armazenamento de documentos textuais.
* **Com perda (*lossy*):** Admite a introdução de uma distorção controlada no sinal em troca de taxas de compressão substancialmente mais elevadas. É a abordagem padrão para fotografias de consumo e *streaming* de vídeo, ecossistemas nos quais o SVH tolera pequenas atenuações de alta frequência sem percepção de degradação da qualidade visual.

### 5.8.2 Transformada de Cossenos Discreta (DCT-II 2D)

A **Transformada de Cossenos Discreta** (DCT) constitui a operação central do padrão JPEG. Diferentemente da DFT, que utiliza uma base complexa, a DCT baseia-se em funções trigonométricas puramente reais. Para um bloco de imagem $f(x,y)$ de dimensões $N \times N$, a **DCT-II 2D** mapeia o sinal espacial para o domínio das frequências espaciais, gerando a matriz de coeficientes $C(u,v)$ por meio de:

<a id="eq-05-dct"></a>
$$
C(u,v) = \alpha(u)\,\alpha(v) \sum_{x=0}^{N-1}\sum_{y=0}^{N-1} f(x,y)\,
\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]
\cos\!\left[\frac{\pi(2y+1)v}{2N}\right] \tag{5.9}
$$


onde os fatores de normalização ortogonal são dados por $\alpha(0) = \sqrt{1/N}$ e $\alpha(k) = \sqrt{2/N}$ para $k > 0$. 

Cada coeficiente $C(u,v)$ quantifica a contribuição — ou "peso" — de uma frequência espacial específica dentro daquele bloco. O termo $C(0,0)$ é denominado **componente DC** e representa a intensidade média do bloco (frequência nula). Os demais coeficientes, chamados de **componentes AC** (*Alternating Current*), correspondem às frequências espaciais progressivamente maiores.

### 5.8.3 As Funções de Base da DCT

Sob uma perspectiva geométrica, a [Equação 5.9](#eq-05-dct) realiza a projeção do bloco de pixels sobre um conjunto de funções ortogonais. Para o caso padrão do JPEG ($N=8$), o bloco espacial é decomposto em uma combinação linear de **64 funções de base** bidimensionais, denotadas por $B_{u,v}(x,y)$ e geradas pelo produto de funções cossenoidais:

$$B_{u,v}(x,y) = \cos\left[ \frac{\pi (2x+1)u}{16} \right] \cos\left[ \frac{\pi (2y+1)v}{16} \right]$$

Dessa forma, a operação inversa pode ser interpretada como a reconstrução exata do bloco original por meio da soma ponderada dessas 64 matrizes de base, onde cada coeficiente $C(u,v)$ atua como o peso analítico de sua respectiva componente harmônica. 

A **frequência espacial** indicada pelos índices $(u,v)$ determina o número de ciclos de oscilação ao longo das dimensões horizontais e verticais do bloco. Como ilustrado na [Figura 5](#fig-05-dct-basis) — cujo código isola cada base aplicando a transformação inversa sobre impulsos unitários —, essas 64 funções são organizadas em uma matriz $8 \times 8$. O canto superior esquerdo ($u=0, v=0$) exibe o padrão uniforme de frequência nula (DC), enquanto o avanço para a direita (eixo $u$) ou para baixo (eixo $v$) mapeia variações harmônicas progressivamente maiores, representando transições rápidas, bordas e texturas nas orientações horizontais, verticais e diagonais.

> ### 📝 DCT vs DFT: Vantagem da Compactação de Energia
>
> Tanto a DCT quanto a DFT mapeiam um bloco $N \times N$ espacial em uma matriz de coeficientes de mesma dimensão. Contudo, para imagens naturais, a DCT apresenta maior eficiência na **compactação de energia** nas baixas frequências. Isso ocorre porque a DCT assume implicitamente uma simetria par do sinal nas fronteiras do bloco, o que equivale a uma extensão periódica contínua, minimizando o efeito de espalhamento espectral (*ringing*). Como consequência, a maioria dos coeficientes AC decai rapidamente para valores próximos de zero, otimizando o *pipeline* de compressão sem introduzir degradação visual perceptível.

### 5.8.4 Concentração de Energia e Reconstrução Progressiva

Antes da aplicação da DCT, os pixels do bloco de intensidade são rotineiramente transladados (subtraindo-se $128$ para imagens de 8 bits) a fim de centralizar o sinal em torno de zero, eliminando componentes contínuas desnecessárias. Ao computar a DCT sobre o bloco resultante, a propriedade de **compactação de energia** torna-se evidente: a quase totalidade da variância e da informação da imagem original concentra-se no coeficiente DC ($C(0,0)$) e nos primeiros harmônicos AC de baixa frequência.

A [Figura 5.15](#fig-05-dct-bloco) demonstra esse fenômeno por meio de uma reconstrução progressiva por truncamento abrupto. Em vez de utilizar todos os 64 coeficientes, o algoritmo preserva apenas os $k$ primeiros componentes — selecionados com base em uma varredura que prioriza as baixas frequências espaciais — e anula os demais. 

A síntese inversa (**IDCT**) realizada com apenas uma fração dos coeficientes (como 15% ou 30%) já é capaz de recuperar as estruturas e a iluminação macro do bloco original de pixels. À medida que harmônicos de frequências mais altas são progressivamente reincorporados, os detalhes finos e as transições rápidas são restaurados. Esse comportamento valida o princípio da compressão perceptual: as altas frequências descartadas possuem pouca energia e sua ausência, em condições normais, gera um impacto visual secundário na percepção do observador.

In [ ]:
%%writefile tmp/fig_05_dct_bloco.cpp
#define MM_OUT "tmp/fig_05_dct_bloco.png"
#include <opencv2/opencv.hpp>
#include <cmath>
#include <vector>
#include <string>
#include <algorithm>
#include <iostream>
#include <iomanip>
#include "morph.hpp"

//| label: fig-05-dct-bloco
//| fig-cap: "DCT 2D em bloco 8×8: coeficientes e reconstrução progressiva."
//| echo: true
//| output: false

// DCT-II 2D ortogonal (separável).
cv::Mat dct2(const cv::Mat& bloco) {
    cv::Mat tmp, result;
    cv::dct(bloco, tmp);  // DCT nas linhas
    cv::dct(tmp.t(), result);  // DCT nas colunas
    return result.t();
}

// IDCT-II 2D ortogonal.
cv::Mat idct2(const cv::Mat& coefs) {
    cv::Mat tmp, result;
    cv::idct(coefs, tmp);  // IDCT nas linhas
    cv::idct(tmp.t(), result);  // IDCT nas colunas
    return result.t();
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_19.png");
// [pdi:state-io:end]

    // Bloco 8×8 centralizado da imagem
    int cy = img_gray.h / 2;
    int cx = img_gray.w / 2;

    // Extrai o bloco 8x8 e converte para float
    cv::Mat img_cv(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    cv::Mat bloco_mat = img_cv(cv::Rect(cx, cy, 8, 8)).clone();
    cv::Mat bloco_float;
    bloco_mat.convertTo(bloco_float, CV_32F);
    bloco_float -= 128.0f;

    cv::Mat C = dct2(bloco_float);

    std::cout << "Coeficientes DCT do bloco 8x8:" << std::endl;
    for (int i = 0; i < 8; i++) {
        for (int j = 0; j < 8; j++) {
            std::cout << std::setw(4) << static_cast<int>(std::round(C.at<float>(i, j))) << " ";
        }
        std::cout << std::endl;
    }

    float dc_energy = C.at<float>(0,0) * C.at<float>(0,0);
    float total_energy = 0.0f;
    for (int i = 0; i < 8; i++)
        for (int j = 0; j < 8; j++)
            total_energy += C.at<float>(i, j) * C.at<float>(i, j);

    std::cout << "\nEnergia DC     : " << std::fixed << std::setprecision(1) << dc_energy << std::endl;
    std::cout << "Energia total  : " << std::fixed << std::setprecision(1) << total_energy << std::endl;
    std::cout << "Fracao no DC   : " << std::fixed << std::setprecision(0) 
              << (dc_energy / total_energy * 100.0) << "% "
              << "<- concentracao de energia" << std::endl;

    // Reconstrução progressiva
    std::vector<mm::Image> imgs_rec;
    std::vector<std::string> titles_rec;

    // Bloco original reconstruído
    cv::Mat bloco_orig;
    bloco_float.convertTo(bloco_orig, CV_8U);
    bloco_orig += 128;

    mm::Image img_orig(8, 8, 1);
    std::memcpy(img_orig.data.data(), bloco_orig.data, img_orig.data.size());
    imgs_rec.push_back(img_orig);
    titles_rec.push_back("Bloco original\n(8x8 pixels)");

    // Coeficientes a manter
    std::vector<int> keeps = {1, 4, 10, 20, 40, 64};

    for (int keep : keeps) {
        cv::Mat C_trunc = cv::Mat::zeros(8, 8, CV_32F);

        // Ordena índices por soma das coordenadas (zigzag aproximado)
        std::vector<std::pair<int,int>> indices;
        for (int u = 0; u < 8; u++)
            for (int v = 0; v < 8; v++)
                indices.push_back({u, v});

        std::sort(indices.begin(), indices.end(), 
                  [](const std::pair<int,int>& a, const std::pair<int,int>& b) {
                      return (a.first + a.second) < (b.first + b.second);
                  });

        for (int k = 0; k < keep && k < 64; k++) {
            int u = indices[k].first;
            int v = indices[k].second;
            C_trunc.at<float>(u, v) = C.at<float>(u, v);
        }

        cv::Mat rec_float = idct2(C_trunc) + 128.0f;
        cv::Mat rec;
        rec_float.convertTo(rec, CV_8U);

        mm::Image img_rec(8, 8, 1);
        std::memcpy(img_rec.data.data(), rec.data, img_rec.data.size());
        imgs_rec.push_back(img_rec);

        char buf[64];
        snprintf(buf, sizeof(buf), "%d coef.\n(%.0f%% do total)", keep, (keep / 64.0 * 100.0));
        titles_rec.push_back(buf);
    }

    // Exibe os resultados
    mm::show(imgs_rec, MM_OUT, titles_rec, 4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_dct_bloco.cpp -o tmp/fig_05_dct_bloco \
  && ./tmp/fig_05_dct_bloco \
  && test -f "tmp/fig_05_dct_bloco.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dct_bloco.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_dct_bloco.png"), figsize=(12, 7))

**Figura 5.15:** DCT 2D em bloco 8×8: coeficientes e reconstrução progressiva.


### 5.8.5 O *Pipeline* de Compressão JPEG

O padrão JPEG opera dividindo a imagem em blocos disjuntos de $8 \times 8$ pixels, processados por meio de uma sequência de transformações espaciais, perceptuais e estatísticas. O *pipeline* completo de codificação é estruturado em seis etapas principais:

$$
\text{RGB} \xrightarrow{\text{(1) } YC_bC_r} \xrightarrow{\text{(2) Subamostragem}} \xrightarrow{\text{(3) Blocos } 8 \times 8} \xrightarrow{\text{(4) DCT}} \xrightarrow{\text{(5) Quantização}} \xrightarrow{\text{(6) Codificação Entrópica}}
$$

A [Tabela 5.7](#tbl-05-pipeline-jpeg) detalha a função analítica e o fundamento perceptual que justifica cada uma dessas etapas.

<a id="tbl-05-pipeline-jpeg"></a>

**Tabela 5.7:** Etapas do *pipeline* de compressão JPEG e seus respectivos fundamentos de projeto.

| Etapa | Operação | Fundamento Perceptual e Estatístico |
|:---:|:---|:---|
| **1** | Conversão $RGB \rightarrow YC_bC_r$ | Separa a luminância ($Y$) da crominância ($C_b, C_r$). O sistema visual humano (SVH) apresenta maior sensibilidade a variações de brilho do que de cor. |
| **2** | Subamostragem de crominância (ex: 4:2:0) | Reduz a resolução espacial dos canais de cor pela metade, descartando dados redundantes com impacto visual desprezível. |
| **3–4** | Centralização e aplicação da DCT $8 \times 8$ | Translada os pixels para o intervalo $[-128, 127]$ e compacta a energia espectral do bloco nos coeficientes de baixa frequência. |
| **5** | Quantização linear seletiva | Divide cada coeficiente $C(u,v)$ pelo elemento correspondente da matriz $Q(u,v)$, aplicando arredondamento inteiro. Constitui a principal fonte de compressão com perda. |
| **6** | Varredura em ziguezague e codificação | Ordena os coeficientes quantizados para maximizar sequências nulas consecutivas, otimizando a codificação por comprimento de corrida (RLE) e a codificação de Huffman. |


A **matriz de quantização** $Q(u,v)$ é o mecanismo central de controle do compromisso entre taxa de compressão e qualidade visual. No algoritmo prático da [Figura 5.16](#fig-05-jpeg-pipeline), o fator de qualidade estipulado pelo usuário (escala de 1 a 100) é convertido em um escalar que parametriza a severidade da matriz $Q$. Valores reduzidos de qualidade expandem os divisores de $Q(u,v)$, forçando o truncamento em massa dos coeficientes AC para zero. Quando essa eliminação é excessiva, a descontinuidade nas fronteiras dos blocos adjacentes não é atenuada na reconstrução, gerando os denominados **artefatos de bloco** (*blocking artifacts*).

#### A Lógica da Varredura em Ziguezague

A eficiência do codificador entrópico subsequente à quantização depende diretamente da ordenação dos dados. Como a DCT concentra a energia vital no vértice superior esquerdo da matriz (baixas frequências) e empurra os coeficientes nulos para as extremidades opostas, a leitura linear por linhas ou colunas fragmentaria as sequências de zeros. 

A ordenação em ziguezague soluciona essa limitação ao percorrer a matriz diagonalmente em ordem crescente de frequência espacial. Esse mapeamento agrupa os coeficientes significativos no início do vetor e concentra os coeficientes nulos em uma única sequência contínua ao final do arranjo, permitindo que o algoritmo RLE codifique grandes blocos de dados de forma compacta e eficiente.

> ### 📝 5.9 O que é RLE?
>
> **RLE** (*Run-Length Encoding*) é uma técnica de compressão sem perdas que codifica sequências consecutivas de valores idênticos — especialmente **zeros** — como um par (contagem, valor). No JPEG, após a varredura em ziguezague, os coeficientes quantizados são organizados de modo que os zeros se concentrem ao final do vetor. O RLE então comprime essa longa corrida de zeros com extrema eficiência, otimizando o armazenamento e a transmissão da imagem comprimida.

In [ ]:
# Ainda não portado para esta linguagem nesta versão — referência conceitual em Python.

import numpy as np
import cv2
from scipy.fft import dct, idct

# ── Carregamento Seguro da Imagem da Câmera (skimage) ─────────────────────────
try:
    from skimage import data
    img_gray = data.camera()
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "scikit-image", "-q"])
    from skimage import data
    img_gray = data.camera()

# Redimensiona levemente para 256x256 para manter o padrão e velocidade dos testes anteriores
img_gray = cv2.resize(img_gray, (256, 256))

# ── Tabela de quantização luminância (padrão JPEG) ────────────────────────────
Q_luma = np.array([
    [16,11,10,16,24,40,51,61],
    [12,12,14,19,26,58,60,55],
    [14,13,16,24,40,57,69,56],
    [14,17,22,29,51,87,80,62],
    [18,22,37,56,68,109,103,77],
    [24,35,55,64,81,104,113,92],
    [49,64,78,87,103,121,120,101],
    [72,92,95,98,112,100,103,99]
], dtype=np.float64)

def dct2(bloco):
    """DCT-II 2D ortogonal (separável)."""    
    return dct(dct(bloco.T, norm='ortho').T, norm='ortho')

def idct2(coefs):
    """IDCT-II 2D ortogonal."""    
    return idct(idct(coefs.T, norm='ortho').T, norm='ortho')

def jpeg_compress_block(bloco, Q_table):
    """DCT → quantização → dequantização → IDCT em bloco 8×8."""
    C  = dct2(bloco.astype(np.float64) - 128)
    Cq = np.round(C / Q_table) * Q_table    # quantiza e dequantiza
    return np.clip(idct2(Cq) + 128, 0, 255)

def jpeg_quality_compress(img, qualidade=50):
    """JPEG simplificado: comprime imagem inteira por blocos 8×8."""
    if qualidade < 50:
        escala = 5000 / qualidade
    else:
        escala = 200 - 2 * qualidade
    # Corrigido de 'scala' para 'escala'
    Q = np.clip(np.round(Q_luma * escala / 100), 1, 255)
    
    h, w   = img.shape
    result = np.zeros_like(img, dtype=np.float64)
    for r in range(0, h-7, 8):
        for c in range(0, w-7, 8):
            result[r:r+8, c:c+8] = jpeg_compress_block(img[r:r+8, c:c+8], Q)
    return result.astype(np.uint8)

# ── Comparação de fatores de qualidade ───────────────────────────────────────
qualidades = [10, 25, 50, 75, 90]
imgs_jpeg  = [img_gray]
titles_jpeg = ["Original\n(Cameraman)"]

for q in qualidades:
    rec  = jpeg_quality_compress(img_gray, qualidade=q)
    psnr = cv2.PSNR(img_gray, rec)
    imgs_jpeg.append(rec)
    titles_jpeg.append(f"Q={q}\nPSNR={psnr:.1f}dB")

mm.show(imgs_jpeg, titles=titles_jpeg, cols=3, figsize=(14, 10))

**Figura 5.16:** *Pipeline* JPEG simplificado aplicado à imagem clássica do *Cameraman*: DCT em blocos 8×8, quantização com diferentes fatores de qualidade e reconstrução via IDCT. Os artefatos de bloco (*blocking artifacts*) tornam-se visualmente evidentes em fatores de qualidade reduzidos ($Q=10$ e $Q=25$).


### 5.9.1 Simulador Interativo: Quantização DCT

O simulador da [Figura 5.17](#fig-05-sim-05-dct) permite explorar o impacto do processo de quantização sobre um bloco $8 \times 8$ extraído de uma imagem real, sintetizando em tempo real as seguintes componentes:

* **Bloco original e reconstruído:** Representação direta dos pixels no domínio espacial em escala de cinza [0, 255].
* **Coeficientes DCT:** Distribuição da energia mapeada de forma logarítmica em um gradiente cromático, evidenciando a concentração de intensidade no vértice superior esquerdo (baixas frequências).
* **Coeficientes quantizados:** Exibição dos valores inteiros resultantes da divisão pela matriz $Q(u,v)$, tornando visualmente explícito o surgimento em massa de coeficientes nulos (em tons escuros) conforme o fator de qualidade é reduzido.
* **Métricas de compressão:** Painel de monitoramento que quantifica o Erro Quadrático Médio (MSE), o número de coeficientes preservados e o volume de zeros gerados para a codificação entrópica.

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-05-dct" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-dct * { box-sizing: border-box; }
  #sim-05-dct canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-dct button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-dct button:hover { background: #e8dfcf; }
  #sim-05-dct .sim04_dct_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_dct_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim04_dct_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 90px; }
  .sim04_dct_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim04_dct_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim04_dct_slider_container { display: flex; align-items: center; gap: 8px; margin-bottom: 10px; }
  .sim04_dct_slider_container label { font-size: 11px; font-weight: 700; color: #5e5a4a; min-width: 110px; }
  .sim04_dct_slider_container input[type=range] { flex: 1; cursor: pointer; height: 4px; accent-color: #2980b9; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⊞ Simulador: Quantização DCT-JPEG (bloco 8×8)</span>
  <span class="sim04_dct_pill">blocos 8×8</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Qualidade</div><div id="sim04_dct_qual" class="sim04_dct_stat_value" style="color:#2980b9;">50</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Coef. ≠ 0</div><div id="sim04_dct_nonzero" class="sim04_dct_stat_value" style="color:#27ae60;">–</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Zeros</div><div id="sim04_dct_zeros" class="sim04_dct_stat_value" style="color:#c0392b;">–</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Erro MSE</div><div id="sim04_dct_mse" class="sim04_dct_stat_value" style="color:#b9770e;">–</div></div>
  </div>

  <!-- Grid de Visualização dos Blocos -->
  <div style="display:flex; gap:12px; flex-wrap:wrap; align-items:flex-start; justify-content:center; margin-bottom:14px;">
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Bloco Original (8×8)</div>
      <canvas id="sim04_dct_cvOrig" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Coef. DCT (abs, log)</div>
      <canvas id="sim04_dct_cvDCT" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Coef. Quantizados</div>
      <canvas id="sim04_dct_cvQuant" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Bloco Reconstruído</div>
      <canvas id="sim04_dct_cvRec" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
  </div>

  <!-- Controles -->
  <div class="sim04_dct_panel">
    <div class="sim04_dct_slider_container">
      <label>Qualidade JPEG:</label>
      <input type="range" id="sim04_dct_slider" min="1" max="100" value="50">
      <span id="sim04_dct_slVal" style="font-size:12px; font-family:monospace; font-weight:700; min-width:30px; color:#26241d;">50</span>
    </div>
    <div style="display:flex; gap:6px; flex-wrap:wrap;">
      <button data-q="10" style="flex:1;">Q=10</button>
      <button data-q="25" style="flex:1;">Q=25</button>
      <button data-q="50" style="flex:1;">Q=50</button>
      <button data-q="75" style="flex:1;">Q=75</button>
      <button data-q="95" style="flex:1;">Q=95</button>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04DCT(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const dct_block = [
      [52,55,61,66,70,61,64,73],
      [63,59,55,90,109,85,69,72],
      [62,59,68,113,144,104,66,73],
      [63,58,71,122,154,106,70,69],
      [67,61,68,104,126,88,68,70],
      [79,65,60,70,77,68,58,75],
      [85,71,64,59,55,61,65,83],
      [87,79,69,68,65,76,78,94]
    ];

    const Q_luma = [
      [16,11,10,16,24,40,51,61],[12,12,14,19,26,58,60,55],
      [14,13,16,24,40,57,69,56],[14,17,22,29,51,87,80,62],
      [18,22,37,56,68,109,103,77],[24,35,55,64,81,104,113,92],
      [49,64,78,87,103,121,120,101],[72,92,95,98,112,100,103,99]
    ];

    function dct1d(x) {
      const N = x.length, c = new Array(N).fill(0);
      for (let k = 0; k < N; k++) {
        let sum = 0;
        for (let n = 0; n < N; n++) sum += x[n] * Math.cos(Math.PI*(2*n+1)*k/(2*N));
        const alpha = k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N);
        c[k] = alpha * sum;
      }
      return c;
    }

    function idct1d(c) {
      const N = c.length, x = new Array(N).fill(0);
      for (let n = 0; n < N; n++) {
        let sum = 0;
        for (let k = 0; k < N; k++) {
          const alpha = k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N);
          sum += alpha * c[k] * Math.cos(Math.PI*(2*n+1)*k/(2*N));
        }
        x[n] = sum;
      }
      return x;
    }

    function dct2d(blk) {
      const N=8, rows=blk.map(r=>dct1d(r));
      const cols=[];
      for(let j=0;j<N;j++){const col=rows.map(r=>r[j]);cols.push(dct1d(col));}
      const out=Array.from({length:N},()=>new Array(N));
      for(let i=0;i<N;i++) for(let j=0;j<N;j++) out[i][j]=cols[j][i];
      return out;
    }

    function idct2d(C) {
      const N=8, cols=[];
      for(let j=0;j<N;j++){const col=C.map(r=>r[j]);cols.push(idct1d(col));}
      const rows=Array.from({length:N},()=>new Array(N));
      for(let i=0;i<N;i++) for(let j=0;j<N;j++) rows[i][j]=cols[j][i];
      return rows.map(r=>idct1d(r));
    }

    function getQ(quality) {
      const s = quality<50 ? 5000/quality : 200-2*quality;
      return Q_luma.map(row=>row.map(v=>Math.max(1,Math.min(255,Math.round(v*s/100)))));
    }

    function drawPixels(canvas, data, minV, maxV) {
      const ctx = canvas.getContext('2d');
      const sz = canvas.width/8;
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        const v = (data[i][j]-minV)/(maxV-minV);
        const g = Math.round(v*255);
        ctx.fillStyle='rgb('+g+','+g+','+g+')';
        ctx.fillRect(j*sz,i*sz,sz,sz);
        ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=0.5;
        ctx.strokeRect(j*sz,i*sz,sz,sz);
        ctx.fillStyle=g>128?'#26241d':'#fafaf7';
        ctx.font='bold ' + Math.round(sz*0.28) + 'px monospace';
        ctx.textAlign='center'; ctx.textBaseline='middle';
        const val = data[i][j];
        ctx.fillText(Math.abs(val)>999?'…':String(Math.round(val)), j*sz+sz/2, i*sz+sz/2);
      }
    }

    function drawHeatmap(canvas, data) {
      const ctx = canvas.getContext('2d');
      const sz = canvas.width/8;
      const flat=data.flat(); const mn=Math.min(...flat), mx=Math.max(...flat);
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        const v=(data[i][j]-mn)/(mx-mn||1);
        const r=Math.round(v*220+35), gb=Math.round((1-v)*180+30);
        ctx.fillStyle='rgb('+r+','+gb+','+gb+')';
        ctx.fillRect(j*sz,i*sz,sz,sz);
        ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=0.5;
        ctx.strokeRect(j*sz,i*sz,sz,sz);
        ctx.fillStyle='#ffffff'; ctx.font='bold ' + Math.round(sz*0.22) + 'px monospace';
        ctx.textAlign='center'; ctx.textBaseline='middle';
        const val = data[i][j];
        ctx.fillText(Math.abs(val)>999?'…':String(Math.round(val)), j*sz+sz/2, i*sz+sz/2);
      }
    }

    function update(quality) {
      const Q = getQ(quality);
      const centered = dct_block.map(r=>r.map(v=>v-128));
      const C = dct2d(centered);
      const Cq = C.map((r,i)=>r.map((v,j)=>Math.round(v/Q[i][j])));
      const Cdq = Cq.map((r,i)=>r.map((v,j)=>v*Q[i][j]));
      const rec = idct2d(Cdq).map(r=>r.map(v=>Math.max(0,Math.min(255,Math.round(v+128)))));

      const Clog = C.map(r=>r.map(v=>Math.log1p(Math.abs(v))*(v>=0?1:-1)));
      const Cqlog = Cq.map(r=>r.map(v=>v));

      let nz=0, mse=0;
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        if(Cq[i][j]!==0) nz++;
        mse+=(dct_block[i][j]-rec[i][j])**2;
      }
      mse/=64;

      drawPixels(root.querySelector('#sim04_dct_cvOrig'), dct_block, 0, 255);
      drawHeatmap(root.querySelector('#sim04_dct_cvDCT'), Clog);
      drawHeatmap(root.querySelector('#sim04_dct_cvQuant'), Cqlog);
      drawPixels(root.querySelector('#sim04_dct_cvRec'), rec, 0, 255);

      root.querySelector('#sim04_dct_qual').textContent = quality;
      root.querySelector('#sim04_dct_nonzero').textContent = nz;
      root.querySelector('#sim04_dct_zeros').textContent = (64-nz);
      root.querySelector('#sim04_dct_mse').textContent = mse.toFixed(1);
    }

    window.dct_setQ = function(q){
      root.querySelector('#sim04_dct_slider').value = q;
      root.querySelector('#sim04_dct_slVal').textContent = q;
      update(q);
    };

    root.querySelector('#sim04_dct_slider').addEventListener('input', function(){
      root.querySelector('#sim04_dct_slVal').textContent = this.value;
      update(+this.value);
    });

    root.querySelectorAll('[data-q]').forEach(btn => {
      btn.addEventListener('click', function() {
        dct_setQ(parseInt(this.getAttribute('data-q'), 10));
      });
    });

    update(50);
  }

  function tryInitSim04DCT(){
    var root = document.getElementById('sim-05-dct');
    if (root) initSim04DCT(root); else setTimeout(tryInitSim04DCT, 200);
  }
  tryInitSim04DCT();
})();
</script>
""")

**Figura 5.17:** Simulador interativo de compressão DCT-JPEG: ajuste o fator de qualidade e visualize em tempo real os coeficientes zerados, o bloco reconstruído e o erro de quantização.


## 5.10 Comparação de Formatos de Imagem

A escolha de um formato de armazenamento digital impacta diretamente o compromisso entre qualidade visual, tamanho de arquivo e custo computacional de decodificação. Os três formatos de maior relevância para arquiteturas *web* e sistemas de computação visual são o JPEG, o PNG e o WebP.

### 5.10.1 Características dos Formatos

A [Tabela 5.8](#tbl-05-formatos) sintetiza as propriedades estruturais dos principais formatos de imagem rasterizados.

<a id="tbl-05-formatos"></a>

**Tabela 5.8:** Comparação estrutural entre os principais formatos de imagem rasterizados.

| Característica | JPEG | PNG | WebP |
|:---|:---:|:---:|:---:|
| **Compressão** | Com perda | Sem perda | Com e sem perda. |
| **Transparência (canal alfa)** | Não | Sim | Sim. |
| **Suporte a animação** | Não | Limitado (APNG) | Sim. |
| **Algoritmo base** | DCT + Huffman | DEFLATE (LZ77 + Huffman) | VP8 / VP8L. |
| **Melhor para** | Fotografia | Gráficos, texto e ícones | Uso universal em ambiente Web. |
| **Pior para** | Texto e bordas nítidas | Imagens fotográficas complexas | Compatibilidade legada. |


### 5.10.2 Métricas de Avaliação de Qualidade

Duas métricas objetivas são amplamente adotadas para quantificar a distorção introduzida por processos de compressão:

**Pico da Relação Sinal-Ruído (PSNR, *Peak Signal-to-Noise Ratio*):**
<a id="eq-05-psnr"></a>
$$
\text{PSNR} = 10\,\log_{10}\!\left(\frac{L^2}{\text{MSE}}\right) \quad [\text{dB}] \tag{5.10}
$$


onde $L = 255$ para imagens quantizadas em 8 bits e $\text{MSE}$ representa o **Erro Quadrático Médio** (*Mean Squared Error*). Valores de PSNR acima de 40 dB indicam excelente fidelidade; entre 30 dB e 40 dB representam boa qualidade; e valores inferiores a 30 dB correspondem a degradações visuais facilmente perceptíveis.

**Índice de Similaridade Estrutural (SSIM, *Structural Similarity Index*):**
<a id="eq-05-ssim"></a>
$$
\text{SSIM}(f,g) = \frac{(2\mu_f\mu_g + c_1)(2\sigma_{fg} + c_2)}{(\mu_f^2+\mu_g^2+c_1)(\sigma_f^2+\sigma_g^2+c_2)} \tag{5.11}
$$


O SSIM avalia janelas locais da imagem com base em três componentes complementares: **luminância** ($\mu_f, \mu_g$), **contraste** ($\sigma_f, \sigma_g$) e **estrutura** ($\sigma_{fg}$), ponderados por constantes de estabilidade $c_1$ e $c_2$. O índice varia no intervalo $[-1, 1]$, onde a unidade representa a identidade perfeita. Ao contrário do PSNR, o SSIM considera a organização espacial dos erros, alinhando-se à percepção do sistema visual humano (SVH).

> ### 📝 PSNR vs SSIM: Aplicação de Métricas Perceptuais
>
> O PSNR possui formulação matemática simples e baixo custo computacional, contudo, tende a superestimar a qualidade em imagens com distorções localizadas ou subestimá-la em variações globais de brilho toleradas pelo observador. O SSIM modela com maior fidelidade a percepção biológica, mas exige maior esforço de processamento. Para análises rigorosas de codificadores, recomenda-se reportar ambas as métricas estatísticas em caráter complementar.

### 5.10.3 Inspeção Visual: Natureza dos Artefatos de Compressão

A natureza matemática do codificador dita o tipo de degradação introduzida em taxas de bits reduzidas. Conforme ilustrado na [Figura 5.18](#fig-05-zoom-artefatos), a compressão agressiva via DCT no padrão JPEG segmenta a imagem em malhas rígidas, gerando os **artefatos de bloco** (*blocking artifacts*). Em contrapartida, algoritmos baseados em codificação preditiva ou representações submetidas a transformadas espaciais avançadas (como o WebP e o JPEG 2000) eliminam as descontinuidades de bloco, mas introduzem perda de textura fina e borramentos característicos ao redor de bordas de alto contraste.

In [ ]:
%%writefile tmp/fig_05_zoom_artefatos.cpp
#define MM_OUT "tmp/fig_05_zoom_artefatos.png"
//| label: fig-05-zoom-artefatos
//| fig-cap: "Análise comparativa de artefatos de compressão sob fator de qualidade reduzido (Q=10). À esquerda, observa-se o artefato de bloco característico da discretização por DCT no JPEG. À direita, evidencia-se o efeito de atenuação e suavização de bordas intrínseco ao padrão WebP."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <opencv2/imgcodecs.hpp>
#include <opencv2/imgproc.hpp>
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>
#include <sys/stat.h>
#include "morph.hpp"

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_19.png");
// [pdi:state-io:end]

    // img_gray is provided as mm::Image

    // Cria diretório de saída se não existir
    std::filesystem::create_directories("imagens/comp_test");

    // Converte mm::Image para cv::Mat
    cv::Mat imgGrayMat;
    if (img_gray.channels == 1) {
        imgGrayMat = cv::Mat(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    } else {
        cv::Mat temp(img_gray.h, img_gray.w, CV_8UC3, img_gray.data.data());
        cv::cvtColor(temp, imgGrayMat, cv::COLOR_BGR2GRAY);
    }

    // Cria versão colorida para cv::imwrite
    cv::Mat imgColor;
    cv::cvtColor(imgGrayMat, imgColor, cv::COLOR_GRAY2BGR);

    // Grava os arquivos comprimidos
    std::vector<int> jpegParams = {cv::IMWRITE_JPEG_QUALITY, 10};
    std::vector<int> webpParams = {cv::IMWRITE_WEBP_QUALITY, 10};
    cv::imwrite("imagens/comp_test/camera_q10.jpg", imgColor, jpegParams);
    cv::imwrite("imagens/comp_test/camera_q10.webp", imgColor, webpParams);

    // Extração de região de interesse para visualização de artefatos (Zoom de 4x)
    // Região: linhas 120-200, colunas 150-230
    cv::Mat regionOriginal = imgGrayMat(cv::Rect(150, 120, 80, 80));
    cv::Mat zoomOriginal;
    cv::resize(regionOriginal, zoomOriginal, cv::Size(320, 320), 0, 0, cv::INTER_NEAREST);

    // Lê arquivos comprimidos
    cv::Mat recJpeg = cv::imread("imagens/comp_test/camera_q10.jpg", cv::IMREAD_GRAYSCALE);
    cv::Mat recWebp = cv::imread("imagens/comp_test/camera_q10.webp", cv::IMREAD_GRAYSCALE);

    // Extrai e aplica zoom nas regiões correspondentes
    cv::Mat regionJpeg = recJpeg(cv::Rect(150, 120, 80, 80));
    cv::Mat zoomJpeg;
    cv::resize(regionJpeg, zoomJpeg, cv::Size(320, 320), 0, 0, cv::INTER_NEAREST);

    cv::Mat regionWebp = recWebp(cv::Rect(150, 120, 80, 80));
    cv::Mat zoomWebp;
    cv::resize(regionWebp, zoomWebp, cv::Size(320, 320), 0, 0, cv::INTER_NEAREST);

    // Converte para mm::Image para exibição
    mm::Image mmOriginal(zoomOriginal.rows, zoomOriginal.cols, 1);
    std::memcpy(mmOriginal.data.data(), zoomOriginal.data, mmOriginal.data.size());

    mm::Image mmJpeg(zoomJpeg.rows, zoomJpeg.cols, 1);
    std::memcpy(mmJpeg.data.data(), zoomJpeg.data, mmJpeg.data.size());

    mm::Image mmWebp(zoomWebp.rows, zoomWebp.cols, 1);
    std::memcpy(mmWebp.data.data(), zoomWebp.data, mmWebp.data.size());

    // Exibe as três imagens lado a lado
    std::vector<mm::Image> images = {mmOriginal, mmJpeg, mmWebp};
    std::vector<std::string> titles = {
        "Zoom Original", 
        "JPEG Q=10 (Artefato de Bloco)", 
        "WebP Q=10 (Suavização)"
    };
    mm::show(images, MM_OUT, titles, 3);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_zoom_artefatos.cpp -o tmp/fig_05_zoom_artefatos \
  && ./tmp/fig_05_zoom_artefatos \
  && test -f "tmp/fig_05_zoom_artefatos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_zoom_artefatos.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_zoom_artefatos.png"), figsize=(14, 5))

**Figura 5.18:** Análise comparativa de artefatos de compressão sob fator de qualidade reduzido ($Q=10$). À esquerda, observa-se o artefato de bloco característico da discretização por DCT no JPEG. À direita, evidencia-se o efeito de atenuação e suavização de bordas intrínseco ao padrão WebP.


### 5.10.4 Avaliação Quantitativa e Espacial da Compressão

A validação dos algoritmos de compressão com perda exige uma análise que correlacione o custo de armazenamento à fidelidade do sinal reconstruído. Essa avaliação é realizada de forma complementar através de curvas de desempenho global e pelo mapeamento local das distorções induzidas pelos codificadores.

#### 5.10.4.1 Curvas de Taxa-Distorção

A [Figura 5](#fig-05-formatos-comparacao) apresenta a avaliação empírica do *pipeline* JPEG e WebP por meio de **curvas de taxa-distorção**, que monitoram o ganho de compressão (tamanho do arquivo em KB) em função do PSNR. O formato PNG atua como linha de base ideal ($\text{PSNR} = \infty$), pois sua natureza *lossless* impede qualquer degradação, embora demande um volume de dados substancialmente maior. 

A análise das curvas demonstra a superioridade e a eficiência do padrão WebP sobre o JPEG tradicional: para atingir um mesmo patamar de fidelidade matemática (como a faixa de excelente qualidade, onde $\text{PSNR} > 40\text{ dB}$), o codificador WebP gera arquivos significativamente menores. Esse comportamento traduz o impacto prático da evolução dos algoritmos na otimização de sistemas de transmissão e armazenamento digital.

> ### 📝 5.11 Tamanho original da imagem
>
> A imagem *Cameraman* ($256 \times 256$ pixels em escala de cinza) ocupa **64 KB** em formato bruto (sem compressão). Como referência, o PNG *lossless* comprime esse volume para **36,2 KB** — evidenciando que a compressão sem perdas já reduz significativamente o armazenamento para imagens com regiões homogêneas. Em contrapartida, os formatos com perda (JPEG e WebP) atingem tamanhos ainda menores: o JPEG com qualidade 95 ocupa 22,3 KB (PSNR ≈ 45 dB), enquanto o WebP com qualidade 90 atinge 12,5 KB com PSNR equivalente, demonstrando sua superioridade em eficiência de compressão.

#### 5.11.0.1 Mapeamento Espacial de Erros e Correlação Perceptual

Embora o PSNR ofereça um indicativo numérico rápido, métricas globais falham em discriminar como a perda de informação se distribui geometricamente sobre a imagem. A [Figura 5.19](#fig-05-ssim-artefatos) soluciona essa limitação ao associar as reconstruções em diferentes qualidades aos seus respectivos mapas de erro absoluto e ao SSIM.

Os mapas residuais — obtidos pela diferença absoluta normalizada entre a imagem original e a comprimida — revelam a assinatura espacial intrínseca de cada arquitetura de codificação:

* **Em altas qualidades ($Q=95$ a $Q=75$):** As distorções concentram-se predominantemente ao redor de transições abruptas de intensidade (bordas), fruto do espelhamento espectral decorrente do descarte de altas frequências. O índice SSIM permanece próximo à unidade, atestando a integridade das estruturas originais.
* **Em qualidades agressivas ($Q=50$ a $Q=25$):** O erro assume uma estrutura de malha ortogonal regularizada. Esse padrão geométrico evidencia o surgimento dos **artefatos de bloco** (*blocking artifacts*), indicando que a quantização severa corrompeu a correlação espacial entre blocos adjacentes de $8 \times 8$ pixels. 

O SSIM captura essa degradação morfológica de forma muito mais sensível que o PSNR, penalizando o escore final à medida que a organização estrutural e as texturas finas — às quais o sistema visual humano é altamente responsivo — são eliminadas pelo codificador.

In [ ]:
# Ainda não portado para esta linguagem nesta versão — referência conceitual em Python.

import os
import numpy as np
import cv2

try:
    from skimage.metrics import structural_similarity as ssim
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "scikit-image", "-q"])
    from skimage.metrics import structural_similarity as ssim

# Garante a existência do diretório de testes
os.makedirs("imagens/comp_test", exist_ok=True)

qualidades_ssim = [25, 50, 75, 95]
imgs_ssim   = [img_gray]
titles_ssim = ["Original"]

for q in qualidades_ssim:
    path = f"imagens/comp_test/camera_ssim_q{q}.jpg"
    
    # GRAVAÇÃO FORÇADA: Gera e grava o JPEG com a qualidade atual no caminho correto
    img_compactada = jpeg_quality_compress(img_gray, qualidade=q)
    cv2.imwrite(path, img_compactada)
    
    # Leitura segura do arquivo recém-gravado
    rec = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    
    if rec is None: 
        continue
        
    if rec.shape != img_gray.shape:
        rec = cv2.resize(rec, (img_gray.shape[1], img_gray.shape[0]))
    
    psnr_v = cv2.PSNR(img_gray, rec)
    ssim_v, _ = ssim(img_gray, rec, full=True)
    
    # Diferença absoluta normalizada para evidenciar a estrutura espacial do erro
    diff_vis = cv2.normalize(np.abs(img_gray.astype(float) - rec.astype(float)),
                             None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    imgs_ssim  += [rec, diff_vis]
    titles_ssim += [f"Q={q}\nPSNR={psnr_v:.1f}dB | SSIM={ssim_v:.3f}",
                    f"Mapa de erro (Q={q})\n(Bordas e blocagem)"]

mm.show(imgs_ssim, titles=titles_ssim, cols=3, figsize=(14, 14))

**Figura 5.19:** Análise espacial de degradação: imagens reconstruídas e respectivos mapas de erro absoluto normalizados para diferentes fatores de qualidade JPEG.


> ### 📝 5.12 Interpretando os mapas de erro
>
> Os mapas de erro apresentados foram **normalizados individualmente** (`cv2.NORM_MINMAX`) para maximizar o contraste visual e revelar a estrutura espacial das distorções. Isso significa que:
>
> - Em **Q=95**, o erro absoluto é da ordem de **0.5–1.5 níveis de cinza** (imperceptível visualmente), mas a normalização o amplifica para preto-branco para evidenciar sua localização em bordas e transições.
> - Em **Q=25**, o erro absoluto é **10–20 vezes maior** (5–15 níveis de cinza), mas a normalização também o leva ao mesmo intervalo [0, 255].
>
> Portanto, **a intensidade do branco nos mapas NÃO é comparável entre diferentes qualidades** — os mapas servem apenas para revelar a **assinatura espacial** do erro (bordas vs blocos), não sua magnitude. A magnitude correta é dada pelos valores de PSNR e SSIM, que mostram claramente que Q=95 tem erro muito menor que Q=25.

### Síntese — Compressão JPEG

O processo de compressão no padrão JPEG baseia-se na aplicação combinada de transformações espaciais, perceptuais e estatísticas para reduzir as redundâncias de uma imagem. A [Tabela 5.9](#tbl-05-sintese-jpeg) resume o papel de cada etapa no *pipeline* e seu respectivo impacto na redução de dados.

<a id="tbl-05-sintese-jpeg"></a>

**Tabela 5.9:** Síntese das etapas do *pipeline* de compressão JPEG e seus respectivos impactos.

| Etapa | Operação Analítica | Mecanismo de Ganho / Compressão |
|:---|:---|:---|
| **Conversão $YC_bC_r$** | Isolamento dos canais de luminância e crominância. | Modela a percepção do SVH, permitindo tratar cor e brilho de forma independente. |
| **Subamostragem 4:2:0** | Redução da resolução espacial dos canais de cor ($C_b$ e $C_r$). | Elimina aproximadamente 50% dos dados brutos com impacto visual mínimo. |
| **DCT $8 \times 8$** | Mapeamento do domínio espacial para o domínio de frequências espaciais. | Compactação de energia, concentrando a informação vital nos primeiros coeficientes. |
| **Quantização Linear** | Divisão inteira dos coeficientes por uma matriz de ponderação $Q(u,v)$. | Principal fonte de compressão com perda; elimina altas frequências imperceptíveis. |
| **Codificação Entrópica** | Aplicação de algoritmos RLE e codificação de Huffman. | Compressão estatística sem perda, otimizada pelas longas corridas de coeficientes nulos. |


#### Artefatos de Degradação Característicos

A aplicação de taxas de compressão excessivamente agressivas (fatores de qualidade reduzidos) introduz distorções previsíveis na imagem reconstruída, decorrentes das limitações matemáticas do modelo:

* **Artefatos de bloco (*blocking artifacts*):** Descontinuidades geométricas visíveis nas fronteiras dos blocos de $8 \times 8$ pixels, causadas pela perda de correlação espacial após a quantização severa das componentes AC.
* **Efeito de espalhamento (*ringing*):** Oscilações fantasmas ou distorções de "fumaça" ao redor de bordas nítidas e de alto contraste, provocadas pela eliminação abrupta de harmônicos de alta frequência necessários para reconstruir funções degrau.
* **Perda de textura fina:** Atenuação de detalhes de alta frequência e baixo contraste (como gramados, tecidos ou porosidade), fazendo com que regiões originalmente texturizadas assumam um aspecto excessivamente liso ou homogeneizado.

## 5.13 Aplicação Prática: Remoção de Ruído por Filtragem Híbrida

Reunindo as técnicas consolidadas ao longo deste capítulo, apresenta-se um *pipeline* completo de **restauração de imagens** que combina a análise espectral no domínio da frequência com a filtragem adaptativa no domínio espacial. O objetivo é atenuar um ruído misto (composto por degradação Gaussiana e interferência periódica) preservando ao máximo os detalhes estruturais da imagem original.

$$
\text{Imagem Ruidosa} \xrightarrow{\text{FFT2}} \xrightarrow{\text{Filtro Notch Gaussiano}} \xrightarrow{\text{IFFT2}} \xrightarrow{\text{Filtro Bilateral}} \text{Imagem Restaurada}
$$

> ### 📝 Avaliação Complementar: PSNR vs. SSIM
>
> O par de métricas estatísticas PSNR e SSIM fornece uma avaliação qualitativa e morfológica complementar do processo de restauração:
>
> * **PSNR:** Penaliza uniformemente o desvio quadrático médio pixel a pixel.
> * **SSIM:** Avalia a preservação de estruturas locais perceptualmente relevantes (luminância, contraste e contornos).
>
> Na prática, existe um compromisso analítico (*trade-off*) entre **redução de ruído** e **preservação de detalhes**: filtros espaciais excessivamente agressivos atenuam bem o ruído de alta frequência, mas degradam texturas finas e suavizam bordas nítidas — o que **reduz simultaneamente** tanto o PSNR quanto o SSIM em relação à imagem original. O desafio do projeto de filtros é encontrar o ponto de equilíbrio que maximize ambas as métricas, garantindo uma restauração fiel e visualmente agradável.

### 5.13.1 Análise de Desempenho e Conclusão do Capítulo

Os resultados numéricos e visuais gerados pela [Figura 5.20](#fig-05-pipeline-denoising) demonstram a relevância prática de associar diferentes domínios de processamento. A inserção simultânea de ruído periódico e estocástico corrompe as propriedades morfológicas do sinal, reduzindo severamente os índices de similaridade e a relação sinal-ruído da imagem de referência.

O isolamento e a supressão dos picos harmônicos no domínio da frequência por meio da máscara *notch* removem as franjas de interferência senoidais espalhadas sobre o espaço bi-dimensional. Como evidenciado nos dados impressos da [Figura 5.20](#fig-05-pipeline-denoising), essa filtragem cirúrgica promove um salto imediato e substancial na métrica PSNR. Contudo, o ruído Gaussiano de alta frequência permanece ativo de forma homogênea no espectro, exigindo uma abordagem complementar.

A restauração final é consolidada no domínio espacial com a introdução do filtro bilateral. Diferentemente de operadores passa-baixas convencionais (como o Gaussiano ou de média), que suavizariam indiscriminadamente o ruído e os contornos estruturais, a filtragem bilateral calcula pesos ponderados pela proximidade geométrica e pela diferença de intensidade radiométrica. Esse comportamento adaptativo atenua as flutuações estocásticas remanescentes nas regiões de transição suave e preserva a nitidez das bordas espaciais. 

A convergência de ambas as abordagens resulta em uma **melhoria substancial e simultânea** do PSNR e do SSIM em relação à imagem ruidosa — embora os valores finais permaneçam inferiores aos da imagem original (PSNR = $\infty$, SSIM = 1,0), devido à perda inevitável de informações espectrais e texturais durante os processos de filtragem. A atenuação suave (gaussiana) dos picos no espectro evita artefatos de *ringing*, enquanto o filtro bilateral elimina o ruído estocástico residual sem comprometer a nitidez das bordas. Os resultados comprovam a eficácia e a complementaridade prática das ferramentas de análise de frequência apresentadas neste capítulo, demonstrando que a filtragem híbrida (frequência + espacial) é superior a qualquer abordagem isolada para a restauração de imagens degradadas por ruído misto.

In [ ]:
%%writefile tmp/fig_05_pipeline_denoising.cpp
#define MM_OUT "tmp/fig_05_pipeline_denoising.png"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <cmath>
#include <vector>
#include <string>
#include <numeric>
#include <algorithm>
#include "morph.hpp"

//| label: fig-05-pipeline-denoising
//| fig-cap: "*Pipeline* completo de remoção de ruído misto: (1) adição de ruído gaussiano e periódico; (2) identificação de picos de interferência no espectro de frequências; (3) aplicação de máscara *notch* com atenuação gaussiana suave; (4) pós-processamento via filtro bilateral para eliminação do ruído estocástico residual."
//| echo: true
//| output: true

// Função para calcular SSIM (Wang et al.)
double computeSSIM(const cv::Mat& img1, const cv::Mat& img2) {
    const double C1 = 6.5025, C2 = 58.5225;
    cv::Mat I1, I2;
    img1.convertTo(I1, CV_32F);
    img2.convertTo(I2, CV_32F);

    cv::Mat I1_2 = I1.mul(I1), I2_2 = I2.mul(I2), I1_I2 = I1.mul(I2);

    cv::Mat mu1, mu2, mu1_2, mu2_2, mu1_mu2, sigma1_2, sigma2_2, sigma12;
    cv::GaussianBlur(I1, mu1, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(I2, mu2, cv::Size(11, 11), 1.5);

    mu1_2 = mu1.mul(mu1);
    mu2_2 = mu2.mul(mu2);
    mu1_mu2 = mu1.mul(mu2);

    cv::GaussianBlur(I1_2, sigma1_2, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(I2_2, sigma2_2, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(I1_I2, sigma12, cv::Size(11, 11), 1.5);

    sigma1_2 -= mu1_2;
    sigma2_2 -= mu2_2;
    sigma12 -= mu1_mu2;

    cv::Mat t1 = 2 * mu1_mu2 + C1;
    cv::Mat t2 = 2 * sigma12 + C2;
    cv::Mat t3 = t1.mul(t2);

    cv::Mat t4 = mu1_2 + mu2_2 + C1;
    cv::Mat t5 = sigma1_2 + sigma2_2 + C2;
    cv::Mat t6 = t4.mul(t5);

    cv::Mat ssim_map = t3 / t6;

    cv::Scalar mssim = cv::mean(ssim_map);
    return mssim[0];
}

// Função FFT shift manual
void fftShift(cv::Mat& src) {
    int cx = src.cols / 2;
    int cy = src.rows / 2;

    if (src.channels() == 1) {
        cv::Mat q0(src, cv::Rect(0, 0, cx, cy));
        cv::Mat q1(src, cv::Rect(cx, 0, cx, cy));
        cv::Mat q2(src, cv::Rect(0, cy, cx, cy));
        cv::Mat q3(src, cv::Rect(cx, cy, cx, cy));

        cv::Mat tmp;
        q0.copyTo(tmp);
        q3.copyTo(q0);
        tmp.copyTo(q3);

        q1.copyTo(tmp);
        q2.copyTo(q1);
        tmp.copyTo(q2);
    } else {
        // Para canal complexo (2 canais)
        std::vector<cv::Mat> channels;
        cv::split(src, channels);

        for (auto& ch : channels) {
            cv::Mat q0(ch, cv::Rect(0, 0, cx, cy));
            cv::Mat q1(ch, cv::Rect(cx, 0, cx, cy));
            cv::Mat q2(ch, cv::Rect(0, cy, cx, cy));
            cv::Mat q3(ch, cv::Rect(cx, cy, cx, cy));

            cv::Mat tmp;
            q0.copyTo(tmp);
            q3.copyTo(q0);
            tmp.copyTo(q3);

            q1.copyTo(tmp);
            q2.copyTo(q1);
            tmp.copyTo(q2);
        }

        cv::merge(channels, src);
    }
}

void fftShiftInverse(cv::Mat& src) {
    fftShift(src); // Inverso é o mesmo
}

// Função equivalente a suprimir_pico_gaussiano
void suprimirPicoGaussiano(cv::Mat& mask, int cy, int cx, double sigma = 3.0) {
    cv::Mat dist(mask.rows, mask.cols, CV_32F);
    for (int y = 0; y < mask.rows; ++y) {
        for (int x = 0; x < mask.cols; ++x) {
            float dy = y - cy;
            float dx = x - cx;
            dist.at<float>(y, x) = std::sqrt(dy * dy + dx * dx);
        }
    }

    cv::Mat notch;
    cv::exp(-(dist.mul(dist)) / (2 * sigma * sigma), notch);

    cv::Mat one = cv::Mat::ones(mask.size(), mask.type());
    cv::Mat result = one - notch;
    mask = mask.mul(result);
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_19.png");
// [pdi:state-io:end]

    // img_gray é fornecido automaticamente

    // ── 1. Construção do ruído misto ─────────────────────────────────────────────
    cv::RNG rng(42);
    int h_img = img_gray.h;
    int w_img = img_gray.w;

    // Converte a imagem para matriz OpenCV
    cv::Mat img_gray_mat(h_img, w_img, CV_8UC1, img_gray.data.data());
    cv::Mat img_gray_f;
    img_gray_mat.convertTo(img_gray_f, CV_32F);

    // Ruído gaussiano
    cv::Mat ruido_gauss(h_img, w_img, CV_32F);
    rng.fill(ruido_gauss, cv::RNG::NORMAL, 0, 15);

    // Ruído periódico
    int u0 = 15, v0 = 10;
    cv::Mat X2(h_img, w_img, CV_32F), Y2(h_img, w_img, CV_32F);
    for (int y = 0; y < h_img; ++y) {
        for (int x = 0; x < w_img; ++x) {
            X2.at<float>(y, x) = x;
            Y2.at<float>(y, x) = y;
        }
    }

    cv::Mat ruido_period(h_img, w_img, CV_32F);
    for (int y = 0; y < h_img; ++y) {
        for (int x = 0; x < w_img; ++x) {
            ruido_period.at<float>(y, x) = 30 * std::sin(2 * CV_PI * (u0 * X2.at<float>(y, x) / w_img + 
                                                                    v0 * Y2.at<float>(y, x) / h_img));
        }
    }

    // Imagem ruidosa
    cv::Mat img_noisy_f = img_gray_f + ruido_gauss + ruido_period;
    cv::Mat img_noisy;
    cv::threshold(img_noisy_f, img_noisy, 255, 255, cv::THRESH_TRUNC);
    cv::threshold(img_noisy, img_noisy, 0, 0, cv::THRESH_TOZERO);
    img_noisy.convertTo(img_noisy, CV_8U);

    // ── 2. Espectro e identificação dos picos ────────────────────────────────────
    cv::Mat F_n, F_n_shifted;
    cv::Mat img_noisy_f64;
    img_noisy.convertTo(img_noisy_f64, CV_64F);
    cv::dft(img_noisy_f64, F_n, cv::DFT_COMPLEX_OUTPUT);
    fftShift(F_n);

    // Magnitude do espectro
    std::vector<cv::Mat> F_channels;
    cv::split(F_n, F_channels);
    cv::Mat mag;
    cv::magnitude(F_channels[0], F_channels[1], mag);
    cv::Mat log_mag;
    cv::log(mag + 1, log_mag);

    cv::Mat mag_n;
    cv::normalize(log_mag, mag_n, 0, 255, cv::NORM_MINMAX, CV_8U);

    // ── 3. Notch gaussiano nos picos periódicos ──────────────────────────────────
    int cy0 = h_img / 2, cx0 = w_img / 2;
    cv::Mat mascara_notch = cv::Mat::ones(h_img, w_img, CV_64F);

    int dvs[4][2] = {{+v0, +u0}, {-v0, -u0}, {+v0, -u0}, {-v0, +u0}};
    for (auto& dv : dvs) {
        suprimirPicoGaussiano(mascara_notch, cy0 + dv[0], cx0 + dv[1], 3.0);
    }

    // Aplicar máscara no espectro
    std::vector<cv::Mat> F_n_masked(2);
    cv::Mat mask_2ch[2];
    // Converte máscara para 64F e combina com espectro
    cv::Mat mascara_f;
    mascara_notch.convertTo(mascara_f, CV_64F);

    // Multiplica cada canal
    cv::Mat F_n_mul[2];
    cv::multiply(F_channels[0], mascara_f, F_n_mul[0]);
    cv::multiply(F_channels[1], mascara_f, F_n_mul[1]);

    cv::Mat F_n_filtered;
    cv::merge(F_n_mul, 2, F_n_filtered);

    // IFFT
    fftShiftInverse(F_n_filtered);
    cv::Mat img_notch_real;
    cv::idft(F_n_filtered, img_notch_real, cv::DFT_SCALE | cv::DFT_REAL_OUTPUT);

    cv::Mat img_notch;
    cv::threshold(img_notch_real, img_notch, 255, 255, cv::THRESH_TRUNC);
    cv::threshold(img_notch, img_notch, 0, 0, cv::THRESH_TOZERO);
    img_notch.convertTo(img_notch, CV_8U);

    // ── 4. Filtro Bilateral: remoção do ruído gaussiano residual ─────────────────
    cv::Mat img_den;
    cv::bilateralFilter(img_notch, img_den, 7, 25, 7);

    // ── Cálculo das Métricas de Validação ────────────────────────────────────────
    double psnr_n = cv::PSNR(img_gray_mat, img_noisy);
    double ssim_n = computeSSIM(img_gray_mat, img_noisy);
    double psnr_no = cv::PSNR(img_gray_mat, img_notch);
    double ssim_no = computeSSIM(img_gray_mat, img_notch);
    double psnr_d = cv::PSNR(img_gray_mat, img_den);
    double ssim_d = computeSSIM(img_gray_mat, img_den);

    printf("%20s | %9s | %6s\n", "Etapa", "PSNR (dB)", "SSIM");
    printf("%s\n", std::string(42, '-').c_str());
    printf("%20s | %9.2f | %6.4f\n", "Ruidosa (gauss+per)", psnr_n, ssim_n);
    printf("%20s | %9.2f | %6.4f\n", "Após notch", psnr_no, ssim_no);
    printf("%20s | %9.2f | %6.4f\n", "Notch + bilateral", psnr_d, ssim_d);

    // Preparar visualização
    cv::Mat mascara_vis;
    mascara_notch.convertTo(mascara_vis, CV_8U, 255.0);

    // Converter para mm::Image e mostrar
    cv::Mat img_gray_out(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    cv::Mat img_noisy_out(img_noisy);
    cv::Mat mag_n_out(mag_n);
    cv::Mat mascara_vis_out(mascara_vis);
    cv::Mat img_notch_out(img_notch);
    cv::Mat img_den_out(img_den);

    mm::Image img_gray_mm(img_gray);
    mm::Image img_noisy_mm(img_noisy.rows, img_noisy.cols, 1);
    std::memcpy(img_noisy_mm.data.data(), img_noisy.data, img_noisy_mm.data.size());
    mm::Image mag_n_mm(mag_n.rows, mag_n.cols, 1);
    std::memcpy(mag_n_mm.data.data(), mag_n.data, mag_n_mm.data.size());
    mm::Image mascara_vis_mm(mascara_vis.rows, mascara_vis.cols, 1);
    std::memcpy(mascara_vis_mm.data.data(), mascara_vis.data, mascara_vis_mm.data.size());
    mm::Image img_notch_mm(img_notch.rows, img_notch.cols, 1);
    std::memcpy(img_notch_mm.data.data(), img_notch.data, img_notch_mm.data.size());
    mm::Image img_den_mm(img_den.rows, img_den.cols, 1);
    std::memcpy(img_den_mm.data.data(), img_den.data, img_den_mm.data.size());

    std::vector<std::string> titles = {
        "Original",
        "Ruidosa\nPSNR=" + std::to_string(psnr_n) + " dB",
        "Espectro\n(picos visíveis)",
        "Máscara notch\n(gaussiana suave)",
        "Após notch\nPSNR=" + std::to_string(psnr_no) + " dB",
        "Notch + bilateral\nPSNR=" + std::to_string(psnr_d) + " dB  SSIM=" + std::to_string(ssim_d)
    };

    mm::show({img_gray_mm, img_noisy_mm, mag_n_mm, mascara_vis_mm, img_notch_mm, img_den_mm}, 
             MM_OUT, titles, 6);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV $(pkg-config --cflags --libs opencv4) tmp/fig_05_pipeline_denoising.cpp -o tmp/fig_05_pipeline_denoising \
  && ./tmp/fig_05_pipeline_denoising \
  && test -f "tmp/fig_05_pipeline_denoising.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_pipeline_denoising.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_pipeline_denoising.png"), figsize=(20, 4))

**Figura 5.20:** *Pipeline* completo de remoção de ruído misto: (1) adição de ruído gaussiano e periódico; (2) identificação de picos de interferência no espectro de frequências; (3) aplicação de máscara *notch* com atenuação gaussiana suave; (4) pós-processamento via filtro bilateral para eliminação do ruído estocástico residual.


## 5.14 Resumo do Capítulo

A transição do **domínio espacial** para o **domínio da frequência** revela a distribuição espectral de energia da imagem, estabelecendo a base analítica para a filtragem avançada, restauração e compressão de dados. A articulação estrutural desses conceitos é sintetizada no mapa conceitual da [Figura 5.21](#fig-05-mapa-conceitual).

<figure id="fig-05-mapa-conceitual" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-05-mapa-conceitual.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 5.21:</strong> Mapa conceitual das transformações e propriedades no domínio da frequência.</figcaption>
</figure>

### Fundamentos Essenciais

* **DFT e Percepção Visual:** O espectro decompõe a imagem em componentes harmônicas. A **fase** retém a inteligibilidade geométrica da cena e a localização de contornos, enquanto a **magnitude** dita a distribuição de contraste e amplitudes globais.
* **Eficiência Algorítmica:** O Teorema da Convolução viabiliza o processamento de máscaras de grande escala no domínio da frequência via FFT, reduzindo a complexidade computacional assintótica de $O(N^2 K^2)$ no espaço para $O(N^2 \log N)$.
* **Fenômeno de *Ringing*:** Cortes abruptos no espectro (Filtros Ideais) geram oscilações espaciais indesejadas (fenômeno de Gibbs). A atenuação suave por filtros de **Butterworth** ou **Gaussianos** elimina essas descontinuidades.
* **Análise Multirresolução via *Wavelets*:** Superando o caráter puramente global de Fourier, a DWT captura a frequência e a localização espacial simultaneamente, fundamentando o padrão JPEG 2000 e subsidiando representações hierárquicas análogas às extrações de feições em Redes Neurais Convolucionais (CNNs).
* **Compressão Perceptual (DCT):** O *pipeline* JPEG explora as limitações de contraste do sistema visual humano em altas frequências espaciais. A DCT isola a energia de blocos $8 \times 8$, permitindo que a quantização descarte coeficientes AC de detalhes finos sem prejuízo perceptual severo.


**Próximos Passos:** O **Capítulo 6** inaugura a Parte II da obra, aplicando as ferramentas de processamento de imagens na resolução de problemas reais de inspeção industrial. Serão exploradas técnicas de **segmentação e análise de formas** para a detecção automática de falhas em linhas de produção — desde a identificação de defeitos superficiais em peças até a leitura *QRCode* em provas, consolidando a ponte entre a teoria apresentada na Parte I e as demandas práticas da visão computacional.

## 5.15 🤖 Uso do Gemini Notebook como Tutor Complementar

Nesta edição, incentiva-se o uso da plataforma **Gemini Notebook** como ferramenta complementar de aprendizagem — **não como substituta** da leitura atenta, da resolução de exercícios ou da experimentação prática. Baseado em arquiteturas de inteligência artificial, o sistema utiliza exclusivamente o material didático e os documentos fornecidos pelo autor como base de conhecimento, assegurando que as respostas geradas estejam conceitualmente alinhadas ao conteúdo programático e à abordagem pedagógica adotada ao longo desta obra.

> ### ❗ Acesso ao Tutor Inteligente
>
> [🚀 ACESSAR Gemini Notebook: CAPÍTULO 05](https://notebooklm.google.com/notebook/b8b6cd26-ef65-4a10-b7e7-e072d4870ddb)
>
> #### 🌐 Idioma e Linguagem de Programação
>
> O projeto deste capítulo no Gemini Notebook foi construído apenas com o texto em **português** e os exemplos de código em **Python**. Se você está estudando pela edição em inglês ou francês, ou acompanhando a trilha em C++, as respostas do tutor podem não corresponder exatamente à versão que você está lendo.
>
> #### Diretrizes sobre o Conteúdo Gerado por Inteligência Artificial
>
> Embora as ferramentas de inteligência artificial constituam aliados eficientes no processo de aprendizagem e revisão, o conteúdo gerado está sujeito a inconsistências ou imprecisões técnicas. Desse modo, é indispensável a consulta sistemática a livros-texto, artigos científicos e fontes acadêmicas indexadas para a validação rigorosa das informações. Recomenda-se veementemente a execução e a modificação dos exemplos práticos em Python fornecidos neste capítulo como método primário de verificação experimental dos resultados.

## 5.16 Lista de Exercícios

1. **(10%) Implementação Direta da DFT 2D:** Implemente analiticamente a Transformada Discreta de Fourier 2D (DFT) sem o auxílio de funções nativas de bibliotecas (como `np.fft.fft2`), utilizando estritamente a formulação matemática definida na [Equação 5.1](#eq-05-dft) para uma matriz de dimensões $16 \times 16$. Realize a validação numérica comparando os coeficientes gerados com os resultados da função `np.fft.fft2`, certificando-se de que o desvio absoluto máximo seja inferior a $10^{-8}$. Mensure os tempos de execução de ambos os métodos e apresente uma justificativa teórica para a disparidade observada em termos de complexidade assintótica.

2. **(15%) Supressão de Ruído Periódico:** Adicione interferências senoidais com frequências espaciais $(u_0, v_0) \in \{(5,10), (20,5), (30,30)\}$ à imagem de teste do *Cameraman*. Para cada cenário de degradação, projete uma máscara de filtragem *notch* específica no domínio da frequência para isolar e atenuar os picos harmônicos indesejados. Avalie quantitativamente a eficácia do processo de restauração por meio do cálculo das métricas de PSNR e SSIM. Discuta analiticamente o compromisso (*trade-off*) entre a atenuação do ruído senoidal e a indesejada atenuação de feições estruturais legítimas da imagem.

3. **(15%) Análise Comparativa de Operadores Passa-Baixa:** Realize um estudo comparativo entre os filtros passa-baixa Ideal, Gaussiano e Butterworth (com ordens harmônicas $n = 1, 2, 4$), parametrizados com frequências de corte $D_0 = 20, 40, 60$ pixels. Para cada combinação estrutural, calcule os índices PSNR e SSIM da imagem resultante frente ao sinal original de referência. Organize os dados quantitativos em uma tabela estruturada e plote os gráficos unidimensionais das funções de transferência correspondentes ao longo do perfil horizontal $H(u, 0)$.

4. **(15%) Banco de Filtros Multirresolução de Haar:** Desenvolva um script para executar manualmente a decomposição *wavelet* discreta 2D de primeiro nível utilizando a família Haar. O algoritmo deve calcular os coeficientes dos filtros correspondentes passa-baixa ($h$) e passa-alta ($g$), aplicando-os de forma separável sobre as linhas e colunas da matriz, seguidos pela operação de decimação (subamostragem espacial por um fator de 2). Valide numericamente a exatidão da sua implementação confrontando as subbandas obtidas com a saída da função `pywt.dwt2(img, 'haar')`.

5. **(15%) Compressão Esparsa por Limiarização Wavelet:** Aplique a técnica de filtragem por limiarização abrupta (*hard thresholding*) sobre os coeficientes de detalhe da decomposição *wavelet*, adotando os limiares numéricos $T \in \{5, 10, 20, 40, 80\}$ para as famílias Haar, Daubechies (`db4`) e Symlets (`sym4`). Após realizar o processo de síntese por meio da transformada inversa (`pywt.waverec2`), compute os valores de PSNR e SSIM de cada imagem reconstruída. Identifique e justifique qual combinação de família *wavelet* e limiar $T$ maximiza a similaridade estrutural.

6. **(15%) Construção de Codificador JPEG Simplificado:** Implemente o pipeline completo de compressão de dados simulando o padrão JPEG. O fluxo deve englobar: conversão espacial $RGB \rightarrow YC_bC_r$, subamostragem cromática na proporção 4:2:0, segmentação da luminância em blocos disjuntos de $8 \times 8$ pixels, aplicação da DCT-II 2D ortogonal, e quantização linear baseada na matriz normalizada de luminância escalada por fatores de qualidade desejados. Realize a decodificação inversa e compare quantitativamente as reconstruções com os arquivos gerados pela função `cv2.imencode` para os fatores de qualidade de 20, 50 e 80.

7. **(15%) Análise Perceptual em Conteúdos Heterogêneos:** Desenvolva uma imagem sintética composta por três regiões distintas e de características espectrais contrastantes: uma textura fotográfica complexa (representando altas frequências estocásticas), uma área de texto vetorizado com bordas nítidas (representando transições degrau puras) e um gradiente linear contínuo (representando baixas frequências homogêneas). Submeta essa imagem mista aos processos de compressão sob os formatos JPEG, PNG e WebP. Avalie e interprete os resultados correlacionando o tamanho final do arquivo em disco às métricas PSNR e SSIM obtidas, justificando qual formato exibe o melhor desempenho para sinais de natureza heterogênea e por que essa vantagem ocorre em termos de compactação de energia e preservação perceptual.

## Referências do Capítulo


A fundamentação teórica e o desenvolvimento analítico dos conceitos abordados neste capítulo fundamentam-se nas seguintes obras de referência:
* **Gonzalez (2018)** — Formulações clássicas de Transformadas Discretas de Fourier 2D (DFT), projeto de filtros analíticos no domínio da frequência, Transformada Discreta de Cossenos (DCT) e princípios fundamentais de sistemas de compressão de imagens.
* **Oppenheim (2010)** — Teoria formal de sinais e sistemas aplicados no domínio discreto, cobrindo as propriedades matemáticas da DFT e a modelagem analítica do Teorema da Convolução.
* **Mallat (1999)** — Fundamentação matemática da teoria de *wavelets*, formalização da análise multirresolução (MRA) e arquitetura de bancos de filtros diádicos.
* **Wallace (1991)** — Especificação original e aspectos de engenharia do padrão de compressão ISO/IEC JPEG, com ênfase nos critérios psicovisuais para o projeto de matrizes de quantização DCT.
* **Szeliski (2022)** — Modelagem computacional e caracterização de métricas modernas de fidelidade e qualidade perceptual (PSNR e SSIM), bem como a análise comparativa de formatos de imagem rasterizados de alto desempenho.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

MALLAT, St{\'e}phane. **A wavelet tour of signal processing**. Elsevier, 1999.

OPPENHEIM, Alan V.; SCHAFER, Ronald W. **Discrete-Time Signal Processing**. Pearson, 2010.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

WALLACE, Gregory K. **The {JPEG} Still Picture Compression Standard**. 1991.